# Experimento VQE — versão 20.19

## Padrão generalizável: interferência dirigida + topologia de transporte + fronteiras de decisão + Transformer

Esta versão preserva integralmente o **20.16** e acrescenta uma Parte IX desenhada para transformar os resultados já observados em uma história científica auditável.

A sequência agora testa, sem introduzir o bitstring ótimo na função objetivo:

1. os **30 parâmetros nominais** e a solução de referência;
2. a redução local para os **7 θ geometricamente ativos**;
3. a possível redução operacional para os **3 blocos transportadores** `θ25 → θ22 → θ17`;
4. a necessidade individual de cada um dos 7 por **leave-one-out**;
5. as **6 permutações** possíveis dos três blocos transportadores;
6. o path-sum dentro das **portas primitivas** desses três blocos;
7. os comprimentos Fubini–Study de trajetórias `30`, `7` e `3`, sempre acompanhados da fidelidade do endpoint;
8. figuras-resumo que contam a sequência `30 → 7 → 3 → 1 trajetória` sem construir um score arbitrário.

> **Regra de interpretação:** o notebook continua não chamando a trajetória interpolada de “caminho de mínima ação”. O comprimento Fubini–Study é calculado para trajetórias explicitamente definidas. Uma geodésica só poderá ser afirmada se uma minimização de comprimento for feita posteriormente.


## Parte X — do caso particular ao padrão transferível

A nova Parte X não tenta ensinar ao modelo que `theta_22` é universalmente importante. Ela extrai descritores que continuam fazendo sentido quando mudamos a cardinalidade, os retornos/risco dos ativos ou a própria quantidade de parâmetros do ansatz.

A hipótese geral a ser testada é:

$$
\text{problema }(n,k,H)
\longrightarrow
\text{subespaço ativo}
\longrightarrow
\text{grafo de transporte}
\longrightarrow
\text{histórias coerentes}
\longrightarrow
\text{seleção ótima}.
$$

Esses objetos — e não o número nominal de um $\theta_j$ — são os candidatos a padrão transferível para o Transformer.


## Parte XI — novos controles da versão 20.19

A versão 20.19 acrescenta testes que surgiram diretamente dos resultados da 20.18:

- as **seis ordens** dos três transportadores são comparadas com os mesmos ângulos, agora com histórias resolvidas;
- a coerência é decomposta em **multiplicidade de histórias** e **alinhamento de sinais/fases**;
- um teste de **reachability do alvo** separa, de forma diagnóstica, limitação de expressividade de dificuldade da otimização por energia;
- a rota `k-em-n` é convertida em um **grafo de transporte**, com relays e alongamento combinatório;
- os 129 problemas clássicos são transformados em **fronteiras de decisão**, incluindo um limiar analítico exato para choques de retorno;
- é gerado um manifesto pequeno de instâncias que realmente devem ser levadas ao VQE/QGT nas próximas cardinalidades.

> Os testes de reachability usam o bitstring conhecido **somente como diagnóstico de expressividade**. Esses valores nunca entram no treinamento como features.


## Como interpretar o circuito antes dos testes

O circuito começa aplicando portas `X` em $k$ qubits. Isso prepara **um estado-base inicial com peso de Hamming $k$**; não significa que a solução ótima já foi inserida no circuito.

Depois, os blocos parametrizados redistribuem a amplitude entre estados que mantêm a cardinalidade:

- `CY`: bloco lógico de **dois qubits**, decomposto com uma rotação controlada `CRY`;
- `CCY`: bloco lógico de **três qubits**, decomposto com `RY` e `CCX`;
- `RY`: operação primitiva de um qubit;
- `CX`: operação primitiva de dois qubits;
- `CCX`: operação primitiva de três qubits.

Portanto, não é correto dizer que toda porta é simplesmente uma junção de dois spins. O Hamiltoniano do portfólio é diagonal e contém termos de um e dois corpos, `Z` e `ZZ`, enquanto o **ansatz** usa blocos de dois e três qubits para navegar no subespaço de Dicke.

O índice $j$ em $\theta_j$ não deve ser fornecido isoladamente ao Transformer como significado físico. O objeto transferível é a descrição estrutural:

$$
(\text{tipo de bloco},\; \text{qubits/ativos},\; \text{distância},\;
\text{posição},\; \text{período},\; \text{termos do Hamiltoniano tocados}).
$$


### Célula 1 — Importações, parâmetros e pastas do experimento

**Em termos simples:** esta célula configura somente o que é necessário para selecionar 100 vetores e executar as varreduras individuais.

**O que é configurado:**

- os parâmetros financeiros $q$, $r_f$ e a cardinalidade $k=4$;
- a máscara dos 10% melhores;
- a seleção final de exatamente 100 vetores completos;
- os índices $\theta_j$ testados;
- 201 pontos regulares por período, além do valor original inserido exatamente;
- checkpoints por par `(vetor, theta)`, permitindo retomar uma execução interrompida;
- um diretório novo de saída para as tabelas e figuras corrigidas.

A grade usa o período estrutural de cada porta: `RY` em $[0,2\pi]$ e `CRY` em $[0,4\pi]$. Uma auditoria posterior verifica se, para o observável de probabilidade, os dois ciclos de uma `CRY` são de fato diferentes.

A versão 20.7 procura primeiro seus próprios checkpoints e, quando eles ainda não existem, reutiliza automaticamente os checkpoints da versão 20.6. Assim, a campanha de aproximadamente 180 mil avaliações não precisa ser refeita apenas para corrigir os resumos e gráficos.


In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO ÚNICA
# ============================================================

from __future__ import annotations

from itertools import combinations, permutations
from pathlib import Path
import ast
import hashlib
import json
import math
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import spearmanr

NOTEBOOK_VERSION = "20.19-directed-interference-transport-boundaries-transformer"
RANDOM_SEED = 42

# Parâmetros do problema financeiro.
Q_VALUE = 0.5
RISK_FREE = 0.0475
TARGET_K = 4
ENERGY_ATOL = 1e-8

# ------------------------------------------------------------
# ÚNICO CAMINHO DE ENTRADA
# Altere somente esta linha quando o merge.pkl estiver em outro local.
# ------------------------------------------------------------
MERGE_PKL = Path(r"C:\Users\Marlon_Kelly\Downloads\merge.pkl")

# A máscara conserva aproximadamente os 10% melhores vetores salvos.
TOP_FRACTION = 0.10

# Reavalia mais candidatos do que o número final para escolher 100 vetores
# usando as métricas exatas reconstruídas neste notebook.
MAX_EXACT_REEVALUATION = 300
N_AUDIT_VECTORS = 100
N_ANCHORS = 1

# Parâmetros varridos individualmente em cada um dos 100 vetores.
ACTIVE_THETA_INDICES = [2, 14, 17, 19, 22, 25, 27]
DETAILED_THETA_INDICES = [17, 2, 14, 19, 22, 25, 27, 3, 24]

# 201 pontos regulares por período, mais o ponto original quando necessário.
# Isso controla o custo: 100 vetores x 9 thetas x aproximadamente 202 pontos.
SWEEP_POINTS_PER_PERIOD = 201
COMMON_PHASE_POINTS = 201

# Critério apenas para informar se uma curva é numericamente plana.
SWEEP_FLAT_ABS_TOL = 1e-10
SWEEP_FLAT_REL_TOL = 1e-8
PERIODICITY_ATOL = 1e-9
PERIODICITY_RTOL = 1e-7

# Checkpoints novos desta versão podem ser reutilizados para retomar a campanha.
REUSE_SWEEP_CHECKPOINTS = True

# Saídas corrigidas da versão 20.7.
OUTPUT_ROOT = Path("vqe_r") / "pipeline_v20_17_compression_causal_geometry_pathsum"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
DISTRIBUTION_DIR = OUTPUT_ROOT / "distributions"

# Checkpoints produzidos pela versão 20.6 são matematicamente compatíveis,
# pois o motor de varredura não foi alterado; somente a análise foi corrigida.
LEGACY_OUTPUT_ROOT = Path("vqe_r") / "pipeline_v20_11_single_vector_symmetric_sweep"
LEGACY_CHECKPOINT_DIR = LEGACY_OUTPUT_ROOT / "checkpoints"

for directory in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR, DISTRIBUTION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "notebook_version": NOTEBOOK_VERSION,
    "merge_pkl": str(MERGE_PKL),
    "target_k": TARGET_K,
    "q_value": Q_VALUE,
    "risk_free": RISK_FREE,
    "top_fraction": TOP_FRACTION,
    "n_audit_vectors": N_AUDIT_VECTORS,
    "n_sweep_vectors": N_ANCHORS,
    "max_exact_reevaluation": MAX_EXACT_REEVALUATION,
    "active_theta_indices": ACTIVE_THETA_INDICES,
    "detailed_theta_indices": DETAILED_THETA_INDICES,
    "sweep_points_per_period": SWEEP_POINTS_PER_PERIOD,
    "reuse_sweep_checkpoints": REUSE_SWEEP_CHECKPOINTS,
    "legacy_checkpoint_dir": str(LEGACY_CHECKPOINT_DIR),
    "optimizer_used": False,
    "cobyla_calls": 0,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("Saídas:", OUTPUT_ROOT.resolve())
print("Checkpoints legados procurados em:", LEGACY_CHECKPOINT_DIR.resolve())

# Parte I — carregar e auditar somente o `merge.pkl`

O carregamento não procura nomes alternativos em vários diretórios. Existe um único caminho configurado em `MERGE_PKL`.

A célula seguinte aceita `DataFrame`, lista de dicionários ou dicionário serializado, mas não reconstrói nem modifica o banco original.


### Célula 2 — Carregamento controlado do banco `merge.pkl`

**Em termos simples:** esta célula abre o único arquivo de entrada e o transforma em um `DataFrame` de trabalho.

Ela verifica se o caminho existe, aceita três formatos serializados (`DataFrame`, lista de registros ou dicionário) e interrompe a execução caso o arquivo esteja vazio ou tenha um tipo inesperado.

**Saída principal:** `merge_df`, que contém o banco original carregado em memória. O arquivo em disco não é modificado.


In [ ]:
# ============================================================
# 2. CARREGAMENTO ÚNICO DO merge.pkl
# ============================================================

# Resolve "~", converte para caminho absoluto e verifica o arquivo antes da leitura.
merge_path = MERGE_PKL.expanduser().resolve()
if not merge_path.is_file():
    raise FileNotFoundError(
        "merge.pkl não encontrado. Caminho configurado: "
        f"{merge_path}"
    )

# O pickle é lido uma única vez. As conversões seguintes ocorrem apenas em memória.
loaded_object = pd.read_pickle(merge_path)

# Padroniza diferentes formatos serializados para um único DataFrame.
if isinstance(loaded_object, pd.DataFrame):
    merge_df = loaded_object.copy()
elif isinstance(loaded_object, list):
    merge_df = pd.DataFrame(loaded_object)
elif isinstance(loaded_object, dict):
    merge_df = pd.DataFrame(loaded_object)
else:
    raise TypeError(
        "O merge.pkl deve conter DataFrame, lista de registros ou dicionário; "
        f"tipo encontrado: {type(loaded_object)}"
    )

if merge_df.empty:
    raise ValueError("O merge.pkl foi carregado, mas está vazio.")

print("Arquivo:", merge_path)
print("Shape:", merge_df.shape)
print("Colunas:", merge_df.columns.tolist())
display(merge_df.head())


### Célula 3 — Identificação das colunas e conversão dos dados

**Em termos simples:** bancos gerados em versões diferentes podem usar nomes diferentes para a mesma informação. Esta célula cria um mapa de aliases e identifica qual coluna representa retorno, covariância, vetor de parâmetros, energia, probabilidade e bitstring.

Também são definidas funções para converter conteúdos salvos como texto em objetos numéricos:

- `parse_tickers`: recupera os nomes dos ativos;
- `parse_vector`: transforma retornos e vetores $\theta$ em arrays;
- `parse_matrix`: reconstrói a matriz de covariância;
- `normalize_bitstring`: padroniza bitstrings para uma sequência de zeros e uns.

**Importante:** essa normalização ocorre apenas na cópia em memória. O `merge.pkl` original permanece intacto.


In [ ]:
# ============================================================
# 3. NORMALIZAÇÃO DO ESQUEMA SEM ALTERAR O ARQUIVO ORIGINAL
# ============================================================

# Cada chave representa um conceito do experimento; a lista contém nomes de
# coluna aceitos para esse mesmo conceito em versões diferentes do banco.
COLUMN_ALIASES = {
    "tickers": ["tickers", "assets", "asset_names"],
    "assets_return": ["assets_return", "assets_returns", "expected_returns", "mu"],
    "covariance": ["covariance", "covariance_matrix", "sigma"],
    "best_parameters": ["best_parameters", "theta", "theta_final"],
    "initial_point": ["initial_point", "initial_theta", "theta_initial"],
    "objective": ["objective_function_value", "energy", "final_energy"],
    "best_objective": ["best_objective_function_value", "exact_energy", "optimal_energy"],
    "p_best": [
        "p_exact_eval",
        "probability_best_answer",
        "probability_best_answer_shots",
        "p_best",
        "prob_best",
    ],
    "gap": ["gap_exact_eval", "energy_gap", "gap"],
    "dominant_bitstring": [
        "most_frequent_bitstring",
        "most_frequen_bitstring",
        "best_answer",
        "dominant_bitstring",
    ],
    "counts": ["counts", "measurement_counts"],
    "status": ["status"],
}


def first_existing_column(frame, aliases, required=False):
    """Retorna o primeiro alias realmente presente no DataFrame."""
    for name in aliases:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(f"Nenhuma das colunas obrigatórias foi encontrada: {aliases}")
    return None


# Resultado final do mapeamento: conceito lógico -> nome real no merge.pkl.
RESOLVED_COLUMNS = {
    key: first_existing_column(
        merge_df,
        aliases,
        required=key in {"tickers", "assets_return", "covariance", "best_parameters"},
    )
    for key, aliases in COLUMN_ALIASES.items()
}


def parse_serialized(value):
    """Converte texto serializado em lista, dicionário ou array quando possível."""
    if isinstance(value, (np.ndarray, list, tuple, dict, pd.Series, pd.Index, pd.DataFrame)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                pass
        cleaned = text.strip("[]()")
        arr = np.fromstring(cleaned.replace(",", " "), sep=" ")
        if arr.size:
            return arr
    return value


def parse_tickers(value):
    """Padroniza a lista de tickers e rejeita nomes vazios."""
    parsed = parse_serialized(value)
    if isinstance(parsed, str):
        items = [item.strip() for item in parsed.replace(";", ",").split(",")]
    elif isinstance(parsed, dict):
        items = list(parsed.keys())
    else:
        items = list(parsed)
    tickers = [str(item).strip().strip("'\"") for item in items]
    if not tickers or any(not item for item in tickers):
        raise ValueError(f"Tickers inválidos: {value}")
    return tickers


def parse_vector(value, tickers=None):
    """Converte um vetor salvo em texto, Series, dicionário ou lista para NumPy."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.Series):
        if tickers is not None and set(tickers).issubset(set(parsed.index.astype(str))):
            return parsed.reindex(tickers).to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        if tickers is not None and set(tickers).issubset(set(map(str, parsed.keys()))):
            return np.asarray([parsed[ticker] for ticker in tickers], dtype=float)
        return np.asarray(list(parsed.values()), dtype=float)
    return np.asarray(parsed, dtype=float).reshape(-1)


def parse_matrix(value, tickers=None):
    """Reconstrói uma matriz numérica e preserva a ordem dos tickers quando possível."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.DataFrame):
        if tickers is not None:
            return parsed.loc[tickers, tickers].to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        frame = pd.DataFrame(parsed)
        if tickers is not None and set(tickers).issubset(frame.index) and set(tickers).issubset(frame.columns):
            return frame.loc[tickers, tickers].to_numpy(dtype=float)
        return frame.to_numpy(dtype=float)
    array = np.asarray(parsed, dtype=float)
    if array.ndim == 1:
        n = int(round(np.sqrt(array.size)))
        if n * n != array.size:
            raise ValueError("Covariância unidimensional não forma uma matriz quadrada.")
        array = array.reshape(n, n)
    return array


def normalize_bitstring(value):
    """Remove prefixos e espaços, retornando apenas bitstrings binários válidos."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).replace(" ", "").replace("'", "").replace('"', "")
    if text.startswith("0b"):
        text = text[2:]
    return text if set(text).issubset({"0", "1"}) else None


print(json.dumps(RESOLVED_COLUMNS, indent=2, ensure_ascii=False))


### Célula 4 — Reconstrução do problema financeiro e auditoria do banco

**Em termos simples:** a primeira linha válida do banco é usada para recuperar os ativos, o vetor de retornos $\mu$ e a matriz de covariância $\Sigma$.

A covariância é explicitamente simetrizada:

$$
\Sigma_{\mathrm{sim}}=\frac{\Sigma+\Sigma^\mathsf{T}}{2}.
$$

Depois, uma amostra de até 200 linhas recebe uma impressão digital (`hash`). Se aparecer mais de um hash, o banco contém mais de um problema/Hamiltoniano e a execução é interrompida.

A célula também estima a cardinalidade pelos bitstrings salvos e cria `problem_summary_df`, com retorno, variância e conexão de risco de cada ativo.


In [ ]:
# ============================================================
# 4. EXTRAIR O HAMILTONIANO E AUDITAR CONSISTÊNCIA DO BANCO
# ============================================================

# Usa a primeira linha com um vetor theta válido como referência do problema.
first_valid_index = merge_df[
    merge_df[RESOLVED_COLUMNS["best_parameters"]].notna()
].index[0]
first_row = merge_df.loc[first_valid_index]

tickers = parse_tickers(first_row[RESOLVED_COLUMNS["tickers"]])
mu = parse_vector(first_row[RESOLVED_COLUMNS["assets_return"]], tickers=tickers)
sigma = parse_matrix(first_row[RESOLVED_COLUMNS["covariance"]], tickers=tickers)
# Corrige pequenas assimetrias numéricas sem alterar a parte simétrica do risco.
sigma = 0.5 * (sigma + sigma.T)

N_ASSETS = len(tickers)
if mu.shape != (N_ASSETS,):
    raise ValueError(f"Retornos com shape {mu.shape}; esperado {(N_ASSETS,)}.")
if sigma.shape != (N_ASSETS, N_ASSETS):
    raise ValueError(
        f"Covariância com shape {sigma.shape}; esperado {(N_ASSETS, N_ASSETS)}."
    )
if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("Retornos ou covariância possuem valores não finitos.")

# A cardinalidade é inferida do bitstring salvo quando possível.
bit_col = RESOLVED_COLUMNS["dominant_bitstring"]
if bit_col is not None:
    saved_bits = merge_df[bit_col].map(normalize_bitstring).dropna()
    inferred_weights = saved_bits.map(lambda value: value.count("1"))
    inferred_k = int(inferred_weights.mode().iloc[0]) if not inferred_weights.empty else TARGET_K
else:
    inferred_k = TARGET_K

if inferred_k != TARGET_K:
    warnings.warn(
        f"A cardinalidade modal inferida foi k={inferred_k}; "
        f"o experimento está configurado para k={TARGET_K}."
    )

# Confirma que uma amostra do merge representa o mesmo problema.
def problem_fingerprint(row):
    """Cria um hash a partir de tickers, retornos e covariância de uma linha."""
    row_tickers = parse_tickers(row[RESOLVED_COLUMNS["tickers"]])
    row_mu = parse_vector(row[RESOLVED_COLUMNS["assets_return"]], tickers=row_tickers)
    row_sigma = parse_matrix(row[RESOLVED_COLUMNS["covariance"]], tickers=row_tickers)
    payload = (
        "|".join(row_tickers).encode("utf-8")
        + np.asarray(row_mu, dtype=np.float64).tobytes()
        + np.asarray(row_sigma, dtype=np.float64).tobytes()
    )
    return hashlib.sha256(payload).hexdigest()[:16]

# Uma amostra aleatória é suficiente para detectar mistura evidente de problemas,
# sem reler e converter necessariamente todas as linhas do banco.
sample_size = min(200, len(merge_df))
sampled_rows = merge_df.sample(sample_size, random_state=RANDOM_SEED)
fingerprints = sampled_rows.apply(problem_fingerprint, axis=1)
if fingerprints.nunique() != 1:
    raise RuntimeError(
        "O merge.pkl contém mais de um Hamiltoniano na amostra auditada. "
        "Este notebook 20.6 executa um problema por vez; filtre o merge antes de continuar."
    )

DATA_HASH = fingerprints.iloc[0]
min_cov_eigenvalue = float(np.linalg.eigvalsh(sigma).min())

# Tabela por ativo usada posteriormente como descrição estrutural do problema.
problem_summary_df = pd.DataFrame({
    "asset_index": np.arange(N_ASSETS, dtype=int),
    "ticker": tickers,
    "return_sum": mu,
    "variance": np.diag(sigma),
    "risk_connection_abs": np.sum(np.abs(sigma), axis=1),
})

print("n =", N_ASSETS, "| k =", TARGET_K)
print("tickers =", tickers)
print("problem_hash =", DATA_HASH)
print("menor autovalor da covariância =", f"{min_cov_eigenvalue:.3e}")
display(problem_summary_df)


# Parte II — referência clássica e rigidez dos pares

## O que esta parte representa

Antes de analisar o circuito quântico, o notebook resolve exatamente o problema clássico. Como existem 10 ativos e o portfólio deve selecionar 4, o número total de soluções válidas é

$$
\binom{10}{4}=210.
$$

Isso significa que é possível avaliar todos os 210 portfólios e conhecer, sem aproximação:

- a energia mínima exata;
- todos os bitstrings ótimos;
- a distância energética entre decisões concorrentes;
- quais pares de ativos possuem decisões mais rígidas.

## Mínimos condicionados de cada par

Para cada par de ativos $(i,j)$, fixamos os valores de decisão $x_i=a$ e $x_j=b$, com $a,b\in\{0,1\}$. Em seguida, procuramos o melhor portfólio que respeita essas duas decisões e continua selecionando exatamente $k$ ativos:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_{\ell}x_{\ell}=k}}
E(x).
$$

Assim, cada par possui quatro energias condicionadas:

$$
E_{ij}^{00},\qquad
E_{ij}^{01},\qquad
E_{ij}^{10},\qquad
E_{ij}^{11}.
$$

Esses valores permitem medir quanto custa trocar a decisão de um ativo, dos dois ativos e quão separado está o melhor estado do par em relação ao segundo melhor. Mais adiante, essas métricas serão ligadas aos blocos quânticos que atuam sobre os mesmos qubits.


### Célula 5 — Solução clássica exata e rigidez dos pares de ativos

**Em termos simples:** para 10 ativos escolhendo exatamente 4, todos os portfólios válidos podem ser enumerados:

$$
N_{\mathrm{portfólios}}=\binom{10}{4}=210.
$$

A função objetivo avaliada para cada bitstring é

$$
E(x)=q\,x^\mathsf{T}\Sigma x-(1-q)\,\mu^\mathsf{T}x+r_f,
$$

com a restrição $\sum_i x_i=k$.

Para cada par de ativos $(i,j)$ e cada estado $a,b\in\{0,1\}$, é calculado o melhor portfólio condicionado:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_\ell x_\ell=k}}
E(x).
$$

Esses quatro mínimos permitem medir:

- `G_ij`: separação entre o melhor e o segundo melhor estado condicionado do par;
- gaps de trocar apenas $i$, apenas $j$ ou os dois;
- não aditividade da troca conjunta.

**Saídas principais:** `enumeration_df`, `asset_decision_df`, `pair_gap_df`, `exact_energy` e os bitstrings ótimos.


In [ ]:
# ============================================================
# 5. ENUMERAÇÃO CLÁSSICA EXATA E GAPS CONDICIONAIS
# ============================================================

PAIR_STATES = ("00", "01", "10", "11")


def portfolio_objective(x_binary):
    """Calcula E(x)=q*x.T*Sigma*x-(1-q)*mu.T*x+r_f para um portfólio binário."""
    x = np.asarray(x_binary, dtype=float).reshape(-1)
    return float(
        Q_VALUE * x @ sigma @ x
        - (1.0 - Q_VALUE) * mu @ x
        + RISK_FREE
    )


def bitstring_asset_order(x):
    """Converte o vetor binário para a ordem natural dos ativos."""
    return "".join(str(int(value)) for value in np.asarray(x, dtype=int))


def enumerate_portfolios(k_value=TARGET_K):
    """Enumera exatamente todas as combinações de k ativos entre N_ASSETS."""
    rows = []
    for selected_indices in combinations(range(N_ASSETS), int(k_value)):
        x = np.zeros(N_ASSETS, dtype=int)
        x[list(selected_indices)] = 1
        bits = bitstring_asset_order(x)
        rows.append({
            "x_asset_order": tuple(int(v) for v in x),
            "bitstring_asset_order": bits,
            "bitstring_qiskit_order": bits[::-1],
            "selected_assets": tuple(tickers[index] for index in selected_indices),
            "objective": portfolio_objective(x),
        })
    return pd.DataFrame(rows).sort_values(
        ["objective", "bitstring_asset_order"]
    ).reset_index(drop=True)


# Para n=10 e k=4, esta tabela possui C(10,4)=210 linhas.
enumeration_df = enumerate_portfolios(TARGET_K)
exact_energy = float(enumeration_df.iloc[0]["objective"])
optimal_mask = np.isclose(
    enumeration_df["objective"].to_numpy(dtype=float),
    exact_energy,
    atol=ENERGY_ATOL,
    rtol=0.0,
)
optimal_df = enumeration_df.loc[optimal_mask].copy()
exact_asset_bitstrings = sorted(optimal_df["bitstring_asset_order"].unique())
exact_qiskit_bitstrings = sorted(optimal_df["bitstring_qiskit_order"].unique())
exact_x = np.asarray([int(value) for value in exact_asset_bitstrings[0]], dtype=int)
energy_span = float(enumeration_df["objective"].max() - exact_energy)


def build_asset_decision_margins():
    """Mede o custo mínimo de inverter a decisão de cada ativo no ótimo clássico."""
    selected = np.flatnonzero(exact_x == 1)
    excluded = np.flatnonzero(exact_x == 0)
    rows = []
    for asset_index in range(N_ASSETS):
        alternatives = []
        if exact_x[asset_index] == 1:
            for replacement in excluded:
                trial = exact_x.copy()
                trial[asset_index] = 0
                trial[replacement] = 1
                alternatives.append(portfolio_objective(trial))
        else:
            for removed in selected:
                trial = exact_x.copy()
                trial[asset_index] = 1
                trial[removed] = 0
                alternatives.append(portfolio_objective(trial))
        margin = max(min(alternatives) - exact_energy, 0.0)
        rows.append({
            "asset_index": asset_index,
            "ticker": tickers[asset_index],
            "selected_exact": int(exact_x[asset_index]),
            "decision_margin": float(margin),
            "decision_margin_relative": float(margin / max(energy_span, ENERGY_ATOL)),
        })
    return pd.DataFrame(rows).sort_values("decision_margin", ascending=False)


asset_decision_df = build_asset_decision_margins()


def conditional_pair_best(i, j, state):
    """Retorna o melhor portfólio sob x_i=a e x_j=b para um estado ab."""
    a, b = int(state[0]), int(state[1])
    subset = enumeration_df.loc[
        enumeration_df["x_asset_order"].map(
            lambda x: int(x[i]) == a and int(x[j]) == b
        )
    ]
    if subset.empty:
        raise RuntimeError(f"Estado inviável para par {(i, j)}: {state}")
    return subset.iloc[0]


# Para cada par, calculamos E_00, E_01, E_10 e E_11 e derivamos os gaps.
pair_rows = []
for i, j in combinations(range(N_ASSETS), 2):
    states = {state: conditional_pair_best(i, j, state) for state in PAIR_STATES}
    ordered = sorted(PAIR_STATES, key=lambda state: (states[state]["objective"], state))
    best_state, second_state = ordered[:2]
    global_state = f"{exact_x[i]}{exact_x[j]}"
    a_star, b_star = map(int, global_state)
    flip_i = f"{1-a_star}{b_star}"
    flip_j = f"{a_star}{1-b_star}"
    flip_both = f"{1-a_star}{1-b_star}"
    delta_i = max(float(states[flip_i]["objective"] - exact_energy), 0.0)
    delta_j = max(float(states[flip_j]["objective"] - exact_energy), 0.0)
    delta_both = max(float(states[flip_both]["objective"] - exact_energy), 0.0)
    # G_ij mede quão rigidamente o melhor estado condicionado do par se separa do segundo.
    gij = max(float(states[second_state]["objective"] - states[best_state]["objective"]), 0.0)
    pair_rows.append({
        "asset_index_i": i,
        "asset_index_j": j,
        "asset_i": tickers[i],
        "asset_j": tickers[j],
        "pair": f"{tickers[i]}/{tickers[j]}",
        "global_pair_state": global_state,
        "G_ij": gij,
        "G_ij_relative": gij / max(energy_span, ENERGY_ATOL),
        "single_flip_i_gap": delta_i,
        "single_flip_j_gap": delta_j,
        "joint_flip_gap": delta_both,
        "joint_flip_nonadditivity": delta_both - delta_i - delta_j,
        **{f"E_{state}": float(states[state]["objective"]) for state in PAIR_STATES},
    })

pair_gap_df = pd.DataFrame(pair_rows).sort_values("G_ij", ascending=False).reset_index(drop=True)

print("Portfólios válidos:", len(enumeration_df))
print("Energia exata:", exact_energy)
print("Bitstring ótimo — ordem dos ativos:", exact_asset_bitstrings)
print("Bitstring ótimo — ordem Qiskit:", exact_qiskit_bitstrings)
print("Ativos selecionados:", optimal_df.iloc[0]["selected_assets"])
display(asset_decision_df)
display(pair_gap_df.head(15))


# Parte III — reconstrução exata do Hamiltoniano e do ansatz do modelo 20.1

A construção abaixo foi separada do gerador de banco, mas preserva a mesma lógica do notebook 20.1:

- mesmo QUBO/Ising;
- mesmo estado inicial de peso $k$;
- mesmos blocos `CY` e `CCY`;
- mesma ordenação rastreável dos parâmetros;
- mesmo `ANSATZ_SEED`.

O notebook interrompe a execução caso o vetor salvo tenha dimensão incompatível ou caso a auditoria estrutural falhe.


### Célula 6 — Dependências quânticas sem importação de otimizadores

**Em termos simples:** esta célula carrega somente as classes necessárias para montar o QUBO, converter para Ising, construir o circuito e calcular o `Statevector`.

Nenhum método como COBYLA, SPSA ou outro otimizador variacional é importado. Isso garante que as células seguintes apenas atribuam valores de $\theta$ e avaliem diretamente o circuito.


In [ ]:
# ============================================================
# 6. DEPENDÊNCIAS QUÂNTICAS — SEM OTIMIZADOR
# ============================================================

# Dependências para formular o problema binário e construir o circuito.
# Não há importação de classes de otimização variacional.
from docplex.mp.model import Model
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

print("Dependências quânticas carregadas. Nenhum otimizador foi importado.")


### Célula 7 — Construção do QUBO e do Hamiltoniano de Ising

**Em termos simples:** o problema financeiro clássico é escrito com variáveis binárias $x_i\in\{0,1\}$ e a restrição de selecionar exatamente $k$ ativos:

$$
\sum_i x_i=k.
$$

O modelo é convertido para QUBO e depois para um Hamiltoniano de Ising:

$$
H=\sum_i h_i Z_i+\sum_{i<j}J_{ij}Z_iZ_j+\text{constante}.
$$

A célula exige que o Hamiltoniano seja diagonal, isto é, sem termos `X` ou `Y`. Em seguida, compara a energia mínima do Ising com a energia obtida pela enumeração dos 210 portfólios. Se elas não coincidirem, a execução para.

**Saídas principais:** `ising`, `ising_offset` e `hamiltonian_terms_df`.


In [ ]:
# ============================================================
# 7. QUBO E HAMILTONIANO DE ISING
# ============================================================


def build_docplex_ising():
    """Constrói o modelo binário, converte para QUBO/Ising e audita a energia exata."""
    model = Model(name=f"portfolio_n{N_ASSETS}_k{TARGET_K}")
    variables = np.array(
        [model.binary_var(name=f"x_{index}") for index in range(N_ASSETS)],
        dtype=object,
    )
    # Termo quadrático de risco x^T Sigma x.
    risk_expression = model.sum(
        float(sigma[row, column]) * variables[row] * variables[column]
        for row in range(N_ASSETS)
        for column in range(N_ASSETS)
    )
    # Termo linear de retorno esperado mu^T x.
    return_expression = model.sum(
        float(mu[index]) * variables[index]
        for index in range(N_ASSETS)
    )
    model.minimize(
        Q_VALUE * risk_expression
        - (1.0 - Q_VALUE) * return_expression
        + RISK_FREE
    )
    # Restrição de cardinalidade: exatamente TARGET_K variáveis devem valer 1.
    model.add_constraint(
        model.sum(variables.tolist()) == int(TARGET_K),
        ctname="budget",
    )

    # Docplex -> QuadraticProgram -> QUBO -> operador Ising e deslocamento constante.
    quadratic_program = from_docplex_mp(model=model)
    qubo = QuadraticProgramToQubo().convert(quadratic_program)
    ising, offset = qubo.to_ising()

    # O problema de portfólio deve gerar apenas termos diagonais I, Z e ZZ.
    labels = [str(label) for label in ising.paulis.to_labels()]
    non_diagonal = [label for label in labels if "X" in label or "Y" in label]
    if non_diagonal:
        raise RuntimeError(f"Hamiltoniano não diagonal: {non_diagonal[:10]}")

    # Auditoria independente: a menor energia Ising deve coincidir com a enumeração.
    exact_result = NumPyMinimumEigensolver().compute_minimum_eigenvalue(operator=ising)
    exact_energy_ising = float(np.real(exact_result.eigenvalue + offset))
    if not np.isclose(exact_energy_ising, exact_energy, atol=1e-10, rtol=0.0):
        raise RuntimeError(
            "Energia Ising e enumeração clássica não coincidem: "
            f"{exact_energy_ising} vs {exact_energy}"
        )

    terms_df = pd.DataFrame({
        "pauli_label": labels,
        "coefficient": np.real(np.asarray(ising.coeffs)).astype(float),
        "body_order": [label.count("Z") for label in labels],
    })
    return model, quadratic_program, qubo, ising, float(offset), terms_df


model, quadratic_program, qubo, ising, ising_offset, hamiltonian_terms_df = build_docplex_ising()
print("Termos Ising:", len(hamiltonian_terms_df))
display(hamiltonian_terms_df.sort_values(["body_order", "pauli_label"]))


### Célula 8 — Construção rastreável do ansatz de Dicke

**Em termos simples:** esta célula reproduz o circuito parametrizado usado no experimento 20.1 e registra a origem estrutural de cada parâmetro.

- `CY_parameterized` cria um bloco lógico de dois qubits cuja operação parametrizada primitiva é `CRY`;
- `CCY_parameterized` cria um bloco lógico de três qubits usando `RY` e `CCX`;
- as portas `X` iniciais preparam apenas um estado-base com peso de Hamming $k$;
- cada parâmetro recebe informações como posição, distância entre qubits e tipo de bloco.

O número esperado de parâmetros é

$$
N_\theta=\frac{k(2n-k-1)}{2}.
$$

**Saídas principais:** `ansatz`, `structure_df`, `initial_x_qubits` e `N_PARAMETERS`.


In [ ]:
# ============================================================
# 8. PORTAS E ANSATZ DE DICKE RASTREÁVEL — CÓPIA DA CABEÇA 20.1
# ============================================================


def CY_parameterized(identifier):
    """Cria o bloco lógico CY com uma rotação controlada CRY(theta)."""
    param = ParameterVector(name=f"x{identifier}", length=1)
    qc = QuantumCircuit(2)
    qc.cry(param[0], 1, 0)
    return qc.to_gate(label="CY")


def CCY_parameterized(identifier):
    """Cria o bloco lógico CCY com rotações RY(theta) e controles CCX."""
    param = ParameterVector(name=f"y{identifier}", length=1)
    qc = QuantumCircuit(3)
    qc.ry(param[0], 0)
    qc.ccx(2, 1, 0)
    qc.ry(-param[0], 0)
    qc.ccx(2, 1, 0)
    return qc.to_gate(label="CCY")


def dicke_parameter_count(n_value, k_value):
    """Número esperado de parâmetros do ansatz: k(2n-k-1)/2."""
    return int(k_value * (2 * n_value - k_value - 1) / 2)


def build_tracked_dicke_ansatz(n_value, k_value, seed):
    """Constrói o ansatz e registra a origem lógica de cada parâmetro."""
    # Preserva o estado aleatório global para que a construção do ansatz não
    # altere outras rotinas aleatórias do notebook.
    numpy_state = np.random.get_state()
    try:
        np.random.seed(int(seed))
        qr = QuantumRegister(n_value, "q")
        qc = QuantumCircuit(qr)

        # Prepara um estado-base com exatamente k excitações; não injeta o ótimo clássico.
        initial_x_qubits = []
        for excitation_index in range(k_value):
            qubit = n_value - excitation_index - 1
            qc.x(qubit)
            initial_x_qubits.append(qubit)

        # Cada bloco criado gera um registro que depois será ligado ao theta_index real.
        records = []
        aux = 1
        for l_value in range(n_value)[::-1]:
            for i_value in range(l_value - 1, l_value - 1 - k_value, -1):
                if i_value >= 0:
                    unique_name = f"{l_value}{i_value}{aux}{np.random.randint(0, int(1e8))}"
                    if i_value == l_value - 1:
                        gate = CY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CY"
                    else:
                        gate = CCY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[i_value + 1], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CCY"
                    records.append({
                        "parameter_object": gate_parameter,
                        "parameter_name": str(gate_parameter),
                        "l": int(l_value),
                        "i": int(i_value),
                        "distance": int(l_value - i_value),
                        "ansatz_gate_type": gate_type,
                    })
                aux += 1
    finally:
        np.random.set_state(numpy_state)

    # A ordem dos parâmetros é extraída do circuito decomposto, a mesma forma usada
    # nas avaliações por Statevector.
    decomposed = qc.decompose()
    ordered_parameters = list(decomposed.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(ordered_parameters)
    }
    structure_rows = []
    for record in records:
        record = record.copy()
        parameter_object = record.pop("parameter_object")
        structure_rows.append({
            "theta_index": int(parameter_to_index[parameter_object]),
            **record,
        })

    structure_df = pd.DataFrame(structure_rows).sort_values("theta_index").reset_index(drop=True)
    expected = dicke_parameter_count(n_value, k_value)
    if len(structure_df) != expected:
        raise RuntimeError(f"Esperados {expected} parâmetros; encontrados {len(structure_df)}.")

    structure_df["n"] = int(n_value)
    structure_df["k"] = int(k_value)
    structure_df["rho_k_over_n"] = k_value / n_value
    structure_df["u_l"] = structure_df["l"] / max(n_value - 1, 1)
    structure_df["u_distance"] = structure_df["distance"] / max(k_value, 1)
    structure_df["u_theta_index"] = structure_df["theta_index"] / max(expected - 1, 1)
    return decomposed, structure_df, tuple(sorted(initial_x_qubits))


ANSATZ_SEED = RANDOM_SEED + 100 * N_ASSETS + TARGET_K
ansatz, structure_df, initial_x_qubits = build_tracked_dicke_ansatz(
    N_ASSETS, TARGET_K, ANSATZ_SEED
)
N_PARAMETERS = int(ansatz.num_parameters)

print("ANSATZ_SEED =", ANSATZ_SEED)
print("n_parameters =", N_PARAMETERS)
print("qubits com X inicial =", initial_x_qubits)
print("estado-base inicial em ordem Qiskit =", "".join(
    "1" if qubit in initial_x_qubits else "0"
    for qubit in range(N_ASSETS - 1, -1, -1)
))


### Célula 9 — Mapa físico, lógico e financeiro de cada $\theta_j$

**Em termos simples:** esta célula percorre o circuito decomposto e identifica em qual operação física cada parâmetro aparece, quais qubits ele toca e em que posições do circuito ele é usado.

A periodicidade **estrutural da unidade** é definida por:

$$
T_j=
\begin{cases}
4\pi, & \text{se o parâmetro aparece em uma porta CRY},\\
2\pi, & \text{se aparece somente em RY}.
\end{cases}
$$

Uma `RY` muda apenas por uma fase global após $2\pi$. Em uma `CRY`, essa troca de sinal ocorre somente no setor em que o controle vale 1 e pode se tornar uma fase relativa observável; por isso o período seguro da unidade controlada é $4\pi$.

Isso explica por que `theta_2` e `theta_3`, que pertencem a blocos `CY/CRY`, são varridos de zero a $4\pi$. Os demais parâmetros mostrados pertencem a blocos `CCY/RY` e são varridos de zero a $2\pi$.

Entretanto, o observável específico $P(\mathcal{X}_{\mathrm{opt}})$ pode repetir após $2\pi$ mesmo quando a unidade tem período $4\pi$. Por isso a versão 20.6 compara numericamente a primeira e a segunda metade das curvas `CRY` nos 100 vetores.

Os **ativos não mudam durante uma varredura**. Cada $\theta_j$ está ligado a um bloco lógico fixo do ansatz e, portanto, a qubits/ativos fixos. Os nomes são diferentes entre `theta_17`, `theta_25` etc. porque são parâmetros de blocos diferentes, não porque o código esteja trocando ações entre os vetores.

**Saídas principais:** `parameter_map_df`, `parameter_occurrence_df` e `structural_audit`.

In [ ]:
# ============================================================
# 9. MAPA FÍSICO, LÓGICO E FINANCEIRO DOS PARÂMETROS
# ============================================================


def build_physical_parameter_map(ansatz, structure_df):
    """Liga cada theta a operações primitivas, qubits, período e posição no circuito."""
    parameter_order = list(ansatz.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(parameter_order)
    }
    occurrence_rows = []

    # Percorre cada instrução do circuito decomposto e registra todas as ocorrências
    # de parâmetros nas expressões angulares das portas.
    for instruction_position, instruction in enumerate(ansatz.data):
        operation = instruction.operation
        operation_name = str(operation.name).lower()
        qubits = tuple(
            int(ansatz.find_bit(qubit).index)
            for qubit in instruction.qubits
        )
        for parameter_slot, expression in enumerate(operation.params):
            for parameter in getattr(expression, "parameters", set()):
                if parameter not in parameter_to_index:
                    continue
                try:
                    coefficient = float(expression.gradient(parameter))
                except Exception:
                    coefficient = np.nan
                occurrence_rows.append({
                    "theta_index": int(parameter_to_index[parameter]),
                    "parameter_name": str(parameter),
                    "primitive_operation": operation_name,
                    "primitive_qubits": qubits,
                    "instruction_position": int(instruction_position),
                    "parameter_slot": int(parameter_slot),
                    "parameter_coefficient": coefficient,
                })

    # Uma linha por ocorrência física de parâmetro.
    occurrence_df = pd.DataFrame(occurrence_rows)
    physical_rows = []
    for theta_index, group in occurrence_df.groupby("theta_index"):
        operations = sorted(set(group["primitive_operation"].astype(str)))
        primitive_qubits = tuple(sorted({
            int(qubit)
            for qubit_tuple in group["primitive_qubits"]
            for qubit in qubit_tuple
        }))
        # RY(theta+2pi) difere apenas por fase global, mas CRY pode exigir 4pi porque
        # o sinal relativo entre os setores do qubit de controle é observável.
        if "cry" in operations:
            primitive_type = "CRY"
            angular_period = float(4 * np.pi)
            periodicity_class = "four_pi_eligible"
        else:
            primitive_type = "RY"
            angular_period = float(2 * np.pi)
            periodicity_class = "guaranteed_2pi"
        physical_rows.append({
            "theta_index": int(theta_index),
            "primitive_physical_type": primitive_type,
            "primitive_operations": tuple(operations),
            "primitive_parameter_qubits": primitive_qubits,
            "angular_period": angular_period,
            "periodicity_structural_class": periodicity_class,
            "n_occurrences_decomposed": int(len(group)),
            "first_instruction": int(group["instruction_position"].min()),
            "last_instruction": int(group["instruction_position"].max()),
        })

    # Une a descrição lógica do bloco à descrição física observada após decomposição.
    parameter_map = structure_df.merge(
        pd.DataFrame(physical_rows),
        on="theta_index",
        how="left",
        validate="one_to_one",
    ).sort_values("theta_index").reset_index(drop=True)

    def logical_qubits(row):
        """Retorna todos os qubits pertencentes ao bloco lógico CY ou CCY."""
        i_value, l_value = int(row["i"]), int(row["l"])
        if row["ansatz_gate_type"] == "CY":
            return tuple(sorted((i_value, l_value)))
        return tuple(sorted((i_value, i_value + 1, l_value)))

    parameter_map["logical_block_qubits"] = parameter_map.apply(logical_qubits, axis=1)
    parameter_map["logical_assets"] = parameter_map["logical_block_qubits"].map(
        lambda qubits: tuple(tickers[index] for index in qubits)
    )
    parameter_map["block_size"] = parameter_map["logical_block_qubits"].map(len)

    # Auditoria estrutural: qualquer inconsistência interrompe o experimento.
    checks = {
        "all_parameters_mapped": bool(len(parameter_map) == N_PARAMETERS),
        "CY_is_CRY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "primitive_physical_type",
        ].eq("CRY").all()),
        "CCY_is_RY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "primitive_physical_type",
        ].eq("RY").all()),
        "CY_has_two_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "block_size",
        ].eq(2).all()),
        "CCY_has_three_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "block_size",
        ].eq(3).all()),
    }
    failed = [name for name, value in checks.items() if not value]
    if failed:
        raise RuntimeError(f"Auditoria estrutural falhou: {failed}")

    return parameter_map, occurrence_df, checks


parameter_map_df, parameter_occurrence_df, structural_audit = build_physical_parameter_map(
    ansatz, structure_df
)

# Liga cada bloco aos pares clássicos contidos nos seus qubits lógicos.
def pair_metrics_for_block(qubits):
    """Resume os gaps clássicos dos pares de ativos internos ao bloco."""
    pairs = {tuple(sorted(pair)) for pair in combinations(qubits, 2)}
    subset = pair_gap_df.loc[
        pair_gap_df.apply(
            lambda row: tuple(sorted((int(row["asset_index_i"]), int(row["asset_index_j"])))) in pairs,
            axis=1,
        )
    ]
    return pd.Series({
        "n_internal_asset_pairs": int(len(subset)),
        "max_internal_G_ij": float(subset["G_ij"].max()),
        "mean_internal_G_ij": float(subset["G_ij"].mean()),
        "max_internal_joint_gap": float(subset["joint_flip_gap"].max()),
        "max_internal_nonadditivity_abs": float(subset["joint_flip_nonadditivity"].abs().max()),
        "internal_pairs": tuple(subset["pair"].tolist()),
    })

block_pair_metrics = parameter_map_df["logical_block_qubits"].apply(pair_metrics_for_block)
parameter_map_df = pd.concat([parameter_map_df, block_pair_metrics], axis=1)

print(json.dumps(structural_audit, indent=2, ensure_ascii=False))
display(parameter_map_df)


## Auditoria visual da criação do circuito

A tabela anterior é a ponte entre o índice local e a descrição transferível:

- `theta_index`: posição local neste circuito;
- `ansatz_gate_type`: bloco lógico `CY` ou `CCY`;
- `primitive_physical_type`: operação parametrizada observada após decomposição;
- `logical_block_qubits`: qubits realmente envolvidos pelo bloco;
- `logical_assets`: ativos associados a esses qubits;
- `max_internal_G_ij`: maior rigidez clássica entre os pares internos do bloco;
- `angular_period`: domínio máximo usado na varredura.

A célula seguinte mostra o circuito e contabiliza as operações. O estado preparado pelas portas `X` é apenas a semente de peso $k$, não o bitstring ótimo clássico.



### Célula 10 — Auditoria visual e decomposição somente para desenho

**Em termos simples:** esta célula mostra a estrutura simbólica do ansatz e cria uma cópia destinada exclusivamente à visualização.

A cópia visual decompõe as portas `CRY` em rotações `RY` e controles `CX`, mantendo as portas `CCX`. Isso produz um desenho semelhante ao circuito de referência, com:

- portas `X` responsáveis pelo estado inicial;
- rotações `RY` explícitas;
- controles `CX` e `CCX` visíveis;
- nenhuma mudança no objeto `ansatz` usado pelo `Statevector` e pelas varreduras.

Nesta etapa os ângulos ainda são simbólicos. Depois da seleção dos 100 vetores, outra célula atribui valores numéricos a um vetor real e desenha a versão numérica.


In [ ]:

# ============================================================
# 10. VISUALIZAÇÃO SIMBÓLICA — DECOMPOSIÇÃO SOMENTE PARA DESENHO
# ============================================================

# Contagem de portas do circuito que continua sendo usado nas avaliações.
operation_counts_original = pd.DataFrame(
    sorted(ansatz.count_ops().items()),
    columns=["operation", "count"],
)

# Cópia de apresentação: decompõe somente CRY. As portas CCX permanecem visíveis,
# evitando transformar o desenho em uma sequência extensa de H, T e CX.
try:
    ansatz_visual_symbolic = ansatz.decompose(gates_to_decompose=["cry"])
except TypeError:
    # Compatibilidade com versões do Qiskit que aceitam uma string isolada.
    ansatz_visual_symbolic = ansatz.decompose(gates_to_decompose="cry")

operation_counts_visual = pd.DataFrame(
    sorted(ansatz_visual_symbolic.count_ops().items()),
    columns=["operation", "count"],
)

print("PORTAS DO ANSATZ USADO NAS AVALIAÇÕES")
display(operation_counts_original)
print("PORTAS DA CÓPIA DE VISUALIZAÇÃO")
display(operation_counts_visual)

display(parameter_map_df[[
    "theta_index",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_block_qubits",
    "logical_assets",
    "distance",
    "first_instruction",
    "last_instruction",
    "angular_period",
    "internal_pairs",
    "max_internal_G_ij",
]])

# Desenho longo e horizontal, semelhante ao circuito mostrado como referência.
try:
    symbolic_figure = ansatz_visual_symbolic.draw(
        output="mpl",
        fold=-1,
        scale=0.72,
        idle_wires=False,
    )
    display(symbolic_figure)
    symbolic_figure_path = FIGURE_DIR / "ansatz_symbolic_cry_decomposed_visual.png"
    symbolic_figure.savefig(symbolic_figure_path, dpi=180, bbox_inches="tight")
    print("Figura simbólica salva em:", symbolic_figure_path.resolve())
except TypeError:
    # Fallback para versões com assinatura de draw mais restrita.
    symbolic_figure = ansatz_visual_symbolic.draw(output="mpl", fold=-1)
    display(symbolic_figure)
except Exception as exc:
    warnings.warn(
        f"Desenho matplotlib não disponível: {type(exc).__name__}: {exc}"
    )
    print(ansatz_visual_symbolic.draw(output="text", fold=160))


# Parte IV — compatibilidade entre `merge.pkl` e o circuito reconstruído

Antes de interpretar qualquer varredura, o notebook verifica:

1. todos os vetores possuem a dimensão esperada;
2. vetores salvos podem ser atribuídos ao circuito;
3. a distribuição exata recalculada é compatível com as colunas salvas;
4. nenhuma avaliação chama otimizador.


### Célula 11 — Avaliação exata de um vetor $\theta$

**Em termos simples:** esta é a função central usada em todas as intervenções posteriores. Ela recebe um vetor $\theta$, atribui os valores ao ansatz e calcula diretamente o estado quântico:

$$
|\psi(\theta)\rangle=U(\theta)|\psi_0\rangle.
$$

A função `evaluate_theta` devolve, entre outras métricas:

- probabilidade total dos bitstrings ótimos;
- energia esperada $\langle H\rangle$ e gap para a energia exata;
- bitstring dominante;
- massa dentro do subespaço válido de cardinalidade $k$;
- entropia e razão de participação;
- massa acumulada nos 1, 5 e 10 melhores portfólios clássicos.

Quando existe uma distribuição de referência, também são calculadas a distância de variação total

$$
\operatorname{TVD}(p,q)=\frac{1}{2}\sum_z|p_z-q_z|
$$

e a divergência de Jensen–Shannon.

Ao final, uma pequena amostra do banco é reavaliada para verificar compatibilidade entre os vetores salvos e o circuito reconstruído.


In [ ]:
# ============================================================
# 11. FUNÇÕES DE PARSE DOS VETORES E MÉTRICAS EXATAS
# ============================================================

# Converte todos os vetores salvos e mantém apenas aqueles compatíveis com o
# número de parâmetros do ansatz reconstruído.
best_parameters_column = RESOLVED_COLUMNS["best_parameters"]
merge_work_df = merge_df.copy()
merge_work_df["theta_vector"] = merge_work_df[best_parameters_column].map(parse_vector)
merge_work_df["theta_dimension"] = merge_work_df["theta_vector"].map(len)

dimension_counts = merge_work_df["theta_dimension"].value_counts().sort_index()
print("Dimensões encontradas:")
display(dimension_counts.rename_axis("theta_dimension").to_frame("rows"))

merge_work_df = merge_work_df.loc[
    merge_work_df["theta_dimension"].eq(N_PARAMETERS)
].copy()
if merge_work_df.empty:
    raise RuntimeError(
        f"Nenhum vetor do merge possui a dimensão esperada de {N_PARAMETERS} parâmetros."
    )

# Mapeamento entre índices do Statevector e bitstrings na convenção do Qiskit.
all_labels = np.asarray(
    [format(index, f"0{N_ASSETS}b") for index in range(2 ** N_ASSETS)],
    dtype=object,
)
label_to_index = {str(label): int(index) for index, label in enumerate(all_labels)}
valid_bitstrings = enumeration_df["bitstring_qiskit_order"].astype(str).to_numpy(dtype=object)
valid_indices = np.asarray([label_to_index[bitstring] for bitstring in valid_bitstrings], dtype=int)
valid_objectives = enumeration_df["objective"].to_numpy(dtype=float)
optimal_indices = np.asarray([label_to_index[bitstring] for bitstring in exact_qiskit_bitstrings], dtype=int)
optimal_set = set(exact_qiskit_bitstrings)


def total_variation_distance(p, q):
    """Calcula TVD(p,q)=0.5*sum(|p-q|), entre 0 e 1."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(0.5 * np.sum(np.abs(p - q)))


def jensen_shannon_divergence(p, q, epsilon=1e-15):
    """Calcula uma divergência simétrica e finita entre duas distribuições."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(p.sum(), epsilon)
    q = q / max(q.sum(), epsilon)
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + epsilon) / (m + epsilon)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + epsilon) / (m + epsilon)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


def circular_distance(a, b, period):
    """Menor distância entre dois ângulos em um círculo de período conhecido."""
    return float(abs((float(a) - float(b) + 0.5 * period) % period - 0.5 * period))


def shortest_delta_to_target(source, target, period):
    """Deslocamento assinado mais curto da origem até o alvo periódico."""
    return float((float(target) - float(source) + 0.5 * period) % period - 0.5 * period)


def statevector_from_theta(theta):
    """Retorna o vetor de estado complexo após atribuição direta dos parâmetros."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")
    assigned = ansatz.assign_parameters(theta, inplace=False)
    return np.asarray(Statevector.from_instruction(assigned).data, dtype=np.complex128)


def evaluate_theta(
    theta,
    reference_probability=None,
    return_valid_probability=False,
    return_statevector=False,
):
    """Atribui theta ao ansatz e calcula exatamente estado, energia e probabilidades."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")

    # Intervenção direta: não existe passo de otimização entre atribuir theta e avaliar.
    state_data = statevector_from_theta(theta)
    state = Statevector(state_data)
    probability = np.asarray(state.probabilities(), dtype=float)
    # Restringe a distribuição aos C(n,k) bitstrings que respeitam a cardinalidade.
    valid_probability = probability[valid_indices]
    valid_mass = float(valid_probability.sum())
    leakage = max(0.0, 1.0 - valid_mass)
    p_optimal = float(probability[optimal_indices].sum())
    # Energia física completa = valor esperado do operador + offset da conversão QUBO.
    energy = float(np.real(state.expectation_value(ising)) + ising_offset)
    dominant_index = int(np.argmax(probability))
    dominant_bitstring = str(all_labels[dominant_index])

    nonzero = valid_probability[valid_probability > 0.0]
    entropy = float(-np.sum(nonzero * np.log(nonzero)))
    normalized_entropy = float(entropy / np.log(len(valid_probability)))
    participation = float(1.0 / np.sum(valid_probability ** 2))

    dominant_valid_position = int(np.argmax(valid_probability))
    dominant_valid_rank = dominant_valid_position + 1

    result = {
        "p_optimal": p_optimal,
        "expected_energy": energy,
        "energy_gap": float(abs(energy - exact_energy)),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": float(probability[dominant_index]),
        "dominant_is_optimal": bool(dominant_bitstring in optimal_set),
        "dominant_valid_rank": dominant_valid_rank,
        "p_top_1_classical": float(valid_probability[:1].sum()),
        "p_top_5_classical": float(valid_probability[:5].sum()),
        "p_top_10_classical": float(valid_probability[:10].sum()),
        "valid_dicke_mass": valid_mass,
        "leakage_outside_k": leakage,
        "normalized_entropy_valid": normalized_entropy,
        "participation_ratio_valid": participation,
    }
    if reference_probability is not None:
        result["tvd_vs_anchor"] = total_variation_distance(probability, reference_probability)
        result["jsd_vs_anchor"] = jensen_shannon_divergence(probability, reference_probability)
    if return_valid_probability:
        result["valid_probability"] = valid_probability.astype(np.float32)
    if return_statevector:
        result["statevector"] = state_data
    result["full_probability"] = probability
    return result


# Auditoria em uma amostra pequena antes das campanhas longas.
# Compara probabilidade e bitstring salvos com a reavaliação exata atual.
audit_indices = merge_work_df.sample(min(10, len(merge_work_df)), random_state=RANDOM_SEED).index
compatibility_rows = []
for row_index in audit_indices:
    row = merge_work_df.loc[row_index]
    metrics = evaluate_theta(row["theta_vector"])
    saved_p = (
        pd.to_numeric(pd.Series([row[RESOLVED_COLUMNS["p_best"]]]), errors="coerce").iloc[0]
        if RESOLVED_COLUMNS["p_best"] is not None
        else np.nan
    )
    saved_bit = (
        normalize_bitstring(row[RESOLVED_COLUMNS["dominant_bitstring"]])
        if RESOLVED_COLUMNS["dominant_bitstring"] is not None
        else None
    )
    compatibility_rows.append({
        "row_index": row_index,
        "saved_p": saved_p,
        "exact_p_recomputed": metrics["p_optimal"],
        "p_difference": metrics["p_optimal"] - saved_p if np.isfinite(saved_p) else np.nan,
        "saved_dominant": saved_bit,
        "exact_dominant_recomputed": metrics["dominant_bitstring"],
        "dominant_equal": bool(saved_bit == metrics["dominant_bitstring"]) if saved_bit else np.nan,
        "exact_energy_recomputed": metrics["expected_energy"],
        "leakage": metrics["leakage_outside_k"],
    })

compatibility_audit_df = pd.DataFrame(compatibility_rows)
display(compatibility_audit_df)

if compatibility_audit_df["leakage"].max() > 1e-10:
    raise RuntimeError("O circuito reconstruído apresentou vazamento para fora do subespaço k=4.")


### Célula 12 — Máscara dos 10% melhores e seleção de 100 vetores

**Em termos simples:** esta célula reduz o banco aos vetores mais promissores e escolhe 100 vetores completos para a campanha de varredura.

Para cada linha válida são construídos dois rankings percentuais:

- $R_p$: cresce quando a probabilidade salva da solução ótima aumenta;
- $R_{\mathrm{gap}}$: cresce quando o gap de energia diminui.

O score salvo é

$$
Q_{\mathrm{salvo}}=\frac{R_p+R_{\mathrm{gap}}}{2}.
$$

A máscara mantém aproximadamente os 10% superiores. Até 300 candidatos são reavaliados exatamente com `evaluate_theta`, e os 100 melhores pelo score exato são selecionados.

**Importante:** cada âncora é uma linha inteira do banco. A máscara não mistura componentes de vetores diferentes, não zera componentes internos e não muda os ativos associados a cada índice $\theta_j$.

In [ ]:
# ============================================================
# 12. MÁSCARA DOS 10% — 100 VETORES PARA AUDITORIA, 1 PARA VARREDURA
# ============================================================

objective_col = RESOLVED_COLUMNS["objective"]
best_objective_col = RESOLVED_COLUMNS["best_objective"]
p_col = RESOLVED_COLUMNS["p_best"]
gap_col = RESOLVED_COLUMNS["gap"]
status_col = RESOLVED_COLUMNS["status"]

bank = merge_work_df.copy()
bank["saved_p"] = (
    pd.to_numeric(bank[p_col], errors="coerce") if p_col is not None else np.nan
)
bank["saved_objective"] = (
    pd.to_numeric(bank[objective_col], errors="coerce") if objective_col is not None else np.nan
)

if gap_col is not None:
    bank["saved_gap"] = pd.to_numeric(bank[gap_col], errors="coerce").abs()
elif best_objective_col is not None and objective_col is not None:
    bank["saved_gap"] = (
        pd.to_numeric(bank[objective_col], errors="coerce")
        - pd.to_numeric(bank[best_objective_col], errors="coerce")
    ).abs()
else:
    bank["saved_gap"] = (bank["saved_objective"] - exact_energy).abs()

if status_col is not None:
    status_mask = bank[status_col].astype(str).str.lower().eq("ok")
else:
    status_mask = pd.Series(True, index=bank.index)

valid_bank = bank.loc[
    status_mask
    & bank["theta_vector"].notna()
    & bank["saved_gap"].notna()
].copy()

if valid_bank.empty:
    raise RuntimeError("Nenhum vetor válido foi encontrado para a seleção.")

if valid_bank["saved_p"].notna().any():
    valid_bank["p_quality_rank"] = valid_bank["saved_p"].rank(
        pct=True, ascending=True, method="average"
    )
else:
    valid_bank["p_quality_rank"] = 0.5

valid_bank["gap_quality_rank"] = valid_bank["saved_gap"].rank(
    pct=True, ascending=False, method="average"
)
valid_bank["quality_score_saved"] = 0.5 * (
    valid_bank["p_quality_rank"] + valid_bank["gap_quality_rank"]
)

threshold = float(valid_bank["quality_score_saved"].quantile(1.0 - TOP_FRACTION))
top_masked_bank = valid_bank.loc[
    valid_bank["quality_score_saved"].ge(threshold)
].copy()

if len(top_masked_bank) < N_AUDIT_VECTORS:
    raise RuntimeError(
        f"A máscara produziu {len(top_masked_bank)} vetores, mas a auditoria exige "
        f"{N_AUDIT_VECTORS}. Aumente TOP_FRACTION ou verifique o banco."
    )

candidate_count = min(
    max(int(MAX_EXACT_REEVALUATION), int(N_AUDIT_VECTORS)),
    len(top_masked_bank),
)
candidate_pool = top_masked_bank.sort_values(
    ["quality_score_saved", "saved_p", "saved_gap"],
    ascending=[False, False, True],
).head(candidate_count)

exact_candidate_rows = []
for row_index, row in candidate_pool.iterrows():
    row_hash = problem_fingerprint(row)
    if row_hash != DATA_HASH:
        raise RuntimeError(
            f"A linha {row_index} pertence a outro Hamiltoniano: {row_hash} != {DATA_HASH}."
        )
    metrics = evaluate_theta(row["theta_vector"])
    exact_candidate_rows.append({
        "source_row_index": row_index,
        "problem_hash": row_hash,
        "theta_vector": np.asarray(row["theta_vector"], dtype=float).copy(),
        "saved_p": row["saved_p"],
        "saved_gap": row["saved_gap"],
        "quality_score_saved": row["quality_score_saved"],
        **{key: value for key, value in metrics.items() if key != "full_probability"},
    })

exact_candidates_df = pd.DataFrame(exact_candidate_rows)
exact_candidates_df["p_rank_exact"] = exact_candidates_df["p_optimal"].rank(
    pct=True, ascending=True
)
exact_candidates_df["gap_rank_exact"] = exact_candidates_df["energy_gap"].rank(
    pct=True, ascending=False
)
exact_candidates_df["quality_score_exact"] = 0.5 * (
    exact_candidates_df["p_rank_exact"] + exact_candidates_df["gap_rank_exact"]
)

# Os 100 melhores são mantidos apenas para auditar a distribuição dos vetores.
audit_anchors_df = exact_candidates_df.sort_values(
    ["quality_score_exact", "p_optimal", "energy_gap"],
    ascending=[False, False, True],
).head(N_AUDIT_VECTORS).reset_index(drop=True)
audit_anchors_df.insert(
    0, "audit_vector_id", np.arange(len(audit_anchors_df), dtype=int)
)

if len(audit_anchors_df) != N_AUDIT_VECTORS:
    raise RuntimeError(
        f"Esperados {N_AUDIT_VECTORS} vetores de auditoria; "
        f"encontrados {len(audit_anchors_df)}."
    )
if audit_anchors_df["problem_hash"].nunique() != 1:
    raise RuntimeError("Os vetores de auditoria não pertencem ao mesmo Hamiltoniano.")

# Para a varredura, usa apenas o vetor de maior p_optimal; em empate, menor gap.
anchors_df = audit_anchors_df.sort_values(
    ["p_optimal", "energy_gap", "quality_score_exact", "audit_vector_id"],
    ascending=[False, True, False, True],
).head(1).copy().reset_index(drop=True)
anchors_df.insert(0, "anchor_id", np.arange(len(anchors_df), dtype=int))

if len(anchors_df) != 1:
    raise RuntimeError("A versão 20.11 exige exatamente um vetor para a varredura.")

# -----------------------------------------------------------------
# Auditoria dedicada da distribuição de theta_17 entre os 100 vetores
# -----------------------------------------------------------------
audit_theta_matrix = np.vstack(audit_anchors_df["theta_vector"].to_list())
if audit_theta_matrix.shape != (N_AUDIT_VECTORS, N_PARAMETERS):
    raise RuntimeError(
        f"Matriz theta de auditoria com forma {audit_theta_matrix.shape}; "
        f"esperado {(N_AUDIT_VECTORS, N_PARAMETERS)}."
    )

theta17_values = audit_theta_matrix[:, 17].astype(float)
theta17_audit_df = pd.DataFrame({
    "audit_vector_id": audit_anchors_df["audit_vector_id"].to_numpy(dtype=int),
    "source_row_index": audit_anchors_df["source_row_index"].to_numpy(),
    "p_optimal": audit_anchors_df["p_optimal"].to_numpy(dtype=float),
    "theta_17_raw": theta17_values,
    "theta_17_canonical_2pi": np.mod(theta17_values, 2 * np.pi),
})

theta17_summary_df = pd.DataFrame([{
    "n_vectors": int(len(theta17_values)),
    "theta_17_raw_min": float(np.min(theta17_values)),
    "theta_17_raw_max": float(np.max(theta17_values)),
    "theta_17_raw_range": float(np.ptp(theta17_values)),
    "theta_17_raw_mean": float(np.mean(theta17_values)),
    "theta_17_raw_std": float(np.std(theta17_values, ddof=1)),
    "n_unique_raw_3dp": int(pd.Series(theta17_values).round(3).nunique()),
    "n_unique_raw_6dp": int(pd.Series(theta17_values).round(6).nunique()),
    "n_unique_raw_9dp": int(pd.Series(theta17_values).round(9).nunique()),
    "all_equal_at_3dp": bool(pd.Series(theta17_values).round(3).nunique() == 1),
    "all_equal_at_6dp": bool(pd.Series(theta17_values).round(6).nunique() == 1),
}])

print("SELEÇÃO DO EXPERIMENTO")
print("Vetores mantidos para auditoria:", len(audit_anchors_df))
print("Vetores usados na varredura:", len(anchors_df))
display(anchors_df.drop(columns=["theta_vector"], errors="ignore"))

print("AUDITORIA DE theta_17 NOS 100 VETORES")
display(theta17_summary_df)
display(theta17_audit_df.sort_values("theta_17_raw").reset_index(drop=True))

audit_anchors_df.to_pickle(TABLE_DIR / "audit_100_selected_vectors.pkl")
theta17_audit_df.to_csv(TABLE_DIR / "theta17_values_100_vectors.csv", index=False)
theta17_summary_df.to_csv(TABLE_DIR / "theta17_summary_100_vectors.csv", index=False)



### Célula 12B — Circuito numérico no estilo da figura de referência

Esta célula escolhe, apenas para apresentação, o vetor com maior `p_optimal` entre os 100 vetores selecionados. Em seguida:

1. atribui os 30 valores de $\theta$ ao ansatz original;
2. decompõe somente as portas `CRY` em `RY` e `CX`;
3. mantém `CCX`, `CX`, `RY` e as portas `X` iniciais visíveis;
4. desenha o circuito horizontalmente, com os valores numéricos das rotações;
5. calcula a fidelidade entre o circuito original ligado e a cópia de visualização.

Essa célula **não substitui `ansatz`**, não altera `anchors_df` e não muda nenhuma varredura.


In [ ]:

# ============================================================
# 12B. CIRCUITO NUMÉRICO DECOMPOSTO — SOMENTE VISUALIZAÇÃO
# ============================================================

# Escolhe uma âncora representativa: maior probabilidade ótima exata e, em caso
# de empate, menor gap energético e menor anchor_id.
visual_anchor_row = anchors_df.sort_values(
    ["p_optimal", "energy_gap", "anchor_id"],
    ascending=[False, True, True],
).iloc[0]

visual_anchor_id = int(visual_anchor_row["anchor_id"])
visual_theta = np.asarray(visual_anchor_row["theta_vector"], dtype=float).copy()

if visual_theta.shape != (N_PARAMETERS,):
    raise RuntimeError(
        f"Vetor visual com dimensão {visual_theta.shape}; esperado ({N_PARAMETERS},)."
    )

# Circuito original numericamente ligado. Ele não é modificado in-place.
visual_bound_original = ansatz.assign_parameters(visual_theta, inplace=False)

# Cópia visual: decompõe apenas CRY -> RY/CX, preservando CCX.
try:
    visual_bound_decomposed = visual_bound_original.decompose(
        gates_to_decompose=["cry"]
    )
except TypeError:
    visual_bound_decomposed = visual_bound_original.decompose(
        gates_to_decompose="cry"
    )

# Auditoria física: as duas representações devem produzir o mesmo estado,
# salvo diferenças numéricas e uma possível fase global.
state_original_visual = np.asarray(
    Statevector.from_instruction(visual_bound_original).data,
    dtype=np.complex128,
)
state_decomposed_visual = np.asarray(
    Statevector.from_instruction(visual_bound_decomposed).data,
    dtype=np.complex128,
)
visual_state_fidelity = float(
    abs(np.vdot(state_original_visual, state_decomposed_visual)) ** 2
)

if not np.isclose(visual_state_fidelity, 1.0, atol=1e-10, rtol=1e-10):
    raise RuntimeError(
        "A decomposição destinada ao desenho alterou o estado: "
        f"fidelidade={visual_state_fidelity:.16f}."
    )

visual_anchor_summary_df = pd.DataFrame([{
    "anchor_id": visual_anchor_id,
    "source_row_index": visual_anchor_row["source_row_index"],
    "p_optimal": float(visual_anchor_row["p_optimal"]),
    "energy_gap": float(visual_anchor_row["energy_gap"]),
    "n_parameters": int(N_PARAMETERS),
    "state_fidelity_original_vs_visual": visual_state_fidelity,
    "visual_only": True,
}])

print("VETOR USADO SOMENTE PARA DESENHAR O CIRCUITO NUMÉRICO")
display(visual_anchor_summary_df)
print("Valores theta usados no desenho:")
display(pd.DataFrame({
    "theta_index": np.arange(N_PARAMETERS, dtype=int),
    "theta_value": visual_theta,
    "theta_over_pi": visual_theta / np.pi,
}))

print("Contagem de portas após decompor CRY para a figura:")
display(pd.DataFrame(
    sorted(visual_bound_decomposed.count_ops().items()),
    columns=["operation", "count"],
))

try:
    numeric_circuit_figure = visual_bound_decomposed.draw(
        output="mpl",
        fold=-1,
        scale=0.72,
        idle_wires=False,
    )
    display(numeric_circuit_figure)
    numeric_circuit_path = FIGURE_DIR / (
        f"ansatz_numeric_anchor_{visual_anchor_id:03d}_cry_decomposed.png"
    )
    numeric_circuit_figure.savefig(
        numeric_circuit_path,
        dpi=180,
        bbox_inches="tight",
    )
    print("Figura numérica salva em:", numeric_circuit_path.resolve())
except TypeError:
    numeric_circuit_figure = visual_bound_decomposed.draw(
        output="mpl",
        fold=-1,
    )
    display(numeric_circuit_figure)
except Exception as exc:
    warnings.warn(
        f"Desenho matplotlib não disponível: {type(exc).__name__}: {exc}"
    )
    print(visual_bound_decomposed.draw(output="text", fold=160))

visual_anchor_summary_df.to_csv(
    TABLE_DIR / "numeric_circuit_visualization_anchor_summary.csv",
    index=False,
)



# Parte V — varredura individual em um único vetor

Os 100 vetores selecionados continuam disponíveis em `audit_anchors_df`, mas não são todos varridos. A varredura utiliza somente o vetor representativo salvo em `anchors_df`.

Para cada parâmetro testado, apenas um componente é alterado. Os outros 29 parâmetros permanecem exatamente iguais aos valores do vetor escolhido.

Parâmetros detalhados:

$$
\{17,2,14,19,22,25,27,3,24\}.
$$

Não existe média, mediana ou envelope entre vetores nesta versão. Cada curva pertence ao mesmo vetor completo.



### Célula 13 — varredura simétrica de um parâmetro por vez no vetor escolhido

Para o único vetor $\boldsymbol{\theta}^{(0)}\in\mathbb{R}^{30}$ e para cada índice $j\in\mathcal{J}$, a célula avalia

$$
\boldsymbol{\theta}^{(0,j)}(\phi)
=
\bigl(
\theta_0^{(0)},\ldots,\theta_{j-1}^{(0)},
\phi,
\theta_{j+1}^{(0)},\ldots,\theta_{29}^{(0)}
\bigr).
$$

A janela é simétrica:

- `RY`: $-2\pi\leq\phi\leq2\pi$;
- `CRY`: $-4\pi\leq\phi\leq4\pi$.

O ponto original bruto é inserido exatamente na grade. Os checkpoints antigos não são reutilizados, porque a identidade do vetor de varredura é auditada nesta versão.


In [ ]:
# ============================================================
# 13. MOTOR DE VARREDURA INDIVIDUAL — UM VETOR, SEM COBYLA
# ============================================================


def inclusive_grid(start, end, step=None, n_points=None):
    """Cria uma grade que inclui explicitamente os limites start e end."""
    start, end = float(start), float(end)
    if step is not None:
        values = np.arange(start, end, float(step), dtype=float)
        if len(values) == 0 or not np.isclose(values[-1], end, atol=1e-12, rtol=0.0):
            values = np.append(values, end)
        else:
            values[-1] = end
        return values
    if n_points is None or n_points < 2:
        raise ValueError("Informe step ou n_points >= 2.")
    return np.linspace(start, end, int(n_points), endpoint=True)


def parameter_period(theta_index):
    """Recupera em parameter_map_df o período estrutural 2pi ou 4pi."""
    row = parameter_map_df.loc[
        parameter_map_df["theta_index"].eq(int(theta_index))
    ]
    if len(row) != 1:
        raise KeyError(f"theta_{theta_index} não foi identificado de forma única.")
    return float(row.iloc[0]["angular_period"])


def parameter_sweep_bounds(theta_index):
    """Usa uma janela simétrica [-T_j, T_j] para visualizar senos/cossenos."""
    period = parameter_period(theta_index)
    return float(-period), float(period)


def run_single_parameter_task(
    anchor_row,
    theta_index,
    n_points=SWEEP_POINTS_PER_PERIOD,
):
    """Varre um theta de uma âncora, mantendo todos os demais componentes fixos."""
    anchor_id = int(anchor_row["anchor_id"])
    theta_index = int(theta_index)
    theta_anchor = np.asarray(anchor_row["theta_vector"], dtype=float).copy()

    period = parameter_period(theta_index)
    original_raw = float(theta_anchor[theta_index])
    original_canonical = float(original_raw % period)

    # Grade regular na janela simétrica [-T_j, T_j] + valor original inserido exatamente.
    sweep_start, sweep_end = parameter_sweep_bounds(theta_index)
    sweep_span = float(sweep_end - sweep_start)
    base_grid = inclusive_grid(sweep_start, sweep_end, n_points=int(n_points))
    grid = np.unique(np.append(base_grid, original_raw))
    grid.sort()
    original_grid_index = int(np.argmin(np.abs(grid - original_raw)))
    if not np.isclose(
        grid[original_grid_index], original_raw, atol=1e-14, rtol=0.0
    ):
        raise RuntimeError(f"Não foi possível inserir o ponto original bruto de theta_{theta_index}.")

    task_stem = f"anchor_{anchor_id:03d}_theta_{theta_index:02d}"
    summary_path = CHECKPOINT_DIR / f"detailed_{task_stem}.pkl"
    distribution_path = CHECKPOINT_DIR / f"detailed_{task_stem}_valid_probabilities.npz"

    # Procura primeiro na versão 20.7 e depois na 20.6. O conteúdo das
    # varreduras é compatível; apenas os resumos e gráficos foram corrigidos.
    if REUSE_SWEEP_CHECKPOINTS:
        candidate_directories = [CHECKPOINT_DIR]
        if LEGACY_CHECKPOINT_DIR != CHECKPOINT_DIR:
            candidate_directories.append(LEGACY_CHECKPOINT_DIR)

        for candidate_directory in candidate_directories:
            candidate_summary = candidate_directory / f"detailed_{task_stem}.pkl"
            candidate_distribution = (
                candidate_directory / f"detailed_{task_stem}_valid_probabilities.npz"
            )
            if not (candidate_summary.exists() and candidate_distribution.exists()):
                continue

            cached = pd.read_pickle(candidate_summary)
            required = {
                "anchor_id", "theta_index", "is_original_theta", "parameter_name",
                "theta_value", "p_optimal", "period",
            }
            if required.issubset(cached.columns):
                return cached, candidate_distribution

    anchor_metrics = evaluate_theta(theta_anchor)
    reference_probability = anchor_metrics["full_probability"]
    parameter_name = str(list(ansatz.parameters)[theta_index])

    rows = []
    probability_matrix = np.empty((len(grid), len(valid_indices)), dtype=np.float32)

    for grid_index, value in enumerate(grid):
        theta_test = theta_anchor.copy()
        theta_test[theta_index] = float(value)

        changed_indices = np.flatnonzero(
            ~np.isclose(theta_test, theta_anchor, atol=1e-14, rtol=0.0)
        )
        is_original = bool(grid_index == original_grid_index)
        if is_original:
            if len(changed_indices) != 0 and not (
                len(changed_indices) == 1 and int(changed_indices[0]) == theta_index
            ):
                raise RuntimeError(
                    f"Auditoria falhou no ponto original de theta_{theta_index}, "
                    f"âncora {anchor_id}: índices alterados = {changed_indices.tolist()}"
                )
        elif changed_indices.tolist() != [theta_index]:
            raise RuntimeError(
                f"Auditoria falhou em theta_{theta_index}, âncora {anchor_id}: "
                f"índices alterados = {changed_indices.tolist()}"
            )

        metrics = evaluate_theta(
            theta_test,
            reference_probability=reference_probability,
            return_valid_probability=True,
        )
        probability_matrix[grid_index] = metrics.pop("valid_probability")
        metrics.pop("full_probability")

        rows.append({
            "anchor_id": anchor_id,
            "source_row_index": anchor_row["source_row_index"],
            "theta_index": theta_index,
            "parameter_name": parameter_name,
            "grid_index": int(grid_index),
            "theta_value": float(value),
            "theta_over_period": float(value / period),
            "theta_over_pi": float(value / np.pi),
            "theta_over_sweep_window": float((value - sweep_start) / sweep_span),
            "sweep_start": float(sweep_start),
            "sweep_end": float(sweep_end),
            "sweep_span": float(sweep_span),
            "period": period,
            "anchor_theta_raw": original_raw,
            "anchor_theta_canonical": original_canonical,
            "is_original_theta": is_original,
            "distance_to_anchor": circular_distance(value, original_raw, period),
            **metrics,
            "optimizer_used": False,
        })

    task_df = pd.DataFrame(rows)

    original_rows = task_df.loc[task_df["is_original_theta"]]
    if len(original_rows) != 1:
        raise RuntimeError(
            f"Esperado um ponto original para theta_{theta_index}, âncora {anchor_id}; "
            f"encontrados {len(original_rows)}."
        )
    original_p = float(original_rows.iloc[0]["p_optimal"])
    if not np.isclose(original_p, anchor_metrics["p_optimal"], atol=1e-12, rtol=1e-10):
        raise RuntimeError(
            f"O ponto original de theta_{theta_index}, âncora {anchor_id}, não reproduziu "
            f"P(X_opt): {original_p} versus {anchor_metrics['p_optimal']}"
        )

    p_min = float(task_df["p_optimal"].min())
    p_max = float(task_df["p_optimal"].max())
    span = p_max - p_min
    flat_threshold = max(
        SWEEP_FLAT_ABS_TOL,
        SWEEP_FLAT_REL_TOL * max(abs(p_max), 1.0),
    )
    task_df["p_span"] = span
    task_df["flat_threshold"] = flat_threshold
    task_df["is_flat_sweep"] = bool(span <= flat_threshold)

    task_df.to_pickle(summary_path)
    np.savez_compressed(
        distribution_path,
        valid_probability=probability_matrix,
        theta_grid=grid,
        valid_bitstrings=valid_bitstrings,
        valid_objectives=valid_objectives,
    )
    return task_df, distribution_path


if len(anchors_df) != N_ANCHORS:
    raise RuntimeError(
        f"A campanha exige {N_ANCHORS} vetores selecionados; encontrados {len(anchors_df)}."
    )

for theta_index in DETAILED_THETA_INDICES:
    if not 0 <= theta_index < N_PARAMETERS:
        raise IndexError(f"theta_{theta_index} não existe no circuito com {N_PARAMETERS} parâmetros.")

all_detailed_frames = []
detailed_distribution_paths = {}
total_tasks = len(anchors_df) * len(DETAILED_THETA_INDICES)
completed_tasks = 0

for _, anchor_row in anchors_df.iterrows():
    anchor_id = int(anchor_row["anchor_id"])
    for theta_index in DETAILED_THETA_INDICES:
        task_df, distribution_path = run_single_parameter_task(anchor_row, theta_index)
        all_detailed_frames.append(task_df)
        detailed_distribution_paths[(anchor_id, int(theta_index))] = distribution_path
        completed_tasks += 1
        if completed_tasks == 1 or completed_tasks % 25 == 0 or completed_tasks == total_tasks:
            print(
                f"tarefas concluídas: {completed_tasks}/{total_tasks} | "
                f"anchor={anchor_id:03d} | theta_{theta_index}"
            )

individual_sweep_df = pd.concat(all_detailed_frames, ignore_index=True)
individual_sweep_path = TABLE_DIR / "individual_sweeps_single_vector.pkl"
individual_sweep_df.to_pickle(individual_sweep_path)

print("Âncoras avaliadas:", individual_sweep_df["anchor_id"].nunique())
print("Thetas avaliados:", sorted(individual_sweep_df["theta_index"].unique().tolist()))
print("Total de avaliações detalhadas:", len(individual_sweep_df))
print("Tabela salva em:", individual_sweep_path.resolve())


### Célula 14 — curvas brutas do único vetor e auditoria de `theta_17`

Esta célula não reúne 100 curvas. Para cada parâmetro, ela mostra apenas a curva do vetor escolhido:

- eixo horizontal bruto e simétrico;
- ponto original bruto;
- mínimo e máximo da curva;
- linha de referência em $P=0.90$;
- nenhuma normalização min–max;
- nenhuma mediana ou envelope entre vetores.

A tabela de `theta_17` construída na Célula 12 continua mostrando os valores provenientes dos 100 vetores, permitindo verificar se a aparente igualdade é exata ou apenas aproximada.


In [ ]:
# ============================================================
# 14. CURVAS BRUTAS DO ÚNICO VETOR E EXTREMOS
# ============================================================

if individual_sweep_df["anchor_id"].nunique() != 1:
    raise RuntimeError(
        "A versão 20.11 espera exatamente um vetor no resultado da varredura."
    )

parameter_order = list(ansatz.parameters)
if len(parameter_order) != N_PARAMETERS:
    raise RuntimeError("A ordem dos parâmetros do ansatz está inconsistente.")

selected_parameter_map_df = parameter_map_df.loc[
    parameter_map_df["theta_index"].isin(DETAILED_THETA_INDICES)
].copy().sort_values("theta_index")
selected_parameter_map_df["parameter_name_from_ansatz"] = selected_parameter_map_df[
    "theta_index"
].map(lambda index: str(parameter_order[int(index)]))

mapping_audit_df = selected_parameter_map_df[[
    "theta_index",
    "parameter_name_from_ansatz",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_block_qubits",
    "logical_assets",
    "angular_period",
]].copy()
mapping_audit_df["period_over_pi"] = mapping_audit_df["angular_period"] / np.pi
mapping_audit_df["mapping_depends_on_anchor"] = False

display(mapping_audit_df)
mapping_audit_df.to_csv(TABLE_DIR / "fixed_theta_asset_mapping.csv", index=False)

original_points_df = individual_sweep_df.loc[
    individual_sweep_df["is_original_theta"]
].copy()
if len(original_points_df) != len(DETAILED_THETA_INDICES):
    raise RuntimeError(
        f"Esperados {len(DETAILED_THETA_INDICES)} pontos originais; "
        f"encontrados {len(original_points_df)}."
    )

anchor_probability_unique_df = pd.DataFrame([{
    "anchor_id": int(original_points_df["anchor_id"].iloc[0]),
    "p_original": float(original_points_df["p_optimal"].mean()),
}])

# Extremos de cada uma das nove curvas.
sweep_extrema_rows = []
for theta_index, group in individual_sweep_df.groupby("theta_index", sort=True):
    group = group.sort_values("theta_value").copy()
    original_row = group.loc[group["is_original_theta"]]
    if len(original_row) != 1:
        raise RuntimeError(f"Ponto original inconsistente para theta_{theta_index}.")
    original_row = original_row.iloc[0]
    min_row = group.loc[group["p_optimal"].idxmin()]
    max_row = group.loc[group["p_optimal"].idxmax()]
    sweep_extrema_rows.append({
        "anchor_id": int(group["anchor_id"].iloc[0]),
        "theta_index": int(theta_index),
        "period": float(group["period"].iloc[0]),
        "sweep_start": float(group["sweep_start"].iloc[0]),
        "sweep_end": float(group["sweep_end"].iloc[0]),
        "theta_original_raw": float(original_row["anchor_theta_raw"]),
        "p_original": float(original_row["p_optimal"]),
        "theta_at_p_min": float(min_row["theta_value"]),
        "p_sweep_min": float(min_row["p_optimal"]),
        "theta_at_p_max": float(max_row["theta_value"]),
        "p_sweep_max": float(max_row["p_optimal"]),
        "p_sweep_range": float(max_row["p_optimal"] - min_row["p_optimal"]),
        "is_flat_sweep": bool(group["is_flat_sweep"].iloc[0]),
    })

sweep_extrema_per_vector_df = pd.DataFrame(sweep_extrema_rows)
display(sweep_extrema_per_vector_df)
sweep_extrema_per_vector_df.to_csv(
    TABLE_DIR / "single_vector_sweep_extrema.csv", index=False
)

# Figura principal: uma curva por theta, usando diretamente theta_value bruto.
n_theta = len(DETAILED_THETA_INDICES)
n_cols = 3
n_rows = int(np.ceil(n_theta / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 4.8 * n_rows), squeeze=False)
axes_flat = axes.ravel()

for panel_index, theta_index in enumerate(DETAILED_THETA_INDICES):
    ax = axes_flat[panel_index]
    group = individual_sweep_df.loc[
        individual_sweep_df["theta_index"].eq(theta_index)
    ].sort_values("theta_value").copy()
    map_row = selected_parameter_map_df.loc[
        selected_parameter_map_df["theta_index"].eq(theta_index)
    ].iloc[0]

    period = float(group["period"].iloc[0])
    sweep_start = float(group["sweep_start"].iloc[0])
    sweep_end = float(group["sweep_end"].iloc[0])
    x = group["theta_value"].to_numpy(dtype=float)
    y = group["p_optimal"].to_numpy(dtype=float)
    original_row = group.loc[group["is_original_theta"]].iloc[0]
    extrema_row = sweep_extrema_per_vector_df.loc[
        sweep_extrema_per_vector_df["theta_index"].eq(theta_index)
    ].iloc[0]

    ax.plot(x, y, linewidth=1.8, label="varredura do vetor único")
    ax.scatter(
        [float(original_row["anchor_theta_raw"])],
        [float(original_row["p_optimal"])],
        s=38,
        zorder=5,
        label="valor original bruto",
    )
    ax.scatter(
        [float(extrema_row["theta_at_p_max"])],
        [float(extrema_row["p_sweep_max"])],
        marker="^",
        s=38,
        zorder=5,
        label="máximo",
    )
    ax.scatter(
        [float(extrema_row["theta_at_p_min"])],
        [float(extrema_row["p_sweep_min"])],
        marker="v",
        s=38,
        zorder=5,
        label="mínimo",
    )
    ax.axhline(0.90, linestyle="--", linewidth=1.0, alpha=0.7)
    ax.axvline(0.0, linestyle=":", linewidth=1.0, alpha=0.7)

    if np.isclose(period, 4 * np.pi):
        ticks = [-4*np.pi, -2*np.pi, 0.0, 2*np.pi, 4*np.pi]
        tick_labels = [r"$-4\pi$", r"$-2\pi$", "0", r"$2\pi$", r"$4\pi$"]
    else:
        ticks = [-2*np.pi, -np.pi, 0.0, np.pi, 2*np.pi]
        tick_labels = [r"$-2\pi$", r"$-\pi$", "0", r"$\pi$", r"$2\pi$"]

    ax.set_xticks(ticks, tick_labels)
    ax.set_xlim(sweep_start, sweep_end)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel(f"valor bruto de theta_{theta_index} (rad)")
    ax.set_ylabel(r"$P(\mathcal{X}_{\mathrm{opt}})$")
    ax.set_title(
        f"theta_{theta_index} | {map_row['ansatz_gate_type']}/"
        f"{map_row['primitive_physical_type']} | janela="
        f"[{sweep_start/np.pi:.0f}π,{sweep_end/np.pi:.0f}π]\n"
        f"qubits={map_row['logical_block_qubits']} | "
        f"ativos={map_row['logical_assets']}"
    )
    ax.grid(alpha=0.25)
    if panel_index == 0:
        ax.legend(loc="best", fontsize=8)

for unused_index in range(n_theta, len(axes_flat)):
    axes_flat[unused_index].axis("off")

focus_anchor = anchors_df.iloc[0]
fig.suptitle(
    "Varredura individual em um único vetor — eixo bruto e simétrico\n"
    f"anchor_id={int(focus_anchor['anchor_id'])} | "
    f"source_row_index={focus_anchor['source_row_index']} | "
    f"p_original={float(focus_anchor['p_optimal']):.6f}",
    fontsize=15,
    y=1.005,
)
fig.tight_layout()
single_vector_figure_path = FIGURE_DIR / "single_vector_symmetric_raw_sweeps.png"
fig.savefig(single_vector_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Figura salva em:", single_vector_figure_path.resolve())

# Auditoria de domínio: nenhuma curva pode começar em zero nesta versão.
domain_audit_df = individual_sweep_df.groupby("theta_index").agg(
    observed_theta_min=("theta_value", "min"),
    observed_theta_max=("theta_value", "max"),
    configured_sweep_start=("sweep_start", "first"),
    configured_sweep_end=("sweep_end", "first"),
).reset_index()
domain_audit_df["min_matches_configuration"] = np.isclose(
    domain_audit_df["observed_theta_min"],
    domain_audit_df["configured_sweep_start"],
    atol=1e-12,
    rtol=0.0,
)
domain_audit_df["max_matches_configuration"] = np.isclose(
    domain_audit_df["observed_theta_max"],
    domain_audit_df["configured_sweep_end"],
    atol=1e-12,
    rtol=0.0,
)
if not domain_audit_df[[
    "min_matches_configuration", "max_matches_configuration"
]].all().all():
    raise RuntimeError("O domínio observado não corresponde à janela simétrica configurada.")

display(domain_audit_df)
domain_audit_df.to_csv(TABLE_DIR / "single_vector_sweep_domain_audit.csv", index=False)


# Parte VI — teste coletivo e cumulativo dos 23 $\theta$ não ativos

## Objetivo do experimento

O circuito possui 30 parâmetros. Nesta etapa:

- os **7 parâmetros ativos** já definidos no notebook (`ACTIVE_THETA_INDICES`) são colocados nos seus **máximos individuais** encontrados na varredura (`theta_at_p_max`);
- os outros **23 parâmetros** são tratados como o conjunto a testar;
- nenhuma otimização é executada nesta parte.

> **Importante:** “melhor ponto” aqui significa o máximo obtido na varredura **individual** de cada um dos 7 parâmetros. Colocar os 7 máximos juntos não é uma nova otimização conjunta; por isso esse vetor é primeiro medido e passa a ser a referência do teste.

### Experimento A — intervenção coletiva

Partindo da referência com os 7 parâmetros fixos, os outros 23 são todos forçados para

\[
\theta_j=\pi/2.
\]

Depois do `assign_parameters`, medimos novamente:

- \(P(\mathcal{X}_{opt})\);
- energia esperada;
- bitstring dominante e sua probabilidade;
- distância de variação total (TVD) da distribuição;
- fidelidade com o statevector de referência.

### Experimento B — intervenção cumulativa aleatória

A ordem dos 23 índices é embaralhada com uma semente fixa. O valor de intervenção continua sendo \(\pi/2\), para não misturar o efeito de **qual parâmetro foi alterado** com o efeito de **qual valor foi escolhido**.

Começamos outra vez no vetor de referência:

1. força 1 dos 23 parâmetros para \(\pi/2\) e mede;
2. mantém essa alteração, força um segundo parâmetro e mede;
3. continua cumulativamente até os 23.

A última linha obrigatoriamente deve reproduzir o Experimento A.


In [ ]:
# ============================================================
# 17. REFERÊNCIA: 7 THETAS NOS MÁXIMOS INDIVIDUAIS
# ============================================================

INTERVENTION_VALUE = np.pi / 2
CHANGE_TOL = 1e-10

# Os 23 testados são exatamente o complemento dos 7 índices ativos.
INSENSITIVE_THETA_INDICES = sorted(
    set(range(N_PARAMETERS)) - set(ACTIVE_THETA_INDICES)
)
if len(ACTIVE_THETA_INDICES) != 7 or len(INSENSITIVE_THETA_INDICES) != 23:
    raise RuntimeError("A divisão esperada é 7 parâmetros ativos + 23 parâmetros testados.")

# Começa no MESMO vetor usado como âncora pela varredura.
theta_reference = np.asarray(anchors_df.iloc[0]["theta_vector"], dtype=float).copy()

# Substitui somente os 7 ativos pelos máximos das respectivas varreduras 1D.
best7 = sweep_extrema_per_vector_df.loc[
    sweep_extrema_per_vector_df["theta_index"].isin(ACTIVE_THETA_INDICES),
    ["theta_index", "theta_at_p_max", "p_sweep_max"],
].copy().sort_values("theta_index")

if set(best7["theta_index"].astype(int)) != set(ACTIVE_THETA_INDICES):
    raise RuntimeError("Nem todos os 7 theta ativos possuem theta_at_p_max disponível.")

for row in best7.itertuples(index=False):
    theta_reference[int(row.theta_index)] = float(row.theta_at_p_max)

# Auditoria: esta tabela mostra exatamente quais 7 valores ficaram fixos.
best7 = best7.rename(columns={"theta_at_p_max": "theta_fixed_in_reference"})
display(best7.reset_index(drop=True))

# Se algum dos 23 já estiver em pi/2 na referência, ele será selecionado pela ordem
# aleatória, mas esse passo específico não produzirá deslocamento numérico.
already_at_target = [
    i for i in INSENSITIVE_THETA_INDICES
    if np.isclose(theta_reference[i], INTERVENTION_VALUE, atol=1e-14, rtol=0.0)
]
print("23 theta testados:", INSENSITIVE_THETA_INDICES)
print("Já estavam em pi/2 na referência:", already_at_target)


In [ ]:
# ============================================================
# 18. MEDIÇÃO DIRETA — ASSIGN_PARAMETERS VISÍVEL
# ============================================================

def measure_theta(theta):
    """Atribui o vetor ao ansatz e mede somente o necessário para este experimento."""
    theta = np.asarray(theta, dtype=float)

    # LINHA-CHAVE: os 30 valores são realmente atribuídos ao circuito, sem otimizador.
    assigned = ansatz.assign_parameters(theta, inplace=False)

    # O Statevector é calculado diretamente do circuito já parametrizado.
    state = Statevector.from_instruction(assigned)
    probabilities = np.asarray(state.probabilities(), dtype=float)

    dominant_index = int(np.argmax(probabilities))
    return {
        "p_optimal": float(probabilities[optimal_indices].sum()),
        "expected_energy": float(np.real(state.expectation_value(ising)) + ising_offset),
        "dominant_bitstring": str(all_labels[dominant_index]),
        "dominant_probability": float(probabilities[dominant_index]),
        "probabilities": probabilities,
        "statevector": np.asarray(state.data, dtype=np.complex128),
    }


reference_metrics = measure_theta(theta_reference)
reference_probability = reference_metrics["probabilities"]
reference_statevector = reference_metrics["statevector"]


def comparison_row(label, theta, n_modified, last_theta=None, modified_indices=()):
    """Compara uma intervenção com a referência dos 7 máximos individuais."""
    metrics = measure_theta(theta)
    probability_delta = metrics["probabilities"] - reference_probability

    # TVD detecta qualquer redistribuição de probabilidade, mesmo sem trocar o bitstring dominante.
    tvd = float(0.5 * np.abs(probability_delta).sum())

    # Fidelidade detecta também mudanças do estado que podem ficar escondidas nas probabilidades.
    fidelity = float(abs(np.vdot(reference_statevector, metrics["statevector"])) ** 2)

    delta_p = float(metrics["p_optimal"] - reference_metrics["p_optimal"])
    delta_e = float(metrics["expected_energy"] - reference_metrics["expected_energy"])
    bit_changed = bool(metrics["dominant_bitstring"] != reference_metrics["dominant_bitstring"])
    actually_changed = np.flatnonzero(
        ~np.isclose(theta, theta_reference, atol=1e-14, rtol=0.0)
    ).astype(int).tolist()

    # Esta coluna diz ONDE a diferença apareceu, sem depender só de P(X_opt).
    signals = []
    if abs(delta_p) > CHANGE_TOL:
        signals.append("P_optimal")
    if abs(delta_e) > CHANGE_TOL:
        signals.append("energia")
    if bit_changed:
        signals.append("bitstring")
    if tvd > CHANGE_TOL:
        signals.append("distribuicao")
    if (1.0 - fidelity) > CHANGE_TOL:
        signals.append("estado")
    signature = "+".join(signals) if signals else "nenhuma"

    return {
        "label": label,
        "n_modified": int(n_modified),
        "last_theta_added": last_theta,
        "modified_indices": tuple(map(int, modified_indices)),
        "n_actually_changed": int(len(actually_changed)),
        "actually_changed_indices": tuple(actually_changed),
        "p_optimal": metrics["p_optimal"],
        "delta_p_optimal": delta_p,
        "expected_energy": metrics["expected_energy"],
        "delta_energy": delta_e,
        "dominant_bitstring": metrics["dominant_bitstring"],
        "dominant_probability": metrics["dominant_probability"],
        "bitstring_changed": bit_changed,
        "tvd_vs_reference": tvd,
        "state_fidelity_vs_reference": fidelity,
        "change_signature": signature,
        "any_detectable_change": bool(
            abs(delta_p) > CHANGE_TOL
            or abs(delta_e) > CHANGE_TOL
            or bit_changed
            or tvd > CHANGE_TOL
            or (1.0 - fidelity) > CHANGE_TOL
        ),
    }


In [ ]:
# ============================================================
# 19. EXPERIMENTO A — TODOS OS 23 THETAS = pi/2
# ============================================================

theta_all23 = theta_reference.copy()

# INTERVENÇÃO: somente os 23 índices testados recebem exatamente o mesmo valor pi/2.
theta_all23[INSENSITIVE_THETA_INDICES] = INTERVENTION_VALUE

# Garante que nenhum dos 7 parâmetros de controle foi alterado por acidente.
if not np.array_equal(
    theta_all23[ACTIVE_THETA_INDICES],
    theta_reference[ACTIVE_THETA_INDICES],
):
    raise RuntimeError("Um dos 7 theta ativos foi alterado no Experimento A.")

experiment_all23_df = pd.DataFrame([
    comparison_row("referencia_7_melhores", theta_reference, 0),
    comparison_row(
        "todos_23_em_pi_sobre_2",
        theta_all23,
        23,
        modified_indices=INSENSITIVE_THETA_INDICES,
    ),
])

display(experiment_all23_df)
experiment_all23_df.to_csv(
    TABLE_DIR / "experiment_all_23_theta_pi_over_2.csv",
    index=False,
)


In [ ]:
# ============================================================
# 20. EXPERIMENTO B — 1, 2, 3, ..., 23 ALTERAÇÕES CUMULATIVAS
# ============================================================

# A semente fixa torna a ordem aleatória totalmente reproduzível.
rng = np.random.default_rng(RANDOM_SEED)
random_order = rng.permutation(INSENSITIVE_THETA_INDICES).astype(int).tolist()

# Reinicia da referência; este experimento não herda o vetor do Experimento A.
theta_progressive = theta_reference.copy()
progressive_rows = [
    comparison_row("referencia_7_melhores", theta_progressive, 0)
]

for step, theta_index in enumerate(random_order, start=1):
    # A cada passo somente UM novo theta é acrescentado ao conjunto já modificado.
    theta_progressive[theta_index] = INTERVENTION_VALUE

    progressive_rows.append(
        comparison_row(
            f"passo_{step:02d}",
            theta_progressive,
            n_modified=step,
            last_theta=int(theta_index),
            modified_indices=random_order[:step],
        )
    )

progressive_23_df = pd.DataFrame(progressive_rows)

# A etapa 23 precisa ser exatamente o mesmo vetor do Experimento A.
if not np.array_equal(theta_progressive, theta_all23):
    raise RuntimeError("O passo 23 não reproduziu o vetor do Experimento A.")

display(progressive_23_df[[
    "n_modified",
    "n_actually_changed",
    "last_theta_added",
    "p_optimal",
    "delta_p_optimal",
    "expected_energy",
    "delta_energy",
    "dominant_bitstring",
    "bitstring_changed",
    "tvd_vs_reference",
    "state_fidelity_vs_reference",
    "change_signature",
    "any_detectable_change",
]])

progressive_23_df.to_csv(
    TABLE_DIR / "experiment_progressive_1_to_23_theta_pi_over_2.csv",
    index=False,
)

# Mostra o primeiro passo em que qualquer diferença mensurável aparece.
changed = progressive_23_df.loc[
    progressive_23_df["n_modified"].gt(0)
    & progressive_23_df["any_detectable_change"]
]
if changed.empty:
    print("Nenhuma mudança detectável apareceu em nenhum dos 23 passos.")
else:
    print("Primeiro passo com mudança detectável:")
    display(changed.head(1))

print("Ordem aleatória usada:", random_order)


# Parte VII — causalidade estrutural dos 7 \(\theta\) sensíveis

## Hipótese

A observação da Parte VI sugere que, **na arquitetura original**, 23 parâmetros são redundantes para o estado encontrado e somente os índices

\[
\theta_{2},\theta_{14},\theta_{17},\theta_{19},\theta_{22},\theta_{25},\theta_{27}
\]

controlam a solução observada.

Agora testamos se essa importância é explicada por:

- **identidade do bloco**: tipo `CY/CCY` e qubits tocados;
- **posição** do bloco no circuito;
- **ordem relativa** entre os 7 blocos;
- ou uma combinação desses fatores.

### Regra para evitar circularidade

O bitstring ótimo já é conhecido pela enumeração clássica, mas **não entra na função objetivo do otimizador**. Em todos os testes, o otimizador vê somente

\[
E(\theta)=\langle\psi(\theta)|H|\psi(\theta)\rangle.
\]

Somente depois da otimização medimos \(P(\mathcal X_{opt})\), bitstring dominante e distância de Hamming até o conjunto ótimo.


In [ ]:
# ============================================================
# 21. CONFIGURAÇÃO DO EXPERIMENTO 20.17 — TESTES CAUSAIS
# ============================================================

from scipy.optimize import minimize

STRUCTURAL_ROOT = OUTPUT_ROOT / "structural_causality_7theta"
STRUCTURAL_TABLE_DIR = STRUCTURAL_ROOT / "tables"
STRUCTURAL_FIGURE_DIR = STRUCTURAL_ROOT / "figures"
for directory in [STRUCTURAL_ROOT, STRUCTURAL_TABLE_DIR, STRUCTURAL_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reotimização principal: todos os 30 parâmetros ficam livres.
ARCHITECTURE_MAXITER = 700
ARCHITECTURE_RESTARTS = 3

# Varredura de posição: 7 blocos x 30 posições. Um restart mantém o custo controlado.
RUN_FULL_POSITION_SCAN = True
POSITION_SCAN_MAXITER = 300
POSITION_SCAN_RESTARTS = 1

# Um parâmetro é marcado como sensível se uma perturbação estrutural de T/4
# produzir mudança observável acima destes limites.
SENSITIVITY_PROB_TOL = 1e-6
SENSITIVITY_ENERGY_TOL = 1e-8
SENSITIVITY_TVD_TOL = 1e-6

print("Saídas causais 20.17:", STRUCTURAL_ROOT.resolve())
print("Reotimização das arquiteturas: 30 parâmetros livres.")
print("Varredura completa de posição:", RUN_FULL_POSITION_SCAN)


### Célula 22 — mapa dos 7 blocos e movimento necessário no bitstring

Esta célula responde primeiro à pergunta estrutural, sem otimização.

Para cada um dos 7 parâmetros ela mostra:

- posição original do bloco;
- tipo `CY` ou `CCY`;
- qubits e ativos tocados;
- primeira e última instrução física associada ao parâmetro.

Também compara o estado-base preparado pelas portas `X` com um bitstring ótimo de referência. Se existir mais de um ótimo degenerado, escolhe-se apenas para **visualização** aquele que já recebe maior probabilidade no estado de referência; o conjunto completo de ótimos continua sendo usado em `P_optimal`.


In [ ]:
# ============================================================
# 22. MAPA ESTRUTURAL DOS 7 BLOCOS + TRANSIÇÃO DE BITSTRING
# ============================================================

# Ordem real dos BLOCOS no circuito, obtida da primeira instrução parametrizada.
ORIGINAL_BLOCK_ORDER = parameter_map_df.sort_values("first_instruction")["theta_index"].astype(int).tolist()
if sorted(ORIGINAL_BLOCK_ORDER) != list(range(N_PARAMETERS)):
    raise RuntimeError("A ordem lógica não contém exatamente os 30 blocos.")

block_position = {theta_index: position for position, theta_index in enumerate(ORIGINAL_BLOCK_ORDER)}

active_structure_df = parameter_map_df.loc[
    parameter_map_df["theta_index"].isin(ACTIVE_THETA_INDICES)
].copy()
active_structure_df["original_block_position"] = active_structure_df["theta_index"].map(block_position)
active_structure_df = active_structure_df.sort_values("original_block_position")

# Estado-base realmente preparado pelos X iniciais, na convenção de bitstring do Qiskit.
initial_bitstring_qiskit = "".join(
    "1" if q in initial_x_qubits else "0"
    for q in range(N_ASSETS - 1, -1, -1)
)

# O ótimo de referência é usado SOMENTE para interpretação posterior.
optimal_probability_by_bitstring = {
    bitstring: float(reference_probability[label_to_index[bitstring]])
    for bitstring in exact_qiskit_bitstrings
}
REFERENCE_OPTIMAL_BITSTRING = max(
    exact_qiskit_bitstrings,
    key=lambda bitstring: optimal_probability_by_bitstring[bitstring],
)


def differing_qubits(bitstring_a, bitstring_b):
    """Converte diferenças na string Qiskit de volta para índices físicos de qubit."""
    return tuple(
        N_ASSETS - 1 - position
        for position, (a, b) in enumerate(zip(bitstring_a, bitstring_b))
        if a != b
    )


def transition_qubits(initial, final):
    removed, added = [], []
    for position, (a, b) in enumerate(zip(initial, final)):
        qubit = N_ASSETS - 1 - position
        if a == "1" and b == "0":
            removed.append(qubit)
        elif a == "0" and b == "1":
            added.append(qubit)
    return tuple(sorted(removed)), tuple(sorted(added))


removed_qubits, added_qubits = transition_qubits(
    initial_bitstring_qiskit, REFERENCE_OPTIMAL_BITSTRING
)
changed_qubits = set(removed_qubits) | set(added_qubits)

active_structure_df["touches_changed_qubit"] = active_structure_df["logical_block_qubits"].map(
    lambda qubits: bool(set(qubits) & changed_qubits)
)
active_structure_df["touches_removed_excitation"] = active_structure_df["logical_block_qubits"].map(
    lambda qubits: bool(set(qubits) & set(removed_qubits))
)
active_structure_df["touches_added_excitation"] = active_structure_df["logical_block_qubits"].map(
    lambda qubits: bool(set(qubits) & set(added_qubits))
)

route_columns = [
    "theta_index", "original_block_position", "ansatz_gate_type",
    "logical_block_qubits", "logical_assets", "primitive_physical_type",
    "first_instruction", "last_instruction", "touches_changed_qubit",
    "touches_removed_excitation", "touches_added_excitation",
]
display(active_structure_df[route_columns].reset_index(drop=True))

print("Bitstring inicial :", initial_bitstring_qiskit)
print("Ótimo de referência:", REFERENCE_OPTIMAL_BITSTRING)
print("Excitações que saem dos qubits:", removed_qubits)
print("Excitações que entram nos qubits:", added_qubits)

active_structure_df[route_columns].to_csv(
    STRUCTURAL_TABLE_DIR / "active7_structural_map.csv", index=False
)


### Célula 23 — reconstrução de uma arquitetura por ordem de blocos

A função abaixo é a peça central do experimento.

Ela **não move instruções primitivas isoladas**. Para cada identidade `theta_index`, reconstrói o bloco lógico inteiro:

- `CY`: `CX → CRY(θ) → CX`;
- `CCY`: `CX → RY(θ) → CCX → RY(-θ) → CCX → CX`.

Assim, quando `theta_17` é deslocado, são deslocadas junto com ele todas as operações que definem o bloco originalmente associado a `theta_17`.

A auditoria ao final reconstrói a ordem original e exige fidelidade unitária com o `ansatz` já usado nas Partes anteriores.


In [ ]:
# ============================================================
# 23. RECONSTRUTOR DO CIRCUITO POR BLOCOS — COM AUDITORIA
# ============================================================

structure_by_theta = structure_df.set_index("theta_index")


def build_circuit_from_block_order(block_order):
    """Reconstrói o ansatz movendo blocos completos e preservando sua identidade theta_j."""
    block_order = list(map(int, block_order))
    if sorted(block_order) != list(range(N_PARAMETERS)):
        raise ValueError("block_order deve ser uma permutação dos 30 theta_index.")

    theta_objects = ParameterVector("theta_struct", N_PARAMETERS)
    qc = QuantumCircuit(N_ASSETS)

    # Preparação inicial é idêntica em TODAS as arquiteturas.
    for qubit in initial_x_qubits:
        qc.x(int(qubit))

    for theta_index in block_order:
        row = structure_by_theta.loc[int(theta_index)]
        i_value, l_value = int(row["i"]), int(row["l"])
        theta_parameter = theta_objects[int(theta_index)]

        # CX externo que pertence ao bloco lógico original.
        qc.cx(i_value, l_value)

        if row["ansatz_gate_type"] == "CY":
            # Equivale ao CY_parameterized original: controle=l, alvo=i.
            qc.cry(theta_parameter, l_value, i_value)
        elif row["ansatz_gate_type"] == "CCY":
            # Equivale exatamente ao interior do CCY_parameterized original.
            qc.ry(theta_parameter, i_value)
            qc.ccx(l_value, i_value + 1, i_value)
            qc.ry(-theta_parameter, i_value)
            qc.ccx(l_value, i_value + 1, i_value)
        else:
            raise ValueError(f"Tipo de bloco inesperado: {row['ansatz_gate_type']}")

        # Segundo CX externo fecha o MESMO bloco lógico.
        qc.cx(i_value, l_value)

    return qc, tuple(theta_objects)


def bind_structural_circuit(circuit, parameter_objects, theta_values):
    """Binding explícito por identidade; não depende da ordenação interna de circuit.parameters."""
    theta_values = np.asarray(theta_values, dtype=float)
    mapping = {
        parameter_objects[j]: float(theta_values[j])
        for j in range(N_PARAMETERS)
    }
    return circuit.assign_parameters(mapping, inplace=False)


# AUDITORIA CRÍTICA: reconstruir a ordem original precisa reproduzir o ansatz anterior.
reconstructed_original, reconstructed_parameters = build_circuit_from_block_order(
    ORIGINAL_BLOCK_ORDER
)
old_state = np.asarray(
    Statevector.from_instruction(ansatz.assign_parameters(theta_reference, inplace=False)).data,
    dtype=np.complex128,
)
new_state = np.asarray(
    Statevector.from_instruction(
        bind_structural_circuit(reconstructed_original, reconstructed_parameters, theta_reference)
    ).data,
    dtype=np.complex128,
)
reconstruction_fidelity = float(abs(np.vdot(old_state, new_state)) ** 2)

if (1.0 - reconstruction_fidelity) > 1e-10:
    raise RuntimeError(
        "A reconstrução por blocos não reproduziu o circuito original. "
        f"Fidelidade={reconstruction_fidelity:.16f}"
    )

print(f"Fidelidade reconstrução/original = {reconstruction_fidelity:.16f}")
print("Auditoria aprovada: agora os blocos podem ser deslocados como unidades completas.")


### Célula 24 — medição e reotimização sem usar o bitstring como alvo

A energia é calculada a partir do **mesmo operador Ising**. Como esse Hamiltoniano é diagonal, sua energia em cada estado-base é pré-calculada uma única vez para acelerar as muitas reotimizações.

`optimize_block_order` recebe apenas:

1. uma ordem dos 30 blocos;
2. um vetor inicial;
3. os índices que podem variar.

Nos quatro testes principais, **todos os 30 parâmetros ficam livres**. Isso é importante: depois de mudar a arquitetura, um dos antigos 23 pode deixar de ser redundante. O experimento não força a conclusão anterior a permanecer verdadeira.


In [ ]:
# ============================================================
# 24. MEDIÇÃO + REOTIMIZAÇÃO DA ENERGIA
# ============================================================

# Energia de cada estado da base computacional. Isto acelera E=<H> sem mudar H.
try:
    _ising_matrix = ising.to_matrix(sparse=True)
    BASIS_ENERGIES = np.real(np.asarray(_ising_matrix.diagonal()).ravel()) + ising_offset
except TypeError:
    BASIS_ENERGIES = np.real(np.diag(np.asarray(ising.to_matrix()))) + ising_offset

valid_rank_lookup = {
    str(row.bitstring_qiskit_order): int(rank)
    for rank, row in enumerate(enumeration_df.itertuples(index=False), start=1)
}
valid_energy_lookup = {
    str(row.bitstring_qiskit_order): float(row.objective)
    for row in enumeration_df.itertuples(index=False)
}


def nearest_optimal_hamming(bitstring):
    return min(
        sum(a != b for a, b in zip(bitstring, optimum))
        for optimum in exact_qiskit_bitstrings
    )


def measure_structural_circuit(circuit, parameter_objects, theta_values, keep_probabilities=False):
    bound = bind_structural_circuit(circuit, parameter_objects, theta_values)
    state = Statevector.from_instruction(bound)
    probabilities = np.asarray(state.probabilities(), dtype=float)
    dominant_index = int(np.argmax(probabilities))
    dominant_bitstring = str(all_labels[dominant_index])
    expected_energy = float(np.dot(probabilities, BASIS_ENERGIES))

    result = {
        "expected_energy": expected_energy,
        "energy_gap": float(expected_energy - exact_energy),
        "p_optimal": float(probabilities[optimal_indices].sum()),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": float(probabilities[dominant_index]),
        "dominant_is_exact_optimum": dominant_bitstring in set(exact_qiskit_bitstrings),
        "nearest_optimal_hamming": int(nearest_optimal_hamming(dominant_bitstring)),
        "dominant_valid_rank": valid_rank_lookup.get(dominant_bitstring, np.nan),
        "dominant_classical_energy": valid_energy_lookup.get(dominant_bitstring, np.nan),
    }
    if keep_probabilities:
        result["probabilities"] = probabilities
        result["statevector"] = np.asarray(state.data, dtype=np.complex128)
    return result


def optimize_block_order(
    label,
    block_order,
    theta_start,
    free_indices=None,
    maxiter=ARCHITECTURE_MAXITER,
    n_restarts=ARCHITECTURE_RESTARTS,
    seed=RANDOM_SEED,
):
    """Minimiza SOMENTE a energia. O bitstring ótimo nunca entra na função objetivo."""
    circuit, parameter_objects = build_circuit_from_block_order(block_order)
    free_indices = np.asarray(
        list(range(N_PARAMETERS)) if free_indices is None else list(free_indices),
        dtype=int,
    )
    theta_start = np.asarray(theta_start, dtype=float).copy()
    periods = parameter_map_df.set_index("theta_index")["angular_period"].to_dict()
    rng_local = np.random.default_rng(int(seed))

    starts = [theta_start.copy()]
    for _ in range(max(0, int(n_restarts) - 1)):
        candidate = theta_start.copy()
        candidate[free_indices] += np.asarray([
            rng_local.uniform(-0.20, 0.20) * periods[int(j)]
            for j in free_indices
        ])
        starts.append(candidate)

    best_theta = theta_start.copy()
    best_metrics = measure_structural_circuit(circuit, parameter_objects, best_theta)
    total_nfev = 0
    optimizer_messages = []

    for restart_id, start in enumerate(starts):
        template = start.copy()

        def energy_objective(free_values):
            theta_trial = template.copy()
            theta_trial[free_indices] = np.asarray(free_values, dtype=float)
            return measure_structural_circuit(
                circuit, parameter_objects, theta_trial
            )["expected_energy"]

        result = minimize(
            energy_objective,
            x0=start[free_indices],
            method="COBYLA",
            options={"maxiter": int(maxiter), "rhobeg": 0.5, "catol": 1e-10},
        )
        total_nfev += int(getattr(result, "nfev", 0))
        optimizer_messages.append(str(getattr(result, "message", "")))

        theta_candidate = template.copy()
        theta_candidate[free_indices] = np.asarray(result.x, dtype=float)
        metrics_candidate = measure_structural_circuit(
            circuit, parameter_objects, theta_candidate
        )

        # Nunca aceita uma saída pior que o melhor ponto já conhecido.
        if metrics_candidate["expected_energy"] < best_metrics["expected_energy"]:
            best_theta = theta_candidate
            best_metrics = metrics_candidate

    return {
        "label": str(label),
        "block_order": tuple(map(int, block_order)),
        "theta_opt": best_theta,
        "metrics": best_metrics,
        "circuit": circuit,
        "parameter_objects": parameter_objects,
        "nfev_total": int(total_nfev),
        "optimizer_messages": tuple(optimizer_messages),
    }


### Célula 25 — quatro arquiteturas causais

Os quatro casos são:

- `original`: controle, mesma ordem da arquitetura usada anteriormente;
- `active_first`: os 7 blocos sensíveis vão para o começo;
- `active_last`: os 7 vão para o final;
- `active_reverse_in_place`: somente os 7 trocam de identidade entre as posições originalmente ocupadas por eles; os 23 restantes permanecem exatamente nos mesmos slots.

Em todos os casos o Hamiltoniano e a preparação inicial permanecem idênticos e os **30 parâmetros são reotimizados**.


In [ ]:
# ============================================================
# 25. REOTIMIZAÇÃO: ORIGINAL / 7 NO INÍCIO / 7 NO FIM / 7 INVERTIDOS
# ============================================================

active_order_original = [j for j in ORIGINAL_BLOCK_ORDER if j in ACTIVE_THETA_INDICES]
inactive_order_original = [j for j in ORIGINAL_BLOCK_ORDER if j not in ACTIVE_THETA_INDICES]

order_active_first = active_order_original + inactive_order_original
order_active_last = inactive_order_original + active_order_original

# Os 23 ficam nos mesmos slots; somente os IDs dos 7 são invertidos entre si.
order_active_reverse = ORIGINAL_BLOCK_ORDER.copy()
active_slots = [position for position, j in enumerate(ORIGINAL_BLOCK_ORDER) if j in ACTIVE_THETA_INDICES]
for slot, new_theta_id in zip(active_slots, active_order_original[::-1]):
    order_active_reverse[slot] = int(new_theta_id)

ARCHITECTURE_ORDERS = {
    "original": ORIGINAL_BLOCK_ORDER,
    "active_first": order_active_first,
    "active_last": order_active_last,
    "active_reverse_in_place": order_active_reverse,
}

architecture_runs = {}
for run_id, (label, order) in enumerate(ARCHITECTURE_ORDERS.items()):
    print(f"Otimizando arquitetura: {label}")
    architecture_runs[label] = optimize_block_order(
        label=label,
        block_order=order,
        theta_start=theta_reference,
        free_indices=range(N_PARAMETERS),  # IMPORTANTE: todos os 30 ficam livres.
        maxiter=ARCHITECTURE_MAXITER,
        n_restarts=ARCHITECTURE_RESTARTS,
        seed=RANDOM_SEED + 1000 * run_id,
    )

architecture_rows = []
architecture_theta_rows = []
for label, run in architecture_runs.items():
    row = {"architecture": label, "nfev_total": run["nfev_total"], **run["metrics"]}
    architecture_rows.append(row)
    for theta_index, value in enumerate(run["theta_opt"]):
        architecture_theta_rows.append({
            "architecture": label,
            "theta_index": int(theta_index),
            "optimized_theta": float(value),
            "originally_sensitive": bool(theta_index in ACTIVE_THETA_INDICES),
            "block_position": int(run["block_order"].index(theta_index)),
        })

architecture_summary_df = pd.DataFrame(architecture_rows)
architecture_theta_long_df = pd.DataFrame(architecture_theta_rows)

display(architecture_summary_df[[
    "architecture", "expected_energy", "energy_gap", "p_optimal",
    "dominant_bitstring", "dominant_probability", "dominant_is_exact_optimum",
    "nearest_optimal_hamming", "dominant_valid_rank", "nfev_total",
]])

architecture_summary_df.to_csv(
    STRUCTURAL_TABLE_DIR / "architecture_reoptimization_summary.csv", index=False
)
architecture_theta_long_df.to_csv(
    STRUCTURAL_TABLE_DIR / "architecture_optimized_theta_long.csv", index=False
)


### Célula 26 — a sensibilidade permaneceu nos mesmos 7?

Depois de cada reotimização, cada um dos 30 parâmetros recebe duas perturbações locais estruturais:

\[
\theta_j\rightarrow\theta_j\pm T_j/4,
\]

onde \(T_j=2\pi\) para os blocos `RY/CCY` e \(T_j=4\pi\) para `CRY/CY`.

A classificação usa simultaneamente:

- mudança em \(P(\mathcal X_{opt})\);
- mudança de energia;
- TVD da distribuição completa;
- troca do bitstring dominante.

Isso permite verificar se, depois de mover os blocos, a sensibilidade **continua nos sete originais ou migra para algum dos antigos 23**.


In [ ]:
# ============================================================
# 26. PROBE DE SENSIBILIDADE DOS 30 PARÂMETROS EM CADA ARQUITETURA
# ============================================================

period_by_theta = parameter_map_df.set_index("theta_index")["angular_period"].to_dict()


def sensitivity_probe(run):
    base_theta = np.asarray(run["theta_opt"], dtype=float)
    circuit = run["circuit"]
    parameter_objects = run["parameter_objects"]
    base = measure_structural_circuit(
        circuit, parameter_objects, base_theta, keep_probabilities=True
    )
    rows = []

    for theta_index in range(N_PARAMETERS):
        shift = float(period_by_theta[theta_index] / 4.0)
        probe_metrics = []
        for sign in (-1.0, +1.0):
            trial = base_theta.copy()
            trial[theta_index] += sign * shift
            measured = measure_structural_circuit(
                circuit, parameter_objects, trial, keep_probabilities=True
            )
            measured["tvd"] = float(
                0.5 * np.abs(measured["probabilities"] - base["probabilities"]).sum()
            )
            probe_metrics.append(measured)

        max_dp = max(abs(m["p_optimal"] - base["p_optimal"]) for m in probe_metrics)
        max_de = max(abs(m["expected_energy"] - base["expected_energy"]) for m in probe_metrics)
        max_tvd = max(m["tvd"] for m in probe_metrics)
        bit_changed = any(
            m["dominant_bitstring"] != base["dominant_bitstring"]
            for m in probe_metrics
        )
        sensitive = bool(
            max_dp > SENSITIVITY_PROB_TOL
            or max_de > SENSITIVITY_ENERGY_TOL
            or max_tvd > SENSITIVITY_TVD_TOL
            or bit_changed
        )
        rows.append({
            "theta_index": int(theta_index),
            "block_position": int(run["block_order"].index(theta_index)),
            "originally_sensitive": bool(theta_index in ACTIVE_THETA_INDICES),
            "probe_shift": shift,
            "max_abs_delta_p_optimal": float(max_dp),
            "max_abs_delta_energy": float(max_de),
            "max_tvd": float(max_tvd),
            "dominant_bitstring_changed": bool(bit_changed),
            "sensitive_after_reorder": sensitive,
        })
    return pd.DataFrame(rows)


sensitivity_frames = []
for architecture, run in architecture_runs.items():
    frame = sensitivity_probe(run)
    frame.insert(0, "architecture", architecture)
    sensitivity_frames.append(frame)

sensitivity_by_architecture_df = pd.concat(sensitivity_frames, ignore_index=True)

sensitivity_by_architecture_df["original7_and_sensitive"] = (
    sensitivity_by_architecture_df["originally_sensitive"]
    & sensitivity_by_architecture_df["sensitive_after_reorder"]
)

sensitivity_summary_df = (
    sensitivity_by_architecture_df
    .groupby("architecture", as_index=False)
    .agg(
        n_sensitive_after_reorder=("sensitive_after_reorder", "sum"),
        n_original7_still_sensitive=("original7_and_sensitive", "sum"),
    )
)

sensitive_only_df = sensitivity_by_architecture_df.loc[
    sensitivity_by_architecture_df["sensitive_after_reorder"]
].sort_values(["architecture", "block_position"])

display(sensitivity_summary_df)
display(sensitive_only_df)

sensitivity_by_architecture_df.to_csv(
    STRUCTURAL_TABLE_DIR / "sensitivity_after_each_architecture.csv", index=False
)


### Célula 27 — deslocamento individual dos 7 blocos por todas as posições

Este é o teste de posição mais direto.

Para cada um dos 7 blocos:

1. remove o bloco da posição original;
2. insere o **mesmo bloco completo** em cada posição de 0 a 29;
3. mantém a ordem relativa dos outros 29 blocos;
4. reotimiza **todos os 30 parâmetros**;
5. mede energia, \(P_{opt}\), bitstring dominante e distância de Hamming;
6. registra também os valores otimizados dos **sete parâmetros originais**.

O ponto da posição original é reutilizado do controle já otimizado, evitando sete reotimizações redundantes.

`RUN_FULL_POSITION_SCAN=False` pode ser usado para pular esta etapa durante uma checagem rápida das células anteriores.


In [ ]:
# ============================================================
# 27. VARREDURA CAUSAL DE POSIÇÃO DOS 7 BLOCOS
# ============================================================


def move_one_block(base_order, theta_index, target_position):
    order = list(map(int, base_order))
    order.remove(int(theta_index))
    order.insert(int(target_position), int(theta_index))
    return order


position_scan_rows = []
position_theta_rows = []

if RUN_FULL_POSITION_SCAN:
    original_control = architecture_runs["original"]
    theta_start_scan = np.asarray(original_control["theta_opt"], dtype=float)

    for moved_theta in ACTIVE_THETA_INDICES:
        original_position = int(ORIGINAL_BLOCK_ORDER.index(int(moved_theta)))
        print(f"Varredura de posição: theta_{moved_theta} (posição original {original_position})")

        for target_position in range(N_PARAMETERS):
            if target_position == original_position:
                run = original_control
            else:
                order = move_one_block(
                    ORIGINAL_BLOCK_ORDER, moved_theta, target_position
                )
                run = optimize_block_order(
                    label=f"theta_{moved_theta}_to_{target_position}",
                    block_order=order,
                    theta_start=theta_start_scan,
                    free_indices=range(N_PARAMETERS),
                    maxiter=POSITION_SCAN_MAXITER,
                    n_restarts=POSITION_SCAN_RESTARTS,
                    seed=RANDOM_SEED + 10000 + 100 * int(moved_theta) + target_position,
                )

            metrics = run["metrics"]
            position_scan_rows.append({
                "moved_theta": int(moved_theta),
                "original_position": original_position,
                "target_position": int(target_position),
                "position_shift": int(target_position - original_position),
                "expected_energy": metrics["expected_energy"],
                "energy_gap": metrics["energy_gap"],
                "p_optimal": metrics["p_optimal"],
                "dominant_bitstring": metrics["dominant_bitstring"],
                "dominant_probability": metrics["dominant_probability"],
                "dominant_is_exact_optimum": metrics["dominant_is_exact_optimum"],
                "nearest_optimal_hamming": metrics["nearest_optimal_hamming"],
                "dominant_valid_rank": metrics["dominant_valid_rank"],
                "nfev_total": run["nfev_total"],
            })

            # Long format: mostra como TODOS os 7 theta respondem ao deslocamento de um deles.
            for tracked_theta in ACTIVE_THETA_INDICES:
                position_theta_rows.append({
                    "moved_theta": int(moved_theta),
                    "target_position": int(target_position),
                    "tracked_theta": int(tracked_theta),
                    "optimized_theta": float(run["theta_opt"][int(tracked_theta)]),
                })

    position_scan_df = pd.DataFrame(position_scan_rows)
    position_theta_trace_df = pd.DataFrame(position_theta_rows)

    position_scan_summary_df = (
        position_scan_df.groupby("moved_theta", as_index=False)
        .agg(
            original_position=("original_position", "first"),
            min_energy_gap=("energy_gap", "min"),
            max_energy_gap=("energy_gap", "max"),
            min_p_optimal=("p_optimal", "min"),
            max_p_optimal=("p_optimal", "max"),
            n_positions_exact_dominant=("dominant_is_exact_optimum", "sum"),
            max_hamming=("nearest_optimal_hamming", "max"),
        )
    )

    display(position_scan_summary_df)

    changed_bitstrings_df = position_scan_df.loc[
        ~position_scan_df["dominant_is_exact_optimum"]
    ].sort_values(["moved_theta", "target_position"])
    print("Posições em que o bitstring dominante deixou de ser ótimo:", len(changed_bitstrings_df))
    display(changed_bitstrings_df.head(40))

    position_scan_df.to_csv(
        STRUCTURAL_TABLE_DIR / "active7_full_position_scan.csv", index=False
    )
    position_theta_trace_df.to_csv(
        STRUCTURAL_TABLE_DIR / "active7_theta_response_during_position_scan.csv", index=False
    )
else:
    print("Varredura completa de posição pulada por RUN_FULL_POSITION_SCAN=False.")


### Célula 28 — visualização mínima e critérios de interpretação

Os mapas abaixo são deliberadamente simples:

- linha = identidade do bloco sensível movido;
- coluna = nova posição do bloco;
- primeiro mapa = probabilidade total dos bitstrings ótimos;
- segundo mapa = distância de Hamming do bitstring dominante até o ótimo mais próximo.

A interpretação é causal:

- **linha quase constante**: aquele bloco tolera deslocamento e sua identidade é mais importante que a posição;
- **faixa estreita de posições boas**: a posição é parte do mecanismo;
- **troca de sensibilidade para antigos 23** na Célula 26: a arquitetura, e não o índice original, determina quais parâmetros controlam;
- **mudança de bitstring com energia quase inalterada**: verificar `dominant_valid_rank` e `energy_gap` antes de interpretar como degenerescência;
- **os mesmos 7 continuam sensíveis em todas as ordens**: evidência favorável a uma subestrutura funcional associada aos qubits/blocos desses sete.


In [ ]:
# ============================================================
# 28. MAPAS MÍNIMOS: P(ÓTIMO) E DISTÂNCIA DE HAMMING
# ============================================================

if RUN_FULL_POSITION_SCAN and not position_scan_df.empty:
    p_matrix = position_scan_df.pivot(
        index="moved_theta", columns="target_position", values="p_optimal"
    ).sort_index()

    fig, ax = plt.subplots(figsize=(12, 4.5))
    image = ax.imshow(p_matrix.to_numpy(), aspect="auto", origin="lower")
    ax.set_yticks(range(len(p_matrix.index)))
    ax.set_yticklabels([f"theta_{j}" for j in p_matrix.index])
    ax.set_xticks(range(N_PARAMETERS))
    ax.set_xticklabels(range(N_PARAMETERS), rotation=90)
    ax.set_xlabel("Nova posição do bloco")
    ax.set_ylabel("Bloco deslocado")
    ax.set_title("P(bitstring ótimo) após reotimização")
    fig.colorbar(image, ax=ax, label="P_optimal")
    fig.tight_layout()
    fig.savefig(STRUCTURAL_FIGURE_DIR / "position_scan_p_optimal.png", dpi=180)
    plt.show()

    hamming_matrix = position_scan_df.pivot(
        index="moved_theta", columns="target_position", values="nearest_optimal_hamming"
    ).sort_index()

    fig, ax = plt.subplots(figsize=(12, 4.5))
    image = ax.imshow(hamming_matrix.to_numpy(), aspect="auto", origin="lower")
    ax.set_yticks(range(len(hamming_matrix.index)))
    ax.set_yticklabels([f"theta_{j}" for j in hamming_matrix.index])
    ax.set_xticks(range(N_PARAMETERS))
    ax.set_xticklabels(range(N_PARAMETERS), rotation=90)
    ax.set_xlabel("Nova posição do bloco")
    ax.set_ylabel("Bloco deslocado")
    ax.set_title("Distância de Hamming do dominante ao ótimo mais próximo")
    fig.colorbar(image, ax=ax, label="Hamming")
    fig.tight_layout()
    fig.savefig(STRUCTURAL_FIGURE_DIR / "position_scan_hamming.png", dpi=180)
    plt.show()


# Parte VIII — ação variacional, geometria do ansatz e soma sobre caminhos

## O que esta parte testa

Os experimentos anteriores mostraram empiricamente que 23 parâmetros podem ser alterados sem modificar a solução, enquanto 7 parâmetros aparecem como sensíveis. Agora queremos saber **qual mecanismo matemático produz essa redução efetiva**.

Para um ansatz parametrizado

\[
|\psi(\boldsymbol\theta)\rangle,
\]

cada parâmetro define uma direção tangente

\[
|\partial_i\psi\rangle=\frac{\partial|\psi\rangle}{\partial\theta_i}.
\]

A métrica de Fubini–Study é obtida do tensor geométrico quântico:

\[
g_{ij}=\operatorname{Re}\left[
\langle\partial_i\psi|\partial_j\psi\rangle-
\langle\partial_i\psi|\psi\rangle
\langle\psi|\partial_j\psi\rangle
\right].
\]

A diagonal \(g_{ii}\) mede quanto o **estado físico**, retirando fase global, se move quando \(\theta_i\) varia.

### Um cuidado importante no ponto ótimo

Em um mínimo variacional esperamos

\[
\frac{\partial E}{\partial\theta_i}\approx0
\]

para **todos** os parâmetros. Portanto o gradiente de energia não deve ser usado sozinho para decidir quais θ são importantes. O diagnóstico de rigidez local será a curvatura

\[
\frac{\partial^2 E}{\partial\theta_i^2}.
\]

Assim conseguimos separar:

- **direção geométrica nula:** \(g_{ii}\approx0\);
- **direção energeticamente plana:** \(\partial_i^2E\approx0\);
- **direção que muda o estado mas não a energia:** \(g_{ii}>0\) e curvatura energética pequena.

A segunda metade desta parte trata o circuito como uma soma discreta sobre histórias. A amplitude final

\[
\langle x_f|U_L\cdots U_1|x_i\rangle
\]

é propagada bloco a bloco, somando coerentemente as contribuições de todos os estados intermediários. Isso é o análogo discreto, no circuito, da lógica de soma sobre caminhos de Feynman.


### Célula 29 — referência física e parâmetros numéricos da Parte VIII

A referência desta etapa é a arquitetura `original` **reotimizada com os 30 θ livres** na Parte VII. Isso evita usar os máximos 1D como se fossem um ótimo conjunto.

O bitstring ótimo continua sendo apenas um **observável posterior**. Ele nunca entra na função objetivo do COBYLA.

Os passos finitos abaixo são pequenos e servem somente para derivadas numéricas do `Statevector`. As tolerâncias de classificação são relativas ao maior sinal observado, para que a tabela mostre também os valores brutos usados na decisão.


In [ ]:
# ============================================================
# 29. CONFIGURAÇÃO — GEOMETRIA VARIACIONAL E PATH-SUM
# ============================================================

from collections import defaultdict
from qiskit.quantum_info import Operator

ACTION_ROOT = OUTPUT_ROOT / "action_variational_pathsum"
ACTION_TABLE_DIR = ACTION_ROOT / "tables"
ACTION_FIGURE_DIR = ACTION_ROOT / "figures"
for directory in [ACTION_ROOT, ACTION_TABLE_DIR, ACTION_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Passo central para derivadas do estado e curvaturas locais.
FD_STEP = 1e-4
QGT_RELATIVE_NULL_TOL = 1e-6
CURVATURE_RELATIVE_NULL_TOL = 1e-6
AMPLITUDE_RELATIVE_NULL_TOL = 1e-6

# Path-sum: apenas zeros numéricos são descartados.
PATH_MATRIX_ELEMENT_TOL = 1e-14
PATH_AMPLITUDE_TOL = 1e-14
TOP_STATES_PER_LAYER = 12

# Trajetória geométrica opcional. Não é chamada de geodésica.
RUN_PARAMETER_PATH_GEOMETRY = True
PARAMETER_PATH_POINTS = 9

# Referência física: circuito ORIGINAL após reotimização conjunta dos 30 parâmetros.
action_run = architecture_runs["original"]
ACTION_CIRCUIT = action_run["circuit"]
ACTION_PARAMETERS = action_run["parameter_objects"]
ACTION_THETA = np.asarray(action_run["theta_opt"], dtype=float).copy()
action_metrics = measure_structural_circuit(
    ACTION_CIRCUIT, ACTION_PARAMETERS, ACTION_THETA, keep_probabilities=True
)
ACTION_STATE = np.asarray(action_metrics["statevector"], dtype=np.complex128)
ACTION_PROBABILITY = np.asarray(action_metrics["probabilities"], dtype=float)

# Se houver degenerescência clássica, escolhe para o path-sum o ótimo que recebe
# maior probabilidade NESTA referência reotimizada. Isso não altera a otimização.
PATH_TARGET_BITSTRING = max(
    exact_qiskit_bitstrings,
    key=lambda bit: float(ACTION_PROBABILITY[label_to_index[bit]]),
)
PATH_TARGET_INDEX = int(label_to_index[PATH_TARGET_BITSTRING])

print("Referência da Parte VIII: arquitetura original reotimizada")
print("Energia =", action_metrics["expected_energy"])
print("P(ótimos) =", action_metrics["p_optimal"])
print("Bitstring alvo apenas para diagnóstico =", PATH_TARGET_BITSTRING)
print("Saídas =", ACTION_ROOT.resolve())


### Célula 30 — QGT/Fubini–Study, gradiente e curvatura de energia

Esta célula faz somente uma operação conceitual: desloca cada θ por \(\pm\varepsilon\), calcula os dois `Statevector` e forma a derivada central

\[
|\partial_i\psi\rangle\approx
\frac{|\psi(\theta_i+\varepsilon)\rangle-|\psi(\theta_i-\varepsilon)\rangle}{2\varepsilon}.
\]

Com essas 30 derivadas, todo o QGT é construído por produtos internos. Na mesma avaliação são calculados:

- gradiente de energia — deve ficar pequeno no mínimo;
- **curvatura de energia** — indica rigidez ou planicidade;
- derivada e curvatura da probabilidade do bitstring ótimo escolhido;
- derivada e curvatura da probabilidade total do conjunto de ótimos.

Nenhuma nova otimização é executada.


In [ ]:
# ============================================================
# 30. DERIVADAS DO ESTADO + QGT + CURVATURAS
# ============================================================


def structural_state(theta_values):
    """Statevector do circuito estrutural original para um vetor theta."""
    bound = bind_structural_circuit(ACTION_CIRCUIT, ACTION_PARAMETERS, theta_values)
    return np.asarray(Statevector.from_instruction(bound).data, dtype=np.complex128)


def energy_from_state(state):
    """Como H é diagonal, E é o produto das probabilidades pelas energias da base."""
    return float(np.dot(np.abs(state) ** 2, BASIS_ENERGIES))


base_state = structural_state(ACTION_THETA)
base_energy = energy_from_state(base_state)
base_p_target = float(abs(base_state[PATH_TARGET_INDEX]) ** 2)
base_p_optimal = float(np.sum(np.abs(base_state[optimal_indices]) ** 2))

state_derivatives = []
finite_rows = []

for theta_index in range(N_PARAMETERS):
    plus = ACTION_THETA.copy()
    minus = ACTION_THETA.copy()
    plus[theta_index] += FD_STEP
    minus[theta_index] -= FD_STEP

    psi_plus = structural_state(plus)
    psi_minus = structural_state(minus)
    dpsi = (psi_plus - psi_minus) / (2.0 * FD_STEP)
    state_derivatives.append(dpsi)

    e_plus, e_minus = energy_from_state(psi_plus), energy_from_state(psi_minus)
    p_target_plus = float(abs(psi_plus[PATH_TARGET_INDEX]) ** 2)
    p_target_minus = float(abs(psi_minus[PATH_TARGET_INDEX]) ** 2)
    p_opt_plus = float(np.sum(np.abs(psi_plus[optimal_indices]) ** 2))
    p_opt_minus = float(np.sum(np.abs(psi_minus[optimal_indices]) ** 2))

    finite_rows.append({
        "theta_index": int(theta_index),
        "energy_gradient_fd": float((e_plus - e_minus) / (2.0 * FD_STEP)),
        "energy_curvature_fd": float((e_plus - 2.0 * base_energy + e_minus) / FD_STEP**2),
        "p_target_gradient_fd": float((p_target_plus - p_target_minus) / (2.0 * FD_STEP)),
        "p_target_curvature_fd": float((p_target_plus - 2.0 * base_p_target + p_target_minus) / FD_STEP**2),
        "p_optimal_gradient_fd": float((p_opt_plus - p_opt_minus) / (2.0 * FD_STEP)),
        "p_optimal_curvature_fd": float((p_opt_plus - 2.0 * base_p_optimal + p_opt_minus) / FD_STEP**2),
    })

DPSI = np.asarray(state_derivatives, dtype=np.complex128)

# <d_i|d_j> e <d_i|psi>; a segunda parcela remove a direção de fase global.
gram = DPSI.conj() @ DPSI.T
overlap_with_state = DPSI.conj() @ base_state
QGT_METRIC = np.real(
    gram - np.outer(overlap_with_state, np.conj(overlap_with_state))
)
QGT_METRIC = 0.5 * (QGT_METRIC + QGT_METRIC.T)

qgt_diag = np.clip(np.diag(QGT_METRIC), 0.0, None)
raw_derivative_norm = np.linalg.norm(DPSI, axis=1)
tangent_norm = np.sqrt(qgt_diag)

finite_difference_df = pd.DataFrame(finite_rows)
variational_geometry_df = parameter_map_df.copy().merge(
    finite_difference_df, on="theta_index", how="left", validate="one_to_one"
)
variational_geometry_df["qgt_diag"] = qgt_diag
variational_geometry_df["raw_state_derivative_norm"] = raw_derivative_norm
variational_geometry_df["fubini_study_tangent_norm"] = tangent_norm
variational_geometry_df["originally_sensitive"] = variational_geometry_df["theta_index"].isin(
    ACTIVE_THETA_INDICES
)

variational_geometry_df.to_csv(
    ACTION_TABLE_DIR / "variational_geometry_qgt_curvature_30theta.csv", index=False
)
display(variational_geometry_df[[
    "theta_index", "originally_sensitive", "ansatz_gate_type", "logical_block_qubits",
    "qgt_diag", "fubini_study_tangent_norm", "energy_gradient_fd",
    "energy_curvature_fd", "p_optimal_curvature_fd",
]].sort_values("qgt_diag", ascending=False))


### Célula 31 — espectro do QGT e dimensão efetiva

Se a família nominal de 30 parâmetros realmente se comporta como uma variedade de dimensão muito menor perto da solução, o espectro de \(g\) deve apresentar poucos autovalores relevantes.

Dois números são registrados:

1. **posto numérico:** quantidade de autovalores acima de uma tolerância relativa;
2. **dimensão de participação:**

\[
d_{\mathrm{PR}}=\frac{(\sum_a\lambda_a)^2}{\sum_a\lambda_a^2}.
\]

Também calculamos qual fração da diagonal do QGT está concentrada nos 7 θ já identificados empiricamente. Essa fração é um diagnóstico, não uma imposição — os 7 não entram no cálculo de \(g\).


In [ ]:
# ============================================================
# 31. ESPECTRO DO QGT E DIMENSÃO EFETIVA
# ============================================================

qgt_eigenvalues = np.linalg.eigvalsh(QGT_METRIC)
qgt_eigenvalues = np.clip(qgt_eigenvalues, 0.0, None)[::-1]
max_eigenvalue = float(qgt_eigenvalues[0]) if len(qgt_eigenvalues) else 0.0
rank_threshold = max(1e-14, QGT_RELATIVE_NULL_TOL * max_eigenvalue)
qgt_numeric_rank = int(np.sum(qgt_eigenvalues > rank_threshold))

if np.sum(qgt_eigenvalues**2) > 0:
    qgt_participation_dimension = float(
        np.sum(qgt_eigenvalues) ** 2 / np.sum(qgt_eigenvalues**2)
    )
else:
    qgt_participation_dimension = 0.0

active_mask = np.array([j in ACTIVE_THETA_INDICES for j in range(N_PARAMETERS)], dtype=bool)
metric_trace = float(np.sum(qgt_diag))
active_metric_fraction = float(np.sum(qgt_diag[active_mask]) / metric_trace) if metric_trace > 0 else np.nan

qgt_spectrum_df = pd.DataFrame({
    "eigenvalue_rank": np.arange(1, N_PARAMETERS + 1, dtype=int),
    "qgt_eigenvalue": qgt_eigenvalues,
    "above_numeric_rank_threshold": qgt_eigenvalues > rank_threshold,
})
qgt_summary_df = pd.DataFrame([{
    "n_nominal_parameters": int(N_PARAMETERS),
    "qgt_numeric_rank": qgt_numeric_rank,
    "qgt_participation_dimension": qgt_participation_dimension,
    "rank_threshold": rank_threshold,
    "active7_metric_trace_fraction": active_metric_fraction,
}])

# Classificação relativa; os valores brutos continuam na tabela para auditoria.
metric_null_threshold = max(1e-14, QGT_RELATIVE_NULL_TOL * float(np.max(qgt_diag)))
curvature_scale = float(np.max(np.abs(variational_geometry_df["energy_curvature_fd"])))
curvature_null_threshold = max(1e-12, CURVATURE_RELATIVE_NULL_TOL * curvature_scale)

variational_geometry_df["geometrically_null"] = variational_geometry_df["qgt_diag"] <= metric_null_threshold
variational_geometry_df["energetically_flat"] = (
    variational_geometry_df["energy_curvature_fd"].abs() <= curvature_null_threshold
)

print("Posto numérico do QGT =", qgt_numeric_rank, "/", N_PARAMETERS)
print("Dimensão de participação =", qgt_participation_dimension)
print("Fração Tr(g) nos 7 theta =", active_metric_fraction)
display(qgt_summary_df)

qgt_spectrum_df.to_csv(ACTION_TABLE_DIR / "qgt_eigenvalue_spectrum.csv", index=False)
qgt_summary_df.to_csv(ACTION_TABLE_DIR / "qgt_effective_dimension_summary.csv", index=False)
variational_geometry_df.to_csv(
    ACTION_TABLE_DIR / "variational_geometry_qgt_curvature_30theta.csv", index=False
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(qgt_spectrum_df["eigenvalue_rank"], np.maximum(qgt_eigenvalues, 1e-18), marker="o")
ax.axhline(rank_threshold, linestyle="--", linewidth=1.0, label="limiar de posto")
ax.set_xlabel("ordem do autovalor")
ax.set_ylabel("autovalor do QGT")
ax.set_title("Espectro da métrica de Fubini–Study no ponto otimizado")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(ACTION_FIGURE_DIR / "qgt_eigenvalue_spectrum.png", dpi=180)
plt.show()


### Célula 32 — influência de cada θ na amplitude do ótimo

Para o bitstring ótimo escolhido apenas como diagnóstico,

\[
A_f(\boldsymbol\theta)=\langle x_f|\psi(\boldsymbol\theta)\rangle,
\]

calculamos

\[
\frac{\partial A_f}{\partial\theta_i}
=\langle x_f|\partial_i\psi\rangle.
\]

Essa quantidade é equivalente à forma “forward/backward”

\[
\left\langle B_i\left|\frac{\partial U_i}{\partial\theta_i}\right|F_{i-1}\right\rangle,
\]

mas aqui é obtida diretamente do `Statevector` derivado, evitando reconstruções adicionais e mantendo a auditoria simples.

Como a probabilidade ótima pode estar próxima de 1, sua primeira derivada também pode zerar no máximo. Por isso guardamos simultaneamente **derivada de amplitude**, QGT e **curvatura de probabilidade**.


In [ ]:
# ============================================================
# 32. DERIVADA DA AMPLITUDE DO BITSTRING ÓTIMO
# ============================================================

reference_amplitude = complex(base_state[PATH_TARGET_INDEX])
d_amplitude = DPSI[:, PATH_TARGET_INDEX]

# Derivada da probabilidade de UM ótimo e da massa em TODOS os ótimos.
d_probability_target = 2.0 * np.real(np.conj(reference_amplitude) * d_amplitude)
d_probability_all_optima = 2.0 * np.real(
    np.sum(np.conj(base_state[optimal_indices])[None, :] * DPSI[:, optimal_indices], axis=1)
)

amplitude_influence_df = pd.DataFrame({
    "theta_index": np.arange(N_PARAMETERS, dtype=int),
    "originally_sensitive": [j in ACTIVE_THETA_INDICES for j in range(N_PARAMETERS)],
    "abs_d_amplitude_target": np.abs(d_amplitude),
    "real_d_amplitude_target": np.real(d_amplitude),
    "imag_d_amplitude_target": np.imag(d_amplitude),
    "d_probability_target": d_probability_target,
    "d_probability_all_optima": d_probability_all_optima,
})

amp_scale = float(amplitude_influence_df["abs_d_amplitude_target"].max())
amp_null_threshold = max(1e-14, AMPLITUDE_RELATIVE_NULL_TOL * amp_scale)
amplitude_influence_df["amplitude_decoupled"] = (
    amplitude_influence_df["abs_d_amplitude_target"] <= amp_null_threshold
)

amplitude_influence_df = amplitude_influence_df.merge(
    parameter_map_df[[
        "theta_index", "ansatz_gate_type", "logical_block_qubits", "logical_assets",
        "first_instruction", "last_instruction",
    ]],
    on="theta_index", how="left", validate="one_to_one"
)

amplitude_influence_df.to_csv(
    ACTION_TABLE_DIR / "target_amplitude_influence_30theta.csv", index=False
)
display(amplitude_influence_df.sort_values("abs_d_amplitude_target", ascending=False))


### Célula 33 — propagação bloco a bloco

Agora o circuito é observado como uma sequência de 30 transformações lógicas.

Começamos exatamente no estado preparado pelas portas `X` e, após cada bloco, registramos:

- \(P(x_f)\) para o ótimo escolhido;
- probabilidade total dos ótimos degenerados;
- bitstring dominante;
- TVD em relação à camada anterior;
- quantidade de estados com probabilidade numericamente relevante;
- os bitstrings mais prováveis e suas fases.

A última linha deve reproduzir o `Statevector` completo da referência. Essa fidelidade é uma auditoria obrigatória antes da interpretação do caminho.


In [ ]:
# ============================================================
# 33. PROPAGAÇÃO CAMADA A CAMADA
# ============================================================


def bound_logical_block(theta_index, theta_value):
    """Constrói somente UM bloco lógico, já com o valor numérico de theta."""
    row = structure_by_theta.loc[int(theta_index)]
    i_value, l_value = int(row["i"]), int(row["l"])
    qc = QuantumCircuit(N_ASSETS)
    qc.cx(i_value, l_value)
    if row["ansatz_gate_type"] == "CY":
        qc.cry(float(theta_value), l_value, i_value)
    else:
        qc.ry(float(theta_value), i_value)
        qc.ccx(l_value, i_value + 1, i_value)
        qc.ry(-float(theta_value), i_value)
        qc.ccx(l_value, i_value + 1, i_value)
    qc.cx(i_value, l_value)
    return qc


initial_circuit = QuantumCircuit(N_ASSETS)
for qubit in initial_x_qubits:
    initial_circuit.x(int(qubit))
layer_state = Statevector.from_instruction(initial_circuit)

layer_rows = []
top_state_rows = []
previous_probability = np.asarray(layer_state.probabilities(), dtype=float)


def record_layer(step, theta_index, state, previous_probability):
    probability = np.asarray(state.probabilities(), dtype=float)
    dominant_index = int(np.argmax(probability))
    row = {
        "step": int(step),
        "theta_index": theta_index,
        "is_original7": bool(theta_index in ACTIVE_THETA_INDICES) if theta_index is not None else False,
        "p_target": float(probability[PATH_TARGET_INDEX]),
        "p_all_optima": float(probability[optimal_indices].sum()),
        "dominant_bitstring": str(all_labels[dominant_index]),
        "dominant_probability": float(probability[dominant_index]),
        "tvd_from_previous": float(0.5 * np.abs(probability - previous_probability).sum()),
        "n_states_probability_gt_1e_12": int(np.sum(probability > 1e-12)),
    }
    top_indices = np.argsort(probability)[::-1][:TOP_STATES_PER_LAYER]
    for rank, basis_index in enumerate(top_indices, start=1):
        amplitude = complex(state.data[int(basis_index)])
        top_state_rows.append({
            "step": int(step),
            "theta_index": theta_index,
            "rank": int(rank),
            "bitstring": str(all_labels[int(basis_index)]),
            "probability": float(probability[int(basis_index)]),
            "amplitude_abs": float(abs(amplitude)),
            "amplitude_phase": float(np.angle(amplitude)),
            "hamming_to_target": int(sum(
                a != b for a, b in zip(str(all_labels[int(basis_index)]), PATH_TARGET_BITSTRING)
            )),
        })
    return row, probability


row0, previous_probability = record_layer(0, None, layer_state, previous_probability)
layer_rows.append(row0)

for step, theta_index in enumerate(ORIGINAL_BLOCK_ORDER, start=1):
    block = bound_logical_block(theta_index, ACTION_THETA[int(theta_index)])
    layer_state = layer_state.evolve(block)
    row, current_probability = record_layer(step, int(theta_index), layer_state, previous_probability)
    row["delta_p_target"] = float(row["p_target"] - layer_rows[-1]["p_target"])
    row["delta_p_all_optima"] = float(row["p_all_optima"] - layer_rows[-1]["p_all_optima"])
    layer_rows.append(row)
    previous_probability = current_probability

layer_dynamics_df = pd.DataFrame(layer_rows)
layer_top_states_df = pd.DataFrame(top_state_rows)

layer_final_state = np.asarray(layer_state.data, dtype=np.complex128)
layer_final_fidelity = float(abs(np.vdot(ACTION_STATE, layer_final_state)) ** 2)
if (1.0 - layer_final_fidelity) > 1e-10:
    raise RuntimeError(f"Propagação camada a camada não reproduziu o circuito: F={layer_final_fidelity}")

print(f"Fidelidade final da propagação por blocos = {layer_final_fidelity:.16f}")
display(layer_dynamics_df)
layer_dynamics_df.to_csv(ACTION_TABLE_DIR / "layer_by_layer_dynamics.csv", index=False)
layer_top_states_df.to_csv(ACTION_TABLE_DIR / "top_basis_states_each_layer.csv", index=False)


### Célula 34 — soma discreta sobre caminhos e interferência

Esta é a implementação mais próxima da ideia de Feynman nesta arquitetura.

Cada bloco atua apenas em 2 ou 3 qubits. Para cada estado-base que possui amplitude na camada atual, calculamos todos os estados-base alcançáveis pelo bloco e somamos

\[
A_{l+1}(x')=\sum_x \langle x'|U_l|x\rangle A_l(x).
\]

A recursão acima é uma **soma exata e coerente sobre estados intermediários**; ela não amostra trajetórias.

Para cada estado de saída também podemos comparar

\[
\left|\sum_r c_r\right|^2
\quad\text{com}\quad
\sum_r|c_r|^2,
\]

onde \(c_r\) são as contribuições recebidas dos predecessores. A diferença mede interferência construtiva ou destrutiva naquela camada.

O código mantém as amplitudes complexas completas. O limiar só remove zeros de máquina.


In [ ]:
# ============================================================
# 34. PATH-SUM DISCRETO EXATO POR PROGRAMAÇÃO DINÂMICA
# ============================================================


def local_block_operator(theta_index, theta_value):
    """Retorna (qubits físicos, matriz local) do mesmo bloco usado no circuito."""
    row = structure_by_theta.loc[int(theta_index)]
    i_value, l_value = int(row["i"]), int(row["l"])

    if row["ansatz_gate_type"] == "CY":
        physical_qubits = (i_value, l_value)  # slots locais 0=i, 1=l
        qc = QuantumCircuit(2)
        qc.cx(0, 1)
        qc.cry(float(theta_value), 1, 0)
        qc.cx(0, 1)
    else:
        physical_qubits = (i_value, i_value + 1, l_value)  # slots 0=i,1=i+1,2=l
        qc = QuantumCircuit(3)
        qc.cx(0, 2)
        qc.ry(float(theta_value), 0)
        qc.ccx(2, 1, 0)
        qc.ry(-float(theta_value), 0)
        qc.ccx(2, 1, 0)
        qc.cx(0, 2)

    return physical_qubits, np.asarray(Operator(qc).data, dtype=np.complex128)


def replace_local_bits(global_index, physical_qubits, local_out_index):
    """Troca apenas os bits dos qubits tocados pelo bloco."""
    out_index = int(global_index)
    for local_slot, physical_qubit in enumerate(physical_qubits):
        bit = (int(local_out_index) >> local_slot) & 1
        if bit:
            out_index |= (1 << int(physical_qubit))
        else:
            out_index &= ~(1 << int(physical_qubit))
    return out_index


def pathsum_step(amplitudes, path_counts, theta_index, theta_value):
    """Propaga uma camada e soma coerentemente todas as contribuições recebidas."""
    physical_qubits, local_u = local_block_operator(theta_index, theta_value)
    next_amplitudes = defaultdict(complex)
    next_counts = defaultdict(int)
    contributions = defaultdict(list)
    n_edges = 0

    for global_in, amp_in in amplitudes.items():
        local_in = sum(
            ((int(global_in) >> int(q)) & 1) << local_slot
            for local_slot, q in enumerate(physical_qubits)
        )
        for local_out, matrix_element in enumerate(local_u[:, local_in]):
            if abs(matrix_element) <= PATH_MATRIX_ELEMENT_TOL:
                continue
            global_out = replace_local_bits(global_in, physical_qubits, local_out)
            contribution = complex(matrix_element) * complex(amp_in)
            next_amplitudes[global_out] += contribution
            next_counts[global_out] += int(path_counts[global_in])
            contributions[global_out].append(contribution)
            n_edges += 1

    # Elimina somente resíduos numéricos após a soma coerente.
    next_amplitudes = {
        int(index): complex(amp)
        for index, amp in next_amplitudes.items()
        if abs(amp) > PATH_AMPLITUDE_TOL
    }
    next_counts = {index: int(next_counts[index]) for index in next_amplitudes}

    interference_by_output = {}
    for out_index, terms in contributions.items():
        coherent = float(abs(sum(terms)) ** 2)
        incoherent = float(sum(abs(term) ** 2 for term in terms))
        interference_by_output[int(out_index)] = coherent - incoherent

    target_terms = contributions.get(PATH_TARGET_INDEX, [])
    target_coherent = float(abs(sum(target_terms)) ** 2) if target_terms else 0.0
    target_incoherent = float(sum(abs(term) ** 2 for term in target_terms))

    return next_amplitudes, next_counts, {
        "n_transition_edges": int(n_edges),
        "target_incoming_terms": int(len(target_terms)),
        "target_interference": float(target_coherent - target_incoherent),
        "interference_l1_all_outputs": float(sum(abs(v) for v in interference_by_output.values())),
        "constructive_interference_all_outputs": float(sum(max(v, 0.0) for v in interference_by_output.values())),
        "destructive_interference_all_outputs": float(-sum(min(v, 0.0) for v in interference_by_output.values())),
    }


initial_index = int(label_to_index[initial_bitstring_qiskit])
path_amplitudes = {initial_index: 1.0 + 0.0j}
path_counts = {initial_index: 1}
pathsum_rows = []

for step, theta_index in enumerate(ORIGINAL_BLOCK_ORDER, start=1):
    p_target_before = float(abs(path_amplitudes.get(PATH_TARGET_INDEX, 0.0j)) ** 2)
    path_amplitudes, path_counts, interference = pathsum_step(
        path_amplitudes, path_counts, int(theta_index), ACTION_THETA[int(theta_index)]
    )
    p_target_after = float(abs(path_amplitudes.get(PATH_TARGET_INDEX, 0.0j)) ** 2)
    total_norm = float(sum(abs(amp) ** 2 for amp in path_amplitudes.values()))

    pathsum_rows.append({
        "step": int(step),
        "theta_index": int(theta_index),
        "is_original7": bool(theta_index in ACTIVE_THETA_INDICES),
        "n_supported_basis_states": int(len(path_amplitudes)),
        "n_histories_to_target": int(path_counts.get(PATH_TARGET_INDEX, 0)),
        "p_target_before": p_target_before,
        "p_target_after": p_target_after,
        "delta_p_target": float(p_target_after - p_target_before),
        "norm_after_step": total_norm,
        **interference,
    })

pathsum_df = pd.DataFrame(pathsum_rows)

# AUDITORIA: a soma sobre caminhos deve reconstruir exatamente o Statevector final.
pathsum_state = np.zeros(2 ** N_ASSETS, dtype=np.complex128)
for basis_index, amplitude in path_amplitudes.items():
    pathsum_state[int(basis_index)] = complex(amplitude)
pathsum_fidelity = float(abs(np.vdot(ACTION_STATE, pathsum_state)) ** 2)
pathsum_norm_error = float(abs(np.vdot(pathsum_state, pathsum_state).real - 1.0))

if (1.0 - pathsum_fidelity) > 1e-10 or pathsum_norm_error > 1e-10:
    raise RuntimeError(
        f"Path-sum não reproduziu o circuito: F={pathsum_fidelity}, erro_norma={pathsum_norm_error}"
    )

print(f"Fidelidade path-sum/circuito = {pathsum_fidelity:.16f}")
print("Número de histórias não nulas chegando ao alvo =", path_counts.get(PATH_TARGET_INDEX, 0))
display(pathsum_df)
pathsum_df.to_csv(ACTION_TABLE_DIR / "discrete_pathsum_interference_by_block.csv", index=False)


### Célula 35 — onde a amplitude e a interferência aparecem

Os dois gráficos abaixo usam a **ordem física dos blocos**.

O primeiro mostra a probabilidade do bitstring alvo depois de cada bloco. O segundo mostra a magnitude de interferência gerada na distribuição naquele bloco.

As linhas verticais marcam apenas as posições ocupadas pelos 7 θ identificados na varredura anterior. Elas não entram no cálculo do path-sum.


In [ ]:
# ============================================================
# 35. VISUALIZAÇÃO DO CAMINHO DE AMPLITUDE E INTERFERÊNCIA
# ============================================================

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(pathsum_df["step"], pathsum_df["p_target_after"], marker="o", linewidth=1.5)
for row in pathsum_df.itertuples(index=False):
    if row.is_original7:
        ax.axvline(row.step, linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("bloco aplicado")
ax.set_ylabel(f"P({PATH_TARGET_BITSTRING})")
ax.set_title("Construção da probabilidade do bitstring ótimo ao longo dos 30 blocos")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ACTION_FIGURE_DIR / "target_probability_along_blocks.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(
    pathsum_df["step"],
    pathsum_df["interference_l1_all_outputs"],
    marker="o", linewidth=1.5,
)
for row in pathsum_df.itertuples(index=False):
    if row.is_original7:
        ax.axvline(row.step, linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("bloco aplicado")
ax.set_ylabel("soma |interferência| nos estados de saída")
ax.set_title("Interferência coerente criada por cada bloco")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ACTION_FIGURE_DIR / "interference_magnitude_along_blocks.png", dpi=180)
plt.show()


### Célula 36 — comprimento Fubini–Study de uma trajetória parametrizada

Para aproximar a ideia de uma “trajetória” no espaço variacional sem introduzir uma dinâmica física artificial, usamos a interpolação periódica mais curta entre:

- o vetor original da âncora;
- o vetor reotimizado da arquitetura original.

Para

\[
\theta(s)=\theta_{\rm início}+s\,\Delta\theta,\qquad 0\le s\le1,
\]

o comprimento induzido pela métrica é

\[
L=\int_0^1\sqrt{\dot\theta^Tg(\theta)\dot\theta}\,ds.
\]

Isso é um **comprimento da trajetória escolhida**, não a prova de que ela seja uma geodésica nem uma ação física.

Além disso, construímos um endpoint que move apenas os 7 θ desde a âncora até seus valores reotimizados e medimos sua fidelidade com o estado ótimo completo. Se essa fidelidade for próxima de 1, temos uma evidência direta de que os 23 deslocamentos restantes não são necessários para alcançar o mesmo estado físico naquela região.


In [ ]:
# ============================================================
# 36. TRAJETÓRIA PARAMETRIZADA NA MÉTRICA DE FUBINI–STUDY
# ============================================================

period_vector = parameter_map_df.sort_values("theta_index")["angular_period"].to_numpy(dtype=float)
theta_path_start = np.asarray(anchors_df.iloc[0]["theta_vector"], dtype=float).copy()

# Usa o menor deslocamento angular permitido por cada periodicidade.
theta_short_delta = np.asarray([
    shortest_delta_to_target(theta_path_start[j], ACTION_THETA[j], period_vector[j])
    for j in range(N_PARAMETERS)
], dtype=float)

# Endpoint que move SOMENTE os 7 parâmetros empiricamente sensíveis.
theta_active_only_endpoint = theta_path_start.copy()
theta_active_only_endpoint[ACTIVE_THETA_INDICES] += theta_short_delta[ACTIVE_THETA_INDICES]
active_only_state = structural_state(theta_active_only_endpoint)
active_only_endpoint_fidelity = float(abs(np.vdot(ACTION_STATE, active_only_state)) ** 2)


def qgt_metric_at_theta(theta_values):
    """QGT por diferenças centrais; usado somente nos poucos pontos da trajetória."""
    psi0 = structural_state(theta_values)
    derivatives = []
    for j in range(N_PARAMETERS):
        plus, minus = np.array(theta_values, float), np.array(theta_values, float)
        plus[j] += FD_STEP
        minus[j] -= FD_STEP
        derivatives.append((structural_state(plus) - structural_state(minus)) / (2.0 * FD_STEP))
    derivatives = np.asarray(derivatives, dtype=np.complex128)
    gram_local = derivatives.conj() @ derivatives.T
    overlap_local = derivatives.conj() @ psi0
    metric = np.real(gram_local - np.outer(overlap_local, np.conj(overlap_local)))
    return 0.5 * (metric + metric.T)


parameter_path_rows = []
if RUN_PARAMETER_PATH_GEOMETRY:
    s_grid = np.linspace(0.0, 1.0, PARAMETER_PATH_POINTS)
    for s_value in s_grid:
        theta_s = theta_path_start + float(s_value) * theta_short_delta
        g_s = qgt_metric_at_theta(theta_s)
        speed_sq = float(theta_short_delta @ g_s @ theta_short_delta)
        eig_s = np.clip(np.linalg.eigvalsh(g_s), 0.0, None)
        max_eig_s = float(eig_s.max()) if len(eig_s) else 0.0
        rank_s = int(np.sum(eig_s > max(1e-14, QGT_RELATIVE_NULL_TOL * max_eig_s)))
        diag_s = np.clip(np.diag(g_s), 0.0, None)
        trace_s = float(diag_s.sum())
        active_fraction_s = float(diag_s[active_mask].sum() / trace_s) if trace_s > 0 else np.nan
        parameter_path_rows.append({
            "s": float(s_value),
            "fubini_study_speed": float(np.sqrt(max(speed_sq, 0.0))),
            "qgt_numeric_rank": rank_s,
            "active7_metric_trace_fraction": active_fraction_s,
        })

    parameter_path_df = pd.DataFrame(parameter_path_rows)
    geometric_path_length = float(np.trapezoid(
        parameter_path_df["fubini_study_speed"].to_numpy(dtype=float),
        parameter_path_df["s"].to_numpy(dtype=float),
    ))
    parameter_path_df.to_csv(ACTION_TABLE_DIR / "parameter_interpolation_geometry.csv", index=False)
else:
    parameter_path_df = pd.DataFrame()
    geometric_path_length = np.nan

path_geometry_summary_df = pd.DataFrame([{
    "active_only_endpoint_fidelity_to_full_optimum": active_only_endpoint_fidelity,
    "full_interpolation_fubini_study_length": geometric_path_length,
    "parameter_path_points": int(PARAMETER_PATH_POINTS if RUN_PARAMETER_PATH_GEOMETRY else 0),
}])

display(path_geometry_summary_df)
if not parameter_path_df.empty:
    display(parameter_path_df)
path_geometry_summary_df.to_csv(ACTION_TABLE_DIR / "parameter_path_geometry_summary.csv", index=False)


### Célula 37 — tabela integrada: os mesmos θ aparecem em diagnósticos independentes?

A última tabela não cria um score arbitrário. Ela coloca lado a lado, para cada bloco:

- sensibilidade por perturbação do 20.15;
- \(g_{ii}\) do QGT;
- curvatura de energia;
- curvatura de \(P(\mathcal X_{opt})\);
- derivada da amplitude do ótimo;
- alteração de \(P(x_f)\) quando o bloco é aplicado;
- magnitude de interferência criada pelo bloco.

A pergunta é simples: **os mesmos sete índices aparecem repetidamente em quantidades que foram calculadas por mecanismos diferentes?**

Se sim, a interpretação de uma subvariedade ativa ganha força. Se os conjuntos divergirem, a divergência é o resultado científico — por exemplo, um θ pode ser geometricamente ativo, mas energeticamente plano.


In [ ]:
# ============================================================
# 37. EVIDÊNCIA INTEGRADA — SEM SCORE ARBITRÁRIO
# ============================================================

original_probe_df = sensitivity_by_architecture_df.loc[
    sensitivity_by_architecture_df["architecture"].eq("original")
, [
    "theta_index", "sensitive_after_reorder", "max_abs_delta_p_optimal",
    "max_abs_delta_energy", "max_tvd", "dominant_bitstring_changed",
]].copy()

pathsum_by_theta_df = pathsum_df[[
    "theta_index", "step", "delta_p_target", "target_interference",
    "interference_l1_all_outputs", "n_histories_to_target",
]].copy()

integrated_evidence_df = variational_geometry_df[[
    "theta_index", "ansatz_gate_type", "logical_block_qubits", "logical_assets",
    "originally_sensitive", "qgt_diag", "geometrically_null",
    "energy_gradient_fd", "energy_curvature_fd", "energetically_flat",
    "p_target_curvature_fd", "p_optimal_curvature_fd",
]].merge(
    amplitude_influence_df[[
        "theta_index", "abs_d_amplitude_target", "d_probability_target",
        "d_probability_all_optima", "amplitude_decoupled",
    ]], on="theta_index", how="left", validate="one_to_one"
).merge(
    original_probe_df, on="theta_index", how="left", validate="one_to_one"
).merge(
    pathsum_by_theta_df, on="theta_index", how="left", validate="one_to_one"
)

# Colunas normalizadas são apenas para comparação visual; nenhum limiar científico
# depende delas e nenhum "score final" é construído.
for source_column, normalized_column in [
    ("qgt_diag", "qgt_diag_relative"),
    ("energy_curvature_fd", "abs_energy_curvature_relative"),
    ("p_optimal_curvature_fd", "abs_p_optimal_curvature_relative"),
    ("abs_d_amplitude_target", "abs_d_amplitude_relative"),
    ("interference_l1_all_outputs", "interference_l1_relative"),
]:
    values = integrated_evidence_df[source_column].abs().to_numpy(dtype=float)
    scale = float(np.nanmax(values)) if np.any(np.isfinite(values)) else 0.0
    integrated_evidence_df[normalized_column] = values / scale if scale > 0 else 0.0

integrated_evidence_df = integrated_evidence_df.sort_values("step").reset_index(drop=True)
integrated_evidence_df.to_csv(
    ACTION_TABLE_DIR / "integrated_30theta_structural_geometric_pathsum_evidence.csv",
    index=False,
)

display(integrated_evidence_df[[
    "theta_index", "step", "originally_sensitive", "sensitive_after_reorder",
    "qgt_diag", "geometrically_null", "energy_curvature_fd", "energetically_flat",
    "p_optimal_curvature_fd", "abs_d_amplitude_target", "delta_p_target",
    "target_interference", "interference_l1_all_outputs",
]])

print("7 theta originalmente identificados:", ACTIVE_THETA_INDICES)
print(
    "theta não nulos geometricamente:",
    integrated_evidence_df.loc[~integrated_evidence_df["geometrically_null"], "theta_index"].astype(int).tolist(),
)
print(
    "theta sensíveis no probe estrutural original:",
    integrated_evidence_df.loc[integrated_evidence_df["sensitive_after_reorder"], "theta_index"].astype(int).tolist(),
)


# Parte IX — compressão 30→7→3, causalidade mínima e figuras de narrativa

## O que esta parte testa

Os resultados da Parte VIII sugerem duas estruturas diferentes:

\[
30 \longrightarrow 7
\]

como **redução geométrica local no ótimo**, e

\[
7 \longrightarrow 3
\]

como possível **redução operacional da rota de bitstrings**.

Esta parte não assume que três parâmetros sejam suficientes para reproduzir o estado quântico completo. Ela mede isso explicitamente.

Os quatro testes centrais são:

1. **30 vs 7 vs 3:** energia, probabilidade ótima, fidelidade ao estado completo, rank do QGT e comprimento Fubini–Study;
2. **leave-one-out:** retirar um dos sete, reotimizar os seis restantes e medir o prejuízo;
3. **6 permutações:** testar todas as ordens possíveis de `θ25`, `θ22`, `θ17` com os 30 parâmetros livres;
4. **granularidade primitiva:** abrir somente os três blocos transportadores para verificar se existe superposição/interferência dentro de `CX/RY/CCX/CRY`.

Nenhum desses testes usa o bitstring ótimo como função objetivo. O otimizador continua vendo apenas \(\langle H\rangle\).

### Célula 38 — configuração mínima dos novos testes

Os três blocos `25, 22, 17` são definidos aqui como **hipótese operacional** porque foram os blocos que alteraram o bitstring na propagação do 20.16. A célula não os redescobre nem os força a serem suficientes.

Os controles `RUN_*` permitem pular testes caros sem alterar as células seguintes.

In [ ]:
# ============================================================
# 38. CONFIGURAÇÃO — TESTES DE COMPRESSÃO E NARRATIVA
# ============================================================

TRANSPORT_THETA_INDICES = [25, 22, 17]

# Reotimizações reduzidas. O controle de 30 parâmetros reutiliza a execução original.
RUN_COMPRESSION_TEST = True
COMPRESSION_MAXITER = 900
COMPRESSION_RESTARTS = 4

RUN_LEAVE_ONE_OUT = True
LEAVE_ONE_OUT_MAXITER = 800
LEAVE_ONE_OUT_RESTARTS = 3

RUN_TRANSPORT_PERMUTATIONS = True
PERMUTATION_MAXITER = 800
PERMUTATION_RESTARTS = 3

RUN_PRIMITIVE_PATHSUM = True
COMPARISON_PATH_POINTS = 9

STORY_ROOT = OUTPUT_ROOT / "story_30_7_3"
STORY_TABLE_DIR = STORY_ROOT / "tables"
STORY_FIGURE_DIR = STORY_ROOT / "figures"
for directory in [STORY_ROOT, STORY_TABLE_DIR, STORY_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

transport_original_order = [
    int(j) for j in ORIGINAL_BLOCK_ORDER if int(j) in TRANSPORT_THETA_INDICES
]
if transport_original_order != TRANSPORT_THETA_INDICES:
    raise RuntimeError(
        "A hipótese de ordem 25→22→17 não coincide com a ordem física atual: "
        f"{transport_original_order}"
    )

print("7 θ geométricos:", ACTIVE_THETA_INDICES)
print("3 θ transportadores:", TRANSPORT_THETA_INDICES)
print("Ordem física dos 3:", transport_original_order)
print("Saídas da Parte IX:", STORY_ROOT.resolve())

### Célula 39 — teste de suficiência: 30 vs 7 vs 3

Todos partem do **mesmo vetor inicial** `theta_path_start` e usam a mesma arquitetura original.

- `30`: reutiliza o ótimo conjunto já calculado;
- `7`: somente os sete θ identificados ficam livres;
- `3`: somente `θ25`, `θ22`, `θ17` ficam livres.

Para evitar uma interpretação enganosa, a tabela mede simultaneamente:

\[
E,\qquad P(\mathcal X_{opt}),\qquad
F(|\psi_k\rangle,|\psi_{30}\rangle),\qquad
\operatorname{rank}(QGT).
\]

Assim, `P_optimal≈1` não é confundido automaticamente com igualdade do estado quântico completo.

In [ ]:
# ============================================================
# 39. COMPRESSÃO 30 vs 7 vs 3 — REOTIMIZAÇÃO E FIDELIDADE
# ============================================================

compression_runs = {"30_all": architecture_runs["original"]}

if RUN_COMPRESSION_TEST:
    compression_runs["7_active"] = optimize_block_order(
        label="compression_7_active",
        block_order=ORIGINAL_BLOCK_ORDER,
        theta_start=theta_path_start,
        free_indices=ACTIVE_THETA_INDICES,
        maxiter=COMPRESSION_MAXITER,
        n_restarts=COMPRESSION_RESTARTS,
        seed=RANDOM_SEED + 20170,
    )
    compression_runs["3_transport"] = optimize_block_order(
        label="compression_3_transport",
        block_order=ORIGINAL_BLOCK_ORDER,
        theta_start=theta_path_start,
        free_indices=TRANSPORT_THETA_INDICES,
        maxiter=COMPRESSION_MAXITER,
        n_restarts=COMPRESSION_RESTARTS,
        seed=RANDOM_SEED + 20171,
    )

compression_rows = []
for label, run in compression_runs.items():
    measured = measure_structural_circuit(
        run["circuit"], run["parameter_objects"], run["theta_opt"],
        keep_probabilities=True,
    )
    state = np.asarray(measured["statevector"], dtype=np.complex128)
    fidelity = float(abs(np.vdot(ACTION_STATE, state)) ** 2)

    # O rank é medido no endpoint encontrado por cada parametrização reduzida.
    g_endpoint = qgt_metric_at_theta(np.asarray(run["theta_opt"], dtype=float))
    eig = np.clip(np.linalg.eigvalsh(g_endpoint), 0.0, None)
    max_eig = float(eig.max()) if len(eig) else 0.0
    qgt_rank = int(np.sum(eig > max(1e-14, QGT_RELATIVE_NULL_TOL * max_eig)))

    compression_rows.append({
        "model": label,
        "n_free_parameters": {"30_all": 30, "7_active": 7, "3_transport": 3}[label],
        "expected_energy": measured["expected_energy"],
        "energy_gap": measured["energy_gap"],
        "p_optimal": measured["p_optimal"],
        "fidelity_to_full30_state": fidelity,
        "dominant_bitstring": measured["dominant_bitstring"],
        "dominant_probability": measured["dominant_probability"],
        "dominant_is_exact_optimum": measured["dominant_is_exact_optimum"],
        "nearest_optimal_hamming": measured["nearest_optimal_hamming"],
        "qgt_numeric_rank_at_endpoint": qgt_rank,
        "nfev_total": run["nfev_total"],
    })

compression_summary_df = pd.DataFrame(compression_rows).sort_values(
    "n_free_parameters", ascending=False
).reset_index(drop=True)

display(compression_summary_df)
compression_summary_df.to_csv(
    STORY_TABLE_DIR / "compression_30_vs_7_vs_3.csv", index=False
)

### Célula 40 — gráficos da compressão

São mostradas separadamente três perguntas diferentes:

1. os modelos chegam ao mesmo **estado**? → fidelidade;
2. chegam ao mesmo **bitstring ótimo**? → \(P_{opt}\);
3. chegam à mesma **energia**? → gap energético.

Não são combinadas em um único score.

In [ ]:
# ============================================================
# 40. FIGURAS — 30 vs 7 vs 3
# ============================================================

if not compression_summary_df.empty:
    labels = compression_summary_df["model"].tolist()
    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.bar(x, compression_summary_df["fidelity_to_full30_state"])
    ax.set_xticks(x, labels)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Fidelidade com |ψ₃₀⟩")
    ax.set_title("O estado completo sobrevive à compressão?")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "compression_fidelity_30_7_3.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.bar(x, compression_summary_df["p_optimal"])
    ax.set_xticks(x, labels)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("P(ótimos)")
    ax.set_title("A solução combinatória sobrevive à compressão?")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "compression_p_optimal_30_7_3.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.bar(x, compression_summary_df["energy_gap"])
    ax.set_xticks(x, labels)
    ax.set_ylabel("E - E_exata")
    ax.set_title("Gap energético após reotimização")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "compression_energy_gap_30_7_3.png", dpi=180)
    plt.show()

### Célula 41 — comprimento Fubini–Study das trajetórias 30, 7 e 3

A comparação de comprimentos só é interpretada junto com a fidelidade do endpoint.

Uma trajetória curta que termina em outro estado **não** é evidência de caminho melhor.

Para cada conjunto de parâmetros livres, usa-se o menor deslocamento angular até o endpoint encontrado e integra-se

\[
L[\gamma]=\int_0^1\sqrt{\dot\theta^T g(\theta)\dot\theta}\,ds.
\]

Isto ainda é o comprimento da interpolação escolhida, não uma geodésica otimizada.

In [ ]:
# ============================================================
# 41. COMPARAÇÃO GEOMÉTRICA DAS TRAJETÓRIAS 30 / 7 / 3
# ============================================================


def fubini_study_path_between(theta_start, theta_target, moving_indices, label, n_points=9):
    """Mede a trajetória interpolada; não procura uma geodésica."""
    theta_start = np.asarray(theta_start, dtype=float).copy()
    theta_target = np.asarray(theta_target, dtype=float).copy()
    moving_indices = np.asarray(list(moving_indices), dtype=int)

    delta = np.zeros(N_PARAMETERS, dtype=float)
    for j in moving_indices:
        delta[j] = shortest_delta_to_target(
            theta_start[j], theta_target[j], period_vector[j]
        )

    rows = []
    for s_value in np.linspace(0.0, 1.0, int(n_points)):
        theta_s = theta_start + float(s_value) * delta
        g_s = qgt_metric_at_theta(theta_s)
        speed_sq = float(delta @ g_s @ delta)
        eig_s = np.clip(np.linalg.eigvalsh(g_s), 0.0, None)
        max_eig_s = float(eig_s.max()) if len(eig_s) else 0.0
        rank_s = int(np.sum(eig_s > max(1e-14, QGT_RELATIVE_NULL_TOL * max_eig_s)))
        rows.append({
            "model": label,
            "s": float(s_value),
            "fubini_study_speed": float(np.sqrt(max(speed_sq, 0.0))),
            "qgt_numeric_rank": rank_s,
        })

    path_df = pd.DataFrame(rows)
    length = float(np.trapezoid(
        path_df["fubini_study_speed"].to_numpy(dtype=float),
        path_df["s"].to_numpy(dtype=float),
    ))

    endpoint_theta = theta_start + delta
    endpoint_state = structural_state(endpoint_theta)
    endpoint_fidelity = float(abs(np.vdot(ACTION_STATE, endpoint_state)) ** 2)
    return path_df, length, endpoint_fidelity


comparison_path_frames = []
comparison_path_rows = []
free_map = {
    "30_all": list(range(N_PARAMETERS)),
    "7_active": ACTIVE_THETA_INDICES,
    "3_transport": TRANSPORT_THETA_INDICES,
}

for label, run in compression_runs.items():
    path_df, length, endpoint_fidelity = fubini_study_path_between(
        theta_path_start,
        np.asarray(run["theta_opt"], dtype=float),
        free_map[label],
        label,
        n_points=COMPARISON_PATH_POINTS,
    )
    comparison_path_frames.append(path_df)
    comparison_path_rows.append({
        "model": label,
        "n_free_parameters": len(free_map[label]),
        "fubini_study_length": length,
        "path_endpoint_fidelity_to_full30": endpoint_fidelity,
    })

comparison_path_df = pd.concat(comparison_path_frames, ignore_index=True)
comparison_path_summary_df = pd.DataFrame(comparison_path_rows).sort_values(
    "n_free_parameters", ascending=False
)

display(comparison_path_summary_df)
comparison_path_summary_df.to_csv(
    STORY_TABLE_DIR / "fubini_study_length_30_vs_7_vs_3.csv", index=False
)
comparison_path_df.to_csv(
    STORY_TABLE_DIR / "fubini_study_paths_30_vs_7_vs_3.csv", index=False
)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(
    comparison_path_summary_df["model"],
    comparison_path_summary_df["fubini_study_length"],
)
ax.set_ylabel("Comprimento Fubini–Study")
ax.set_title("Comprimento da trajetória interpolada: 30 vs 7 vs 3")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(STORY_FIGURE_DIR / "fubini_study_length_30_7_3.png", dpi=180)
plt.show()

### Célula 42 — leave-one-out dos sete θ

Agora a pergunta é diferente de “o θ é sensível quando perturbado?”.

Para cada um dos sete:

1. congela-se **somente aquele θ** no valor inicial;
2. os outros seis θ ativos ficam livres;
3. os 23 restantes também permanecem congelados;
4. a energia é reotimizada;
5. mede-se fidelidade, \(P_{opt}\), energia e bitstring.

Se a retirada de `θ25`, `θ22` ou `θ17` produzir uma perda muito maior que a retirada dos outros quatro, isso separa **atividade geométrica** de **necessidade operacional**.

In [ ]:
# ============================================================
# 42. LEAVE-ONE-OUT DOS 7 PARÂMETROS ATIVOS
# ============================================================

leave_one_out_rows = []
leave_one_out_runs = {}

if RUN_LEAVE_ONE_OUT:
    for omitted_theta in ACTIVE_THETA_INDICES:
        free_six = [j for j in ACTIVE_THETA_INDICES if j != omitted_theta]
        print(f"Leave-one-out: congelando theta_{omitted_theta}; livres={free_six}")

        run = optimize_block_order(
            label=f"leave_out_theta_{omitted_theta}",
            block_order=ORIGINAL_BLOCK_ORDER,
            theta_start=theta_path_start,
            free_indices=free_six,
            maxiter=LEAVE_ONE_OUT_MAXITER,
            n_restarts=LEAVE_ONE_OUT_RESTARTS,
            seed=RANDOM_SEED + 21000 + int(omitted_theta),
        )
        leave_one_out_runs[int(omitted_theta)] = run

        measured = measure_structural_circuit(
            run["circuit"], run["parameter_objects"], run["theta_opt"],
            keep_probabilities=True,
        )
        state = np.asarray(measured["statevector"], dtype=np.complex128)
        fidelity = float(abs(np.vdot(ACTION_STATE, state)) ** 2)

        leave_one_out_rows.append({
            "omitted_theta": int(omitted_theta),
            "is_transport_theta": bool(omitted_theta in TRANSPORT_THETA_INDICES),
            "expected_energy": measured["expected_energy"],
            "energy_gap": measured["energy_gap"],
            "p_optimal": measured["p_optimal"],
            "fidelity_to_full30_state": fidelity,
            "dominant_bitstring": measured["dominant_bitstring"],
            "dominant_probability": measured["dominant_probability"],
            "dominant_is_exact_optimum": measured["dominant_is_exact_optimum"],
            "nearest_optimal_hamming": measured["nearest_optimal_hamming"],
            "nfev_total": run["nfev_total"],
        })

leave_one_out_df = pd.DataFrame(leave_one_out_rows).sort_values("omitted_theta")
if not leave_one_out_df.empty:
    display(leave_one_out_df)
    leave_one_out_df.to_csv(STORY_TABLE_DIR / "leave_one_out_active7.csv", index=False)

### Célula 43 — gráfico de necessidade dos sete

Os gráficos usam diretamente o resultado do leave-one-out.

- `1 − fidelidade` mede perda do **estado quântico completo**;
- `1 − P_optimal` mede perda da **solução combinatória**.

Isso permite que um parâmetro seja importante para o estado sem necessariamente ser indispensável para o bitstring dominante.

In [ ]:
# ============================================================
# 43. FIGURAS — NECESSIDADE INDIVIDUAL DOS 7
# ============================================================

if not leave_one_out_df.empty:
    labels = [f"θ{j}" for j in leave_one_out_df["omitted_theta"]]
    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(x, 1.0 - leave_one_out_df["fidelity_to_full30_state"].to_numpy(dtype=float))
    ax.set_xticks(x, labels)
    ax.set_ylabel("1 - fidelidade")
    ax.set_title("Quanto o estado muda quando cada θ é retirado?")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "leave_one_out_fidelity_loss.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(x, 1.0 - leave_one_out_df["p_optimal"].to_numpy(dtype=float))
    ax.set_xticks(x, labels)
    ax.set_ylabel("1 - P(ótimos)")
    ax.set_title("Quanto a solução combinatória depende de cada θ?")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "leave_one_out_p_optimal_loss.png", dpi=180)
    plt.show()

### Célula 44 — todas as seis ordens possíveis dos três blocos transportadores

Os três blocos ocupam os **mesmos três slots físicos** usados originalmente; somente suas identidades são permutadas.

Os outros 27 blocos mantêm sua ordem.

Depois da mudança, **todos os 30 θ são liberados novamente**. Isto torna o teste conservador: uma ordem só é considerada ruim depois de dar ao circuito inteiro a oportunidade de compensá-la variacionalmente.

In [ ]:
# ============================================================
# 44. SEIS PERMUTAÇÕES DE θ25, θ22, θ17
# ============================================================

transport_slots = [
    position for position, theta_index in enumerate(ORIGINAL_BLOCK_ORDER)
    if theta_index in TRANSPORT_THETA_INDICES
]


def order_with_transport_permutation(permutation):
    order = ORIGINAL_BLOCK_ORDER.copy()
    for slot, theta_index in zip(transport_slots, permutation):
        order[int(slot)] = int(theta_index)
    return order


permutation_rows = []
permutation_runs = {}

if RUN_TRANSPORT_PERMUTATIONS:
    for permutation_id, permutation in enumerate(permutations(TRANSPORT_THETA_INDICES)):
        permutation = tuple(map(int, permutation))
        label = "→".join(map(str, permutation))
        print("Testando ordem:", label)

        if list(permutation) == transport_original_order:
            run = architecture_runs["original"]
        else:
            run = optimize_block_order(
                label=f"transport_perm_{'_'.join(map(str, permutation))}",
                block_order=order_with_transport_permutation(permutation),
                theta_start=ACTION_THETA,
                free_indices=range(N_PARAMETERS),
                maxiter=PERMUTATION_MAXITER,
                n_restarts=PERMUTATION_RESTARTS,
                seed=RANDOM_SEED + 22000 + permutation_id,
            )
        permutation_runs[permutation] = run

        measured = measure_structural_circuit(
            run["circuit"], run["parameter_objects"], run["theta_opt"],
            keep_probabilities=True,
        )
        state = np.asarray(measured["statevector"], dtype=np.complex128)
        fidelity = float(abs(np.vdot(ACTION_STATE, state)) ** 2)

        permutation_rows.append({
            "order": label,
            "is_original_order": bool(list(permutation) == transport_original_order),
            "expected_energy": measured["expected_energy"],
            "energy_gap": measured["energy_gap"],
            "p_optimal": measured["p_optimal"],
            "fidelity_to_original_optimum_state": fidelity,
            "dominant_bitstring": measured["dominant_bitstring"],
            "dominant_probability": measured["dominant_probability"],
            "nearest_optimal_hamming": measured["nearest_optimal_hamming"],
            "nfev_total": run["nfev_total"],
        })

transport_permutations_df = pd.DataFrame(permutation_rows)
if not transport_permutations_df.empty:
    display(transport_permutations_df.sort_values("energy_gap"))
    transport_permutations_df.to_csv(
        STORY_TABLE_DIR / "transport3_all_six_permutations.csv", index=False
    )

### Célula 45 — gráfico da precedência operacional

Este gráfico é deliberadamente simples: uma barra por ordem possível.

Se `25→22→17` for a única ordem que recupera a solução, a precedência é muito forte. Se várias ordens funcionarem, o resultado correto será identificar uma **classe de ordens equivalentes**, não forçar unicidade.

In [ ]:
# ============================================================
# 45. FIGURAS — AS 6 ORDENS POSSÍVEIS
# ============================================================

if not transport_permutations_df.empty:
    order_df = transport_permutations_df.sort_values("order").reset_index(drop=True)
    x = np.arange(len(order_df))

    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.bar(x, order_df["p_optimal"])
    ax.set_xticks(x, order_df["order"], rotation=30, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("P(ótimos)")
    ax.set_title("Todas as ordens de θ25, θ22 e θ17")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "transport3_permutations_p_optimal.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.bar(x, order_df["fidelity_to_original_optimum_state"])
    ax.set_xticks(x, order_df["order"], rotation=30, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Fidelidade com o ótimo original")
    ax.set_title("A ordem preserva o estado quântico completo?")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "transport3_permutations_fidelity.png", dpi=180)
    plt.show()

### Célula 46 — path-sum dentro das portas primitivas dos três transportadores

O path-sum anterior tratava cada bloco lógico como uma única unidade. Portanto, interferência nula entre blocos não exclui superposição temporária **dentro** de um `CCY`.

Esta célula abre somente `θ25`, `θ22` e `θ17` em:

`CX → RY → CCX → RY(-θ) → CCX → CX`.

Os demais 27 blocos continuam sendo aplicados como unidades completas apenas para levar o estado até o ponto correto.

Depois de **cada porta primitiva** dos três blocos medimos:

- número de estados-base no suporte;
- bitstring dominante;
- probabilidade do alvo;
- número de histórias que chegam ao alvo;
- interferência coerente recebida pelos estados de saída.

Uma auditoria final exige que a propagação híbrida reproduza exatamente `ACTION_STATE`.

In [ ]:
# ============================================================
# 46. PATH-SUM EM GRANULARIDADE PRIMITIVA DOS 3 BLOCOS
# ============================================================


def gate_matrix_1q_ry(angle):
    qc = QuantumCircuit(1)
    qc.ry(float(angle), 0)
    return np.asarray(Operator(qc).data, dtype=np.complex128)


def gate_matrix_2q_cx():
    qc = QuantumCircuit(2)
    qc.cx(0, 1)
    return np.asarray(Operator(qc).data, dtype=np.complex128)


def gate_matrix_2q_cry(angle):
    qc = QuantumCircuit(2)
    qc.cry(float(angle), 1, 0)  # local 1=controle, local 0=alvo
    return np.asarray(Operator(qc).data, dtype=np.complex128)


def gate_matrix_3q_ccx():
    qc = QuantumCircuit(3)
    qc.ccx(2, 1, 0)  # locais: 2=l, 1=i+1, 0=i
    return np.asarray(Operator(qc).data, dtype=np.complex128)


def primitive_sequence(theta_index, theta_value):
    """Devolve exatamente as portas primitivas do bloco lógico original."""
    row = structure_by_theta.loc[int(theta_index)]
    i_value, l_value = int(row["i"]), int(row["l"])

    if row["ansatz_gate_type"] == "CY":
        return [
            ("CX", (i_value, l_value), gate_matrix_2q_cx()),
            ("CRY(+θ)", (i_value, l_value), gate_matrix_2q_cry(theta_value)),
            ("CX", (i_value, l_value), gate_matrix_2q_cx()),
        ]

    return [
        ("CX", (i_value, l_value), gate_matrix_2q_cx()),
        ("RY(+θ)", (i_value,), gate_matrix_1q_ry(theta_value)),
        ("CCX", (i_value, i_value + 1, l_value), gate_matrix_3q_ccx()),
        ("RY(-θ)", (i_value,), gate_matrix_1q_ry(-theta_value)),
        ("CCX", (i_value, i_value + 1, l_value), gate_matrix_3q_ccx()),
        ("CX", (i_value, l_value), gate_matrix_2q_cx()),
    ]


def pathsum_matrix_step(amplitudes, path_counts, physical_qubits, local_u):
    """Mesma soma coerente do path-sum, agora para uma única porta primitiva."""
    next_amplitudes = defaultdict(complex)
    next_counts = defaultdict(int)
    contributions = defaultdict(list)

    for global_in, amp_in in amplitudes.items():
        local_in = sum(
            ((int(global_in) >> int(q)) & 1) << local_slot
            for local_slot, q in enumerate(physical_qubits)
        )
        for local_out, matrix_element in enumerate(local_u[:, local_in]):
            if abs(matrix_element) <= PATH_MATRIX_ELEMENT_TOL:
                continue
            global_out = replace_local_bits(global_in, physical_qubits, local_out)
            term = complex(matrix_element) * complex(amp_in)
            next_amplitudes[global_out] += term
            next_counts[global_out] += int(path_counts[global_in])
            contributions[global_out].append(term)

    next_amplitudes = {
        int(index): complex(amp)
        for index, amp in next_amplitudes.items()
        if abs(amp) > PATH_AMPLITUDE_TOL
    }
    next_counts = {index: int(next_counts[index]) for index in next_amplitudes}

    interference = {}
    for out_index, terms in contributions.items():
        coherent = float(abs(sum(terms)) ** 2)
        incoherent = float(sum(abs(term) ** 2 for term in terms))
        interference[int(out_index)] = coherent - incoherent

    return next_amplitudes, next_counts, {
        "target_interference": float(interference.get(PATH_TARGET_INDEX, 0.0)),
        "interference_l1_all_outputs": float(sum(abs(v) for v in interference.values())),
    }


primitive_rows = []
if RUN_PRIMITIVE_PATHSUM:
    amplitudes = {initial_index: 1.0 + 0.0j}
    counts = {initial_index: 1}
    primitive_global_step = 0

    for block_step, theta_index in enumerate(ORIGINAL_BLOCK_ORDER, start=1):
        theta_index = int(theta_index)

        if theta_index not in TRANSPORT_THETA_INDICES:
            amplitudes, counts, _ = pathsum_step(
                amplitudes, counts, theta_index, ACTION_THETA[theta_index]
            )
            continue

        for primitive_index, (gate_name, physical_qubits, matrix) in enumerate(
            primitive_sequence(theta_index, ACTION_THETA[theta_index]), start=1
        ):
            primitive_global_step += 1
            p_before = float(abs(amplitudes.get(PATH_TARGET_INDEX, 0.0j)) ** 2)
            support_before = int(len(amplitudes))

            amplitudes, counts, interference = pathsum_matrix_step(
                amplitudes, counts, physical_qubits, matrix
            )

            p_after = float(abs(amplitudes.get(PATH_TARGET_INDEX, 0.0j)) ** 2)
            dominant_index = max(amplitudes, key=lambda idx: abs(amplitudes[idx]) ** 2)

            primitive_rows.append({
                "primitive_global_step": primitive_global_step,
                "block_step": int(block_step),
                "theta_index": theta_index,
                "primitive_index_in_block": int(primitive_index),
                "gate": gate_name,
                "physical_qubits": tuple(map(int, physical_qubits)),
                "n_supported_before": support_before,
                "n_supported_after": int(len(amplitudes)),
                "dominant_bitstring_after": str(all_labels[int(dominant_index)]),
                "dominant_probability_after": float(abs(amplitudes[dominant_index]) ** 2),
                "p_target_before": p_before,
                "p_target_after": p_after,
                "delta_p_target": float(p_after - p_before),
                "n_histories_to_target": int(counts.get(PATH_TARGET_INDEX, 0)),
                **interference,
            })

    primitive_state = np.zeros(2 ** N_ASSETS, dtype=np.complex128)
    for basis_index, amplitude in amplitudes.items():
        primitive_state[int(basis_index)] = complex(amplitude)
    primitive_final_fidelity = float(abs(np.vdot(ACTION_STATE, primitive_state)) ** 2)

    if (1.0 - primitive_final_fidelity) > 1e-10:
        raise RuntimeError(
            "A decomposição primitiva não reproduziu o circuito completo: "
            f"F={primitive_final_fidelity:.16f}"
        )

primitive_pathsum_df = pd.DataFrame(primitive_rows)
if not primitive_pathsum_df.empty:
    print(f"Fidelidade final da decomposição primitiva = {primitive_final_fidelity:.16f}")
    display(primitive_pathsum_df)
    primitive_pathsum_df.to_csv(
        STORY_TABLE_DIR / "transport3_primitive_pathsum.csv", index=False
    )

### Célula 47 — gráficos intrabloco: suporte e interferência

O primeiro gráfico responde diretamente se os blocos criam superposição temporária:

\[
N_{support}>1.
\]

O segundo verifica se múltiplas contribuições coerentes realmente se recombinam em um mesmo estado de saída.

Se o suporte permanecer 1 e a interferência permanecer 0, a rota é determinística também na granularidade primitiva. Se o suporte abrir e depois fechar, o mecanismo quântico está dentro dos blocos.

In [ ]:
# ============================================================
# 47. FIGURAS — DINÂMICA PRIMITIVA DOS 3 TRANSPORTADORES
# ============================================================

if not primitive_pathsum_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4.4))
    ax.plot(
        primitive_pathsum_df["primitive_global_step"],
        primitive_pathsum_df["n_supported_after"],
        marker="o",
    )
    ax.set_xlabel("Porta primitiva dentro de θ25, θ22 e θ17")
    ax.set_ylabel("Número de estados no suporte")
    ax.set_title("A superposição aparece dentro dos três blocos transportadores?")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "primitive_support_transport3.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 4.4))
    ax.plot(
        primitive_pathsum_df["primitive_global_step"],
        primitive_pathsum_df["interference_l1_all_outputs"],
        marker="o",
    )
    ax.set_xlabel("Porta primitiva dentro de θ25, θ22 e θ17")
    ax.set_ylabel("Σ |interferência|")
    ax.set_title("Interferência coerente em granularidade primitiva")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "primitive_interference_transport3.png", dpi=180)
    plt.show()

### Célula 48 — contração geométrica ao longo da trajetória

Esta figura conta uma história diferente da rota de bitstrings.

O rank do QGT responde **quantas direções tangentes independentes existem localmente**. A fração do traço nos sete responde **quanto da geometria local está concentrada no subconjunto final**.

As duas curvas são mantidas separadas para não misturar escalas.

In [ ]:
# ============================================================
# 48. FIGURAS — CONTRAÇÃO DO QGT NA TRAJETÓRIA
# ============================================================

if not parameter_path_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(parameter_path_df["s"], parameter_path_df["qgt_numeric_rank"], marker="o")
    ax.set_xlabel("s na trajetória interpolada")
    ax.set_ylabel("rank numérico do QGT")
    ax.set_title("Dimensão tangente local ao longo da trajetória")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "qgt_rank_along_path.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(
        parameter_path_df["s"],
        parameter_path_df["active7_metric_trace_fraction"],
        marker="o",
    )
    ax.set_xlabel("s na trajetória interpolada")
    ax.set_ylabel("fração do traço do QGT nos 7")
    ax.set_ylim(0.0, 1.05)
    ax.set_title("Concentração geométrica nos sete parâmetros finais")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(STORY_FIGURE_DIR / "active7_qgt_trace_fraction_along_path.png", dpi=180)
    plt.show()

### Célula 49 — figura da rota efetiva de bitstrings

A figura é construída diretamente de `layer_dynamics_df`: só entram passos em que o bitstring dominante realmente mudou.

Assim, nenhuma transição é escrita manualmente no gráfico. Se outra instância produzir outra rota, a figura muda automaticamente.

In [ ]:
# ============================================================
# 49. FIGURA — ROTA COMPUTACIONAL EXTRAÍDA DOS DADOS
# ============================================================

route_rows = [layer_dynamics_df.iloc[0].to_dict()]
for row in layer_dynamics_df.iloc[1:].itertuples(index=False):
    if str(row.dominant_bitstring) != str(route_rows[-1]["dominant_bitstring"]):
        route_rows.append(row._asdict())

route_df = pd.DataFrame(route_rows)
route_df.to_csv(STORY_TABLE_DIR / "dominant_bitstring_route.csv", index=False)

display(route_df[["step", "theta_index", "dominant_bitstring", "dominant_probability"]])

fig, ax = plt.subplots(figsize=(12, 3.4))
ax.axis("off")
positions = np.linspace(0.08, 0.92, len(route_df))

for idx, row in route_df.reset_index(drop=True).iterrows():
    ax.text(
        positions[idx], 0.58, str(row["dominant_bitstring"]),
        ha="center", va="center", fontsize=12,
        bbox={"boxstyle": "round,pad=0.35", "fill": False},
        transform=ax.transAxes,
    )
    if idx > 0:
        theta_label = f"θ{int(row['theta_index'])}" if pd.notna(row["theta_index"]) else ""
        ax.annotate(
            "", xy=(positions[idx] - 0.055, 0.58), xytext=(positions[idx-1] + 0.055, 0.58),
            xycoords=ax.transAxes, textcoords=ax.transAxes,
            arrowprops={"arrowstyle": "->", "linewidth": 1.5},
        )
        ax.text(
            (positions[idx] + positions[idx-1]) / 2, 0.70, theta_label,
            ha="center", va="center", fontsize=11, transform=ax.transAxes,
        )

ax.set_title("Rota efetiva dos bitstrings dominantes", pad=12)
fig.tight_layout()
fig.savefig(STORY_FIGURE_DIR / "dominant_bitstring_route.png", dpi=180)
plt.show()

### Célula 50 — heatmap final de evidências independentes

Cada coluna é normalizada **separadamente** apenas para visualização. Não existe soma, média ou score final.

As linhas são os 30 θ e as colunas respondem perguntas diferentes:

- sensibilidade no probe original;
- intensidade geométrica local \(g_{ii}\);
- curvatura energética;
- curvatura de \(P_{opt}\);
- perda no leave-one-out;
- fração de posições que falham;
- participação na rota dos três transportadores.

Se os mesmos θ aparecerem repetidamente em diagnósticos independentes, o padrão fica visualmente evidente sem impor pesos arbitrários.

In [ ]:
# ============================================================
# 50. HEATMAP — EVIDÊNCIAS INDEPENDENTES, SEM SCORE
# ============================================================

story_evidence_df = integrated_evidence_df[[
    "theta_index", "sensitive_after_reorder", "qgt_diag",
    "energy_curvature_fd", "p_optimal_curvature_fd",
]].copy()

# Leave-one-out existe apenas para os sete; zero significa "não testado como removido" fora deles.
loo_loss = {}
if not leave_one_out_df.empty:
    loo_loss = {
        int(row.omitted_theta): float(1.0 - row.fidelity_to_full30_state)
        for row in leave_one_out_df.itertuples(index=False)
    }
story_evidence_df["leave_one_out_fidelity_loss"] = story_evidence_df["theta_index"].map(
    lambda j: loo_loss.get(int(j), 0.0)
)

# Fração de posições que deixam de produzir o ótimo dominante.
position_failure = {}
if RUN_FULL_POSITION_SCAN and not position_scan_df.empty:
    for theta_index, group in position_scan_df.groupby("moved_theta"):
        position_failure[int(theta_index)] = float(1.0 - group["dominant_is_exact_optimum"].mean())
story_evidence_df["position_failure_fraction"] = story_evidence_df["theta_index"].map(
    lambda j: position_failure.get(int(j), 0.0)
)

story_evidence_df["transport_route"] = story_evidence_df["theta_index"].isin(
    TRANSPORT_THETA_INDICES
).astype(float)

plot_columns = [
    "sensitive_after_reorder", "qgt_diag", "energy_curvature_fd",
    "p_optimal_curvature_fd", "leave_one_out_fidelity_loss",
    "position_failure_fraction", "transport_route",
]
plot_labels = [
    "probe", "QGT", "E''", "P''", "leave-one-out", "posição", "rota 3",
]

normalized_columns = []
for column in plot_columns:
    values = story_evidence_df[column].astype(float).abs().to_numpy()
    scale = float(np.nanmax(values)) if np.any(np.isfinite(values)) else 0.0
    normalized_columns.append(values / scale if scale > 0 else np.zeros_like(values))

story_matrix = np.column_stack(normalized_columns)
story_evidence_df.to_csv(STORY_TABLE_DIR / "story_evidence_30theta.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 10))
image = ax.imshow(story_matrix, aspect="auto", origin="upper", vmin=0.0, vmax=1.0)
ax.set_yticks(range(len(story_evidence_df)))
ax.set_yticklabels([f"θ{int(j)}" for j in story_evidence_df["theta_index"]])
ax.set_xticks(range(len(plot_labels)))
ax.set_xticklabels(plot_labels, rotation=35, ha="right")
ax.set_title("Evidências independentes por parâmetro — colunas normalizadas separadamente")
fig.colorbar(image, ax=ax, label="intensidade relativa dentro de cada diagnóstico")
fig.tight_layout()
fig.savefig(STORY_FIGURE_DIR / "integrated_story_heatmap_30theta.png", dpi=180)
plt.show()

## Leitura esperada da Parte IX

A Parte IX foi desenhada para distinguir quatro resultados possíveis, sem decidir antecipadamente qual deles ocorrerá:

1. **7 reproduzem o estado e 3 não:** `7 = estado quântico completo`, `3 = rota do bitstring`;
2. **3 também reproduzem o estado:** a compressão efetiva é ainda maior que a indicada pelo rank local;
3. **várias permutações funcionam:** existe uma classe de ordens equivalentes, e não uma ordem causal única;
4. **o suporte abre dentro dos CCY:** a ausência de interferência entre blocos esconde uma dinâmica intrabloco coerente.

A conclusão deve seguir as tabelas produzidas, não o diagrama esperado.

## Checklist de auditoria antes de qualquer interpretação física

1. **Reconstrução por blocos:** a fidelidade entre o ansatz original e a reconstrução deve ser \(\approx1\). Se falhar, testes de posição, permutações e path-sum não devem ser usados.
2. **Controle de reotimização:** a arquitetura `original` precisa recuperar a solução conhecida antes das comparações estruturais.
3. **Gradiente no mínimo:** `energy_gradient_fd` deve ser pequeno de forma geral; não use o gradiente para escolher os sete. Compare principalmente `qgt_diag` e curvaturas.
4. **Rank do QGT:** interprete `rank=7` como propriedade **local do ponto avaliado**, não como dimensão global do ansatz.
5. **30 vs 7 vs 3:** não confunda `P_optimal=1` com fidelidade unitária. Compare sempre as duas colunas.
6. **Comprimento Fubini–Study:** compare comprimentos somente quando os endpoints têm fidelidade compatível. A interpolação não é uma geodésica por construção.
7. **Permutações:** uma ordem que falha após atingir o limite de avaliações deve ser testada com reinícios adicionais antes de ser declarada inacessível.
8. **Path-sum primitivo:** a fidelidade final da decomposição primitiva deve ser \(\approx1\). Só então `n_supported_after` e interferência podem ser interpretados.
9. **Heatmap final:** as colunas são normalizadas separadamente. O gráfico é uma visualização de diagnósticos independentes, não um score físico.

# Parte X — padrão generalizável e interface para Transformer

## Célula 51 — qual é a regra que estamos procurando?

A observação `30 → 7 → 3` é apenas **uma instância**. O objeto científico de interesse é uma relação que sobreviva quando o problema muda.

Para um problema de escolha de $k$ itens entre $n$, todos os bitstrings válidos têm peso de Hamming $k$. Portanto, entre dois estados válidos,

$$
|x|_1=|y|_1=k,
$$

e qualquer mudança de seleção ocorre em pares `1→0` e `0→1`. Assim,

$$
d_H(x,y)=2m,
$$

onde $m$ é o número mínimo de **substituições** necessárias entre as duas escolhas.

No caso atual,

$$
1111000000\rightarrow1001011000
$$

possui $d_H=4$, logo são necessárias pelo menos duas substituições. O circuito realizou essas substituições por uma rota de três blocos porque uma das excitações passou por um nó intermediário.

Essa descrição é transferível para `3 em 10`, `5 em 10` ou outro $k$: mudam o número de parâmetros e a solução, mas continuam fazendo sentido:

- $k/n$;
- $d_H/2$ substituições mínimas;
- o grafo de transporte entre posições ocupadas e vazias;
- o posto local do QGT;
- quais blocos formam o subespaço ativo;
- quantas histórias primitivas contribuem;
- suas amplitudes e fases;
- a interferência coerente que seleciona o estado final.

> **Regra para generalização:** o Transformer não deve receber `theta_22` como significado físico. Ele deve receber um **token de bloco** descrevendo a porta, os qubits/ativos tocados, posição relativa, Hamiltoniano local, cardinalidade e contexto do problema.


In [ ]:
# ============================================================
# 51. CONFIGURAÇÃO — HISTÓRIAS RESOLVIDAS + PADRÃO TRANSFERÍVEL
# ============================================================

PATTERN_ROOT = OUTPUT_ROOT / "pattern_generalization_transformer"
PATTERN_TABLE_DIR = PATTERN_ROOT / "tables"
PATTERN_FIGURE_DIR = PATTERN_ROOT / "figures"
for directory in [PATTERN_ROOT, PATTERN_TABLE_DIR, PATTERN_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# A ordem correta é comparada com a reversa SEM alterar os valores dos θ.
# Isso isola o efeito da ordem do efeito da reotimização.
HISTORY_ORDER_COMPARISONS = {
    "correct_25_22_17": tuple(TRANSPORT_THETA_INDICES),
    "reverse_17_22_25": tuple(reversed(TRANSPORT_THETA_INDICES)),
}

# Cardinalidades que queremos usar como primeira família de generalização.
GENERALIZATION_K_VALUES = [3, 4, 5]

# Perturbações nos retornos em unidades do desvio-padrão transversal de mu.
# Não são resultados VQE; formam um manifesto de novos problemas a gerar.
RETURN_SHOCK_LEVELS = [-1.0, -0.5, 0.5, 1.0]
RISK_SCALE_LEVELS = [0.8, 1.0, 1.2]

print("Saídas da Parte X:", PATTERN_ROOT.resolve())
print("Cardinalidades planejadas:", GENERALIZATION_K_VALUES)


## Célula 52 — decompor as histórias sem perder suas fases

O `path-sum` anterior somava corretamente as amplitudes, mas depois da recombinação perdia a identidade de cada história microscópica. Agora cada ramificação dentro de $\theta_{25}$, $\theta_{22}$ e $\theta_{17}$ recebe um identificador próprio.

Para um estado final $x_f$ calculamos explicitamente

$$
A(x_f)=\sum_{\gamma} A_\gamma,
$$

$$
P_{\rm coh}=\left|\sum_\gamma A_\gamma\right|^2,
\qquad
P_{\rm inc}=\sum_\gamma |A_\gamma|^2,
$$

e

$$
I=P_{\rm coh}-P_{\rm inc}.
$$

- $P_{\rm coh}$ é a probabilidade quântica real;
- $P_{\rm inc}$ é o que obteríamos se destruíssemos a coerência entre as histórias;
- $I$ mede a contribuição líquida da interferência para aquele estado.

A comparação `correct` vs `reverse` usa **os mesmos valores de θ**. Assim, qualquer mudança vem apenas da ordem dos três blocos.


In [ ]:
# ============================================================
# 52. HISTÓRIAS PRIMITIVAS RESOLVIDAS EM AMPLITUDE E FASE
# ============================================================


def history_matrix_step(history_state, physical_qubits, local_u, record_tag=None):
    """Propaga amplitudes mantendo a identidade das histórias rastreadas."""
    next_state = defaultdict(complex)

    for (global_in, history), amp_in in history_state.items():
        local_in = sum(
            ((int(global_in) >> int(q)) & 1) << local_slot
            for local_slot, q in enumerate(physical_qubits)
        )
        for local_out, matrix_element in enumerate(local_u[:, local_in]):
            if abs(matrix_element) <= PATH_MATRIX_ELEMENT_TOL:
                continue
            global_out = replace_local_bits(global_in, physical_qubits, local_out)
            new_history = history
            if record_tag is not None:
                new_history = history + ((record_tag, int(global_in), int(global_out)),)
            next_state[(int(global_out), new_history)] += (
                complex(matrix_element) * complex(amp_in)
            )

    return {
        key: complex(value)
        for key, value in next_state.items()
        if abs(value) > PATH_AMPLITUDE_TOL
    }


def resolve_transport_histories(block_order, theta_values, tracked_indices=TRANSPORT_THETA_INDICES):
    """
    Abre em portas primitivas somente os blocos rastreados.
    Os demais blocos são propagados como operadores lógicos completos.
    """
    history_state = {(int(initial_index), tuple()): 1.0 + 0.0j}

    for block_step, theta_index in enumerate(block_order, start=1):
        theta_index = int(theta_index)
        if theta_index in set(map(int, tracked_indices)):
            for primitive_index, (gate_name, physical_qubits, matrix) in enumerate(
                primitive_sequence(theta_index, theta_values[theta_index]), start=1
            ):
                tag = f"b{block_step}:theta{theta_index}:p{primitive_index}:{gate_name}"
                history_state = history_matrix_step(
                    history_state, physical_qubits, matrix, record_tag=tag
                )
        else:
            physical_qubits, local_u = local_block_operator(
                theta_index, theta_values[theta_index]
            )
            history_state = history_matrix_step(
                history_state, physical_qubits, local_u, record_tag=None
            )

    # Soma as histórias apenas no final para reconstruir o Statevector.
    final_state = np.zeros(2 ** N_ASSETS, dtype=np.complex128)
    for (basis_index, _history), amplitude in history_state.items():
        final_state[int(basis_index)] += complex(amplitude)

    # Auditoria contra o circuito direto da mesma ordem e dos mesmos θ.
    circuit, parameters = build_circuit_from_block_order(block_order)
    bound = bind_structural_circuit(circuit, parameters, theta_values)
    direct_state = np.asarray(Statevector.from_instruction(bound).data, dtype=np.complex128)
    fidelity = float(abs(np.vdot(direct_state, final_state)) ** 2)
    if (1.0 - fidelity) > 1e-10:
        raise RuntimeError(
            "Histórias resolvidas não reconstruíram o circuito: "
            f"F={fidelity:.16f}"
        )
    return history_state, final_state, fidelity


def history_contributions_to_basis(history_state, target_index, order_label, target_label):
    rows = []
    for (basis_index, history), amplitude in history_state.items():
        if int(basis_index) != int(target_index):
            continue
        phase = float(np.angle(amplitude))
        rows.append({
            "order_label": str(order_label),
            "target_label": str(target_label),
            "target_bitstring": str(all_labels[int(target_index)]),
            "history_id": len(rows),
            "history_signature": " | ".join(item[0] for item in history),
            "amplitude_real": float(np.real(amplitude)),
            "amplitude_imag": float(np.imag(amplitude)),
            "amplitude_abs": float(abs(amplitude)),
            "phase_rad": phase,
            "phase_over_pi": float(phase / np.pi),
            "incoherent_probability_contribution": float(abs(amplitude) ** 2),
        })
    return pd.DataFrame(rows)


def summarize_history_contributions(frame, order_label, target_label, target_bitstring):
    if frame.empty:
        return {
            "order_label": order_label,
            "target_label": target_label,
            "target_bitstring": target_bitstring,
            "n_histories": 0,
            "coherent_probability": 0.0,
            "incoherent_probability": 0.0,
            "net_interference": 0.0,
            "coherence_gain": np.nan,
            "phase_resultant": np.nan,
        }

    amplitudes = (
        frame["amplitude_real"].to_numpy(dtype=float)
        + 1j * frame["amplitude_imag"].to_numpy(dtype=float)
    )
    p_coh = float(abs(amplitudes.sum()) ** 2)
    p_inc = float(np.sum(np.abs(amplitudes) ** 2))
    phase_resultant = float(
        abs(amplitudes.sum()) / max(np.sum(np.abs(amplitudes)), 1e-15)
    )
    return {
        "order_label": order_label,
        "target_label": target_label,
        "target_bitstring": target_bitstring,
        "n_histories": int(len(frame)),
        "coherent_probability": p_coh,
        "incoherent_probability": p_inc,
        "net_interference": float(p_coh - p_inc),
        "coherence_gain": float(p_coh / p_inc) if p_inc > 0 else np.nan,
        "phase_resultant": phase_resultant,
    }


In [ ]:
# ============================================================
# 53. ORDEM CORRETA vs ORDEM REVERSA — MESMOS θ, FASES EXPLÍCITAS
# ============================================================

history_detail_frames = []
history_summary_rows = []
order_fixed_theta_rows = []

for order_label, permutation in HISTORY_ORDER_COMPARISONS.items():
    block_order = order_with_transport_permutation(permutation)
    circuit, parameters = build_circuit_from_block_order(block_order)
    measured = measure_structural_circuit(
        circuit, parameters, ACTION_THETA, keep_probabilities=True
    )
    dominant_bit = str(measured["dominant_bitstring"])
    dominant_index = int(label_to_index[dominant_bit])

    history_state, resolved_state, resolved_fidelity = resolve_transport_histories(
        block_order, ACTION_THETA
    )

    order_fixed_theta_rows.append({
        "order_label": order_label,
        "order": "→".join(map(str, permutation)),
        "expected_energy_fixed_theta": measured["expected_energy"],
        "p_original_optimum_fixed_theta": measured["p_optimal"],
        "dominant_bitstring_fixed_theta": dominant_bit,
        "dominant_probability_fixed_theta": measured["dominant_probability"],
        "history_reconstruction_fidelity": resolved_fidelity,
    })

    targets = {
        "original_optimum": int(PATH_TARGET_INDEX),
        "order_dominant": dominant_index,
    }
    for target_label, target_index in targets.items():
        detail = history_contributions_to_basis(
            history_state, target_index, order_label, target_label
        )
        if not detail.empty:
            history_detail_frames.append(detail)
        history_summary_rows.append(
            summarize_history_contributions(
                detail,
                order_label,
                target_label,
                str(all_labels[int(target_index)]),
            )
        )

history_details_df = (
    pd.concat(history_detail_frames, ignore_index=True)
    if history_detail_frames else pd.DataFrame()
)
history_phase_summary_df = pd.DataFrame(history_summary_rows)
order_fixed_theta_df = pd.DataFrame(order_fixed_theta_rows)

display(order_fixed_theta_df)
display(history_phase_summary_df)
if not history_details_df.empty:
    display(history_details_df)

order_fixed_theta_df.to_csv(
    PATTERN_TABLE_DIR / "correct_vs_reverse_fixed_theta.csv", index=False
)
history_phase_summary_df.to_csv(
    PATTERN_TABLE_DIR / "history_phase_summary_correct_vs_reverse.csv", index=False
)
if not history_details_df.empty:
    history_details_df.to_csv(
        PATTERN_TABLE_DIR / "history_amplitudes_phases_correct_vs_reverse.csv", index=False
    )


## Célula 54 — como ler amplitudes e fases

O teste acima responde uma pergunta microscópica:

> **A ordem muda apenas o bitstring final ou muda a maneira como as histórias se somam?**

O diagnóstico principal é

$$
I=P_{\rm coh}-P_{\rm inc}.
$$

Além disso, `phase_resultant` varia de 0 a 1:

- próximo de 1: as amplitudes estão fortemente alinhadas em fase;
- próximo de 0: existe forte cancelamento angular entre elas.

Essas quantidades são candidatas a features transferíveis porque não dependem de o parâmetro se chamar `θ17` ou `θ25`.


In [ ]:
# ============================================================
# 54. FIGURAS — FASORES E GANHO DE COERÊNCIA
# ============================================================

if not history_details_df.empty:
    for (order_label, target_label), group in history_details_df.groupby(
        ["order_label", "target_label"]
    ):
        fig, ax = plt.subplots(figsize=(5.2, 5.2))
        for row in group.itertuples(index=False):
            ax.plot(
                [0.0, float(row.amplitude_real)],
                [0.0, float(row.amplitude_imag)],
                marker="o",
            )
        total = complex(
            group["amplitude_real"].sum(), group["amplitude_imag"].sum()
        )
        ax.plot([0.0, total.real], [0.0, total.imag], linewidth=3, marker="s")
        ax.axhline(0.0, linewidth=0.8)
        ax.axvline(0.0, linewidth=0.8)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("Re(Aγ)")
        ax.set_ylabel("Im(Aγ)")
        ax.set_title(f"Fasores — {order_label} — {target_label}")
        ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(
            PATTERN_FIGURE_DIR / f"phasors_{order_label}_{target_label}.png", dpi=180
        )
        plt.show()

if not history_phase_summary_df.empty:
    plot_df = history_phase_summary_df.loc[
        history_phase_summary_df["target_label"].eq("order_dominant")
    ].copy()
    fig, ax = plt.subplots(figsize=(7, 4.2))
    x = np.arange(len(plot_df))
    width = 0.36
    ax.bar(x - width/2, plot_df["coherent_probability"], width, label="coerente")
    ax.bar(x + width/2, plot_df["incoherent_probability"], width, label="incoerente")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["order_label"], rotation=15, ha="right")
    ax.set_ylabel("Probabilidade")
    ax.set_title("O que a coerência acrescenta em cada ordem?")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(PATTERN_FIGURE_DIR / "coherent_vs_incoherent_orders.png", dpi=180)
    plt.show()


## Célula 55 — o padrão para `k em n`

O padrão mais importante não é o índice do parâmetro; é a **conservação de cardinalidade**.

Defina o conjunto selecionado inicialmente como $S_0$ e o ótimo como $S_\star$. Então

$$
R=S_0\setminus S_\star,\qquad
A=S_\star\setminus S_0,
$$

com

$$
|R|=|A|=\frac{d_H(x_0,x_\star)}{2}.
$$

Cada mudança elementar válida é uma substituição

$$
q_a:1\rightarrow0,\qquad q_b:0\rightarrow1.
$$

Esse objeto continua existindo para `3 em 10`, `4 em 10`, `5 em 10`, etc. O que muda é:

1. o número de parâmetros do ansatz Dicke,
   $$N_\theta=\frac{k(2n-k-1)}{2};$$
2. quais ativos entram e saem;
3. o tamanho e a topologia do grafo de transporte;
4. o posto do QGT no ótimo;
5. o número e as fases das histórias coerentes.

Por isso, em problemas diferentes, devemos comparar **papéis estruturais**, não `theta_index`.


In [ ]:
# ============================================================
# 55. ASSINATURA INVARIANTE DA ROTA + FAMÍLIA k-EM-n
# ============================================================


def selected_qubits(bitstring_qiskit):
    return tuple(
        q for q in range(N_ASSETS)
        if str(bitstring_qiskit)[N_ASSETS - 1 - q] == "1"
    )


def transition_signature(bit_a, bit_b):
    removed, added = transition_qubits(str(bit_a), str(bit_b))
    hamming = int(sum(a != b for a, b in zip(str(bit_a), str(bit_b))))
    return {
        "removed_qubits": removed,
        "added_qubits": added,
        "hamming_distance": hamming,
        "n_replacements": int(hamming // 2),
        "cardinality_before": int(str(bit_a).count("1")),
        "cardinality_after": int(str(bit_b).count("1")),
        "cardinality_preserved": bool(str(bit_a).count("1") == str(bit_b).count("1")),
    }

route_pattern_rows = []
route_reset = route_df.reset_index(drop=True)
for idx in range(1, len(route_reset)):
    previous = route_reset.iloc[idx - 1]
    current = route_reset.iloc[idx]
    signature = transition_signature(
        previous["dominant_bitstring"], current["dominant_bitstring"]
    )
    route_pattern_rows.append({
        "route_step": idx,
        "theta_index_current_instance": int(current["theta_index"]),
        "from_bitstring": str(previous["dominant_bitstring"]),
        "to_bitstring": str(current["dominant_bitstring"]),
        **signature,
    })

route_pattern_df = pd.DataFrame(route_pattern_rows)
display(route_pattern_df)
route_pattern_df.to_csv(
    PATTERN_TABLE_DIR / "current_route_invariant_signature.csv", index=False
)

k_family_rows = []
for k_value in GENERALIZATION_K_VALUES:
    k_family_rows.append({
        "n_assets": int(N_ASSETS),
        "k": int(k_value),
        "k_over_n": float(k_value / N_ASSETS),
        "n_valid_bitstrings": int(math.comb(N_ASSETS, int(k_value))),
        "dicke_parameter_count": int(dicke_parameter_count(N_ASSETS, int(k_value))),
    })

k_family_df = pd.DataFrame(k_family_rows)
display(k_family_df)
k_family_df.to_csv(PATTERN_TABLE_DIR / "k_family_3_4_5_of_10.csv", index=False)


## Célula 56 — variar “o valor das ações” de forma compatível com o Hamiltoniano

No Hamiltoniano atual entram **retornos esperados** $\mu$ e **covariâncias** $\Sigma$; o preço nominal da ação não entra diretamente. Portanto, para estudar mudanças econômicas de maneira coerente com este modelo, o primeiro conjunto de perturbações deve atuar em $\mu$ e $\Sigma$.

Se no futuro houver uma restrição explícita de orçamento/preço, então preço também deve virar uma feature do ativo.

O manifesto abaixo cria problemas controlados de três tipos:

1. mudar $k$ entre 3, 4 e 5;
2. aplicar choques padronizados ao retorno de um ativo;
3. alterar globalmente a escala de risco.

A enumeração clássica para $n=10$ é barata e gera o **rótulo exato** do bitstring. O VQE e os diagnósticos QGT/path-sum podem então ser executados em uma amostra desses problemas para gerar os rótulos quânticos.

> O ótimo clássico é **target de treinamento**, nunca feature de entrada.


In [ ]:
# ============================================================
# 56. MANIFESTO DE PROBLEMAS: k=3/4/5 + CHOQUES DE RETORNO/RISCO
# ============================================================


def objective_for_instance(x_binary, mu_instance, sigma_instance):
    x = np.asarray(x_binary, dtype=float)
    return float(
        Q_VALUE * x @ sigma_instance @ x
        - (1.0 - Q_VALUE) * mu_instance @ x
        + RISK_FREE
    )


def exact_optimum_for_instance(k_value, mu_instance, sigma_instance):
    best_energy = np.inf
    best_bits = []
    for selected_indices in combinations(range(N_ASSETS), int(k_value)):
        x = np.zeros(N_ASSETS, dtype=int)
        x[list(selected_indices)] = 1
        energy = objective_for_instance(x, mu_instance, sigma_instance)
        bits_asset = "".join(map(str, x.tolist()))
        if energy < best_energy - ENERGY_ATOL:
            best_energy = float(energy)
            best_bits = [bits_asset]
        elif abs(energy - best_energy) <= ENERGY_ATOL:
            best_bits.append(bits_asset)
    return float(best_energy), tuple(sorted(best_bits))

mu_scale = float(np.std(mu))
if mu_scale <= 0:
    mu_scale = max(float(np.max(np.abs(mu))), 1.0)

scenario_rows = []
scenario_id = 0
for k_value in GENERALIZATION_K_VALUES:
    # Cenário-base de cada cardinalidade.
    e0, bits0 = exact_optimum_for_instance(k_value, mu, sigma)
    scenario_rows.append({
        "scenario_id": scenario_id,
        "scenario_type": "base",
        "n_assets": N_ASSETS,
        "k": int(k_value),
        "asset_index_shocked": -1,
        "return_shock_sigma_units": 0.0,
        "risk_scale": 1.0,
        "exact_energy": e0,
        "exact_bitstrings_asset_order": bits0,
    })
    scenario_id += 1

    # Choques individuais de retorno.
    for asset_index in range(N_ASSETS):
        for shock in RETURN_SHOCK_LEVELS:
            mu_trial = np.asarray(mu, dtype=float).copy()
            mu_trial[asset_index] += float(shock) * mu_scale
            e, bits = exact_optimum_for_instance(k_value, mu_trial, sigma)
            scenario_rows.append({
                "scenario_id": scenario_id,
                "scenario_type": "return_shock",
                "n_assets": N_ASSETS,
                "k": int(k_value),
                "asset_index_shocked": int(asset_index),
                "return_shock_sigma_units": float(shock),
                "risk_scale": 1.0,
                "exact_energy": e,
                "exact_bitstrings_asset_order": bits,
            })
            scenario_id += 1

    # Escala global de risco.
    for risk_scale in RISK_SCALE_LEVELS:
        if np.isclose(risk_scale, 1.0):
            continue
        sigma_trial = np.asarray(sigma, dtype=float) * float(risk_scale)
        e, bits = exact_optimum_for_instance(k_value, mu, sigma_trial)
        scenario_rows.append({
            "scenario_id": scenario_id,
            "scenario_type": "risk_scale",
            "n_assets": N_ASSETS,
            "k": int(k_value),
            "asset_index_shocked": -1,
            "return_shock_sigma_units": 0.0,
            "risk_scale": float(risk_scale),
            "exact_energy": e,
            "exact_bitstrings_asset_order": bits,
        })
        scenario_id += 1

generalization_manifest_df = pd.DataFrame(scenario_rows)
generalization_manifest_df["dicke_parameter_count"] = generalization_manifest_df["k"].map(
    lambda k_value: dicke_parameter_count(N_ASSETS, int(k_value))
)

display(generalization_manifest_df.head(20))
print("Número de problemas clássicos rotulados:", len(generalization_manifest_df))
generalization_manifest_df.to_csv(
    PATTERN_TABLE_DIR / "generalization_problem_manifest.csv", index=False
)


# Como isso entra no Transformer

## Célula 57 — o Transformer deve aprender relações, não índices de θ

Para generalizar entre `3 em 10`, `4 em 10` e `5 em 10`, o número de parâmetros muda:

$$
N_\theta(10,3)=24,\qquad
N_\theta(10,4)=30,\qquad
N_\theta(10,5)=35.
$$

Logo um vetor fixo `[θ0,...,θ29]` **não pode ser a representação universal**.

A representação proposta tem dois conjuntos de tokens.

### 1. Tokens dos ativos

Um token por ativo contém apenas informação disponível antes da solução:

$$
[\mu_i,\;\sigma_{ii},\;\text{conectividade de risco},\;x_i^{(0)},\;k/n,\ldots].
$$

A matriz de covariância pode entrar como *attention bias* ou como features de pares.

### 2. Tokens dos blocos do circuito

Um token por bloco parametrizado contém:

$$
[\text{gate type},\;i/n,\;l/n,\;(l-i)/n,\;\text{ativos tocados},\;
\theta_i^{\rm inicial},\;k/n,\ldots].
$$

O `theta_index` fica apenas como ID de auditoria; ele não é feature semântica.

### Targets principais

O mesmo encoder pode alimentar cabeças diferentes:

1. **selection head**: logits dos $n$ ativos, com seleção `top-k`;
2. **energy head**: energia prevista;
3. **active-block head**: probabilidade de cada bloco pertencer ao subespaço ativo;
4. **angle head**: $(\sin\phi_j,\cos\phi_j)$ do ângulo ótimo, respeitando a periodicidade;
5. **route/precedence head**: quais blocos transportam ocupação e em que ordem;
6. opcionalmente, **coherence head**: número de histórias, alinhamento de fase e ganho de coerência.

As duas primeiras são as saídas físicas principais. As demais funcionam como **supervisão auxiliar** para ensinar ao Transformer o mecanismo interno que os experimentos 20.16–20.18 estão revelando.


### Correção 20.18.1 — fonte dos qubits lógicos

Na versão 20.18, a exportação de tokens tentou ler `logical_block_qubits` de `structure_df`. Essa coluna é criada apenas em `parameter_map_df`. A célula abaixo usa agora a tabela enriquecida e deriva a ordem da rota diretamente de `route_pattern_df`, removendo também o hard-code `{25:0, 22:1, 17:2}`.


In [ ]:
# ============================================================
# 57. EXPORTAR TOKENS DA INSTÂNCIA ATUAL PARA O TRANSFORMER
#     BUGFIX 20.18.1: usar parameter_map_df para qubits lógicos
# ============================================================

# ---------- tokens dos ativos ----------
initial_selected_set = set(selected_qubits(initial_bitstring_qiskit))
optimal_selected_set = set(selected_qubits(REFERENCE_OPTIMAL_BITSTRING))

asset_token_rows = []
for asset_index in range(N_ASSETS):
    asset_token_rows.append({
        "instance_id": str(DATA_HASH),
        "asset_index": int(asset_index),
        "ticker_audit_only": str(tickers[asset_index]),
        "n_assets": int(N_ASSETS),
        "k": int(TARGET_K),
        "k_over_n": float(TARGET_K / N_ASSETS),
        "expected_return": float(mu[asset_index]),
        "variance": float(sigma[asset_index, asset_index]),
        "risk_connection_abs": float(np.sum(np.abs(sigma[asset_index]))),
        "initial_selected": int(asset_index in initial_selected_set),
        # TARGET — não deve ser usado como entrada.
        "target_optimal_selected": int(asset_index in optimal_selected_set),
    })

transformer_asset_tokens_df = pd.DataFrame(asset_token_rows)

# ---------- tokens dos blocos ----------
loo_lookup = {}
if not leave_one_out_df.empty:
    loo_lookup = {
        int(row.omitted_theta): float(1.0 - row.fidelity_to_full30_state)
        for row in leave_one_out_df.itertuples(index=False)
    }

position_lookup = {}
if RUN_FULL_POSITION_SCAN and not position_scan_df.empty:
    position_lookup = {
        int(j): float(1.0 - group["dominant_is_exact_optimum"].mean())
        for j, group in position_scan_df.groupby("moved_theta")
    }

# IMPORTANTE:
# structure_by_theta vem de structure_df e contém i, l e ansatz_gate_type,
# mas NÃO contém logical_block_qubits. Essa coluna é criada em parameter_map_df.
# Usamos, portanto, a tabela enriquecida como fonte única para os tokens.
parameter_map_by_theta = parameter_map_df.set_index("theta_index")
integrated_by_theta = integrated_evidence_df.set_index("theta_index")

required_block_columns = {
    "ansatz_gate_type", "i", "l", "logical_block_qubits", "angular_period"
}
missing_block_columns = sorted(required_block_columns - set(parameter_map_by_theta.columns))
if missing_block_columns:
    raise RuntimeError(
        "parameter_map_df não contém as colunas necessárias para os tokens: "
        f"{missing_block_columns}"
    )

# A ordem da rota NÃO é mais codificada manualmente como {25:0,22:1,17:2}.
# Ela é extraída da rota observada na Célula 55, o que mantém a interface
# compatível com futuras instâncias em que outros theta/blocos sejam usados.
route_step_lookup = {}
if not route_pattern_df.empty:
    for row in route_pattern_df.itertuples(index=False):
        route_step_lookup[int(row.theta_index_current_instance)] = int(row.route_step) - 1

observed_route_theta_indices = tuple(
    theta for theta, _step in sorted(route_step_lookup.items(), key=lambda item: item[1])
)

print("Rota observada usada como label auxiliar:", observed_route_theta_indices)

block_token_rows = []
for block_position_index, theta_index in enumerate(ORIGINAL_BLOCK_ORDER):
    theta_index = int(theta_index)
    structural = parameter_map_by_theta.loc[theta_index]
    qubits = tuple(map(int, structural["logical_block_qubits"]))
    period = float(structural["angular_period"])

    initial_phase = 2.0 * np.pi * float(theta_path_start[theta_index]) / period
    target_phase = 2.0 * np.pi * float(ACTION_THETA[theta_index]) / period
    local_mu = np.asarray([mu[q] for q in qubits], dtype=float)
    local_var = np.asarray([sigma[q, q] for q in qubits], dtype=float)
    local_cov = np.asarray(sigma[np.ix_(qubits, qubits)], dtype=float)
    evidence = integrated_by_theta.loc[theta_index]

    block_token_rows.append({
        "instance_id": str(DATA_HASH),
        # Apenas auditoria. Não usar como feature física generalizável.
        "theta_index_audit_only": theta_index,
        "block_position": int(block_position_index),
        "block_position_norm": float(block_position_index / max(N_PARAMETERS - 1, 1)),
        "gate_type": str(structural["ansatz_gate_type"]),
        "i_norm": float(int(structural["i"]) / max(N_ASSETS - 1, 1)),
        "l_norm": float(int(structural["l"]) / max(N_ASSETS - 1, 1)),
        "span_norm": float(abs(int(structural["l"]) - int(structural["i"])) / max(N_ASSETS - 1, 1)),
        "n_qubits_touched": int(len(qubits)),
        "qubits_audit_only": qubits,
        "n_assets": int(N_ASSETS),
        "k": int(TARGET_K),
        "k_over_n": float(TARGET_K / N_ASSETS),
        "local_return_mean": float(local_mu.mean()),
        "local_return_std": float(local_mu.std()),
        "local_variance_mean": float(local_var.mean()),
        "local_cov_abs_mean": float(np.mean(np.abs(local_cov))),
        "initial_theta_sin": float(np.sin(initial_phase)),
        "initial_theta_cos": float(np.cos(initial_phase)),
        # -------- targets auxiliares: NUNCA entram como input --------
        "target_is_active": int(theta_index in ACTIVE_THETA_INDICES),
        "target_is_transport": int(theta_index in route_step_lookup),
        "target_route_step": int(route_step_lookup.get(theta_index, -1)),
        "target_theta_sin": float(np.sin(target_phase)),
        "target_theta_cos": float(np.cos(target_phase)),
        "target_qgt_diag": float(evidence["qgt_diag"]),
        "target_energy_curvature": float(evidence["energy_curvature_fd"]),
        "target_p_curvature": float(evidence["p_optimal_curvature_fd"]),
        "target_leave_one_out_loss": float(loo_lookup.get(theta_index, 0.0)),
        "target_position_failure_fraction": float(position_lookup.get(theta_index, 0.0)),
    })

transformer_block_tokens_df = pd.DataFrame(block_token_rows)

# ---------- labels de precedência entre blocos da rota observada ----------
precedence_rows = []
for a, b in permutations(observed_route_theta_indices, 2):
    precedence_rows.append({
        "instance_id": str(DATA_HASH),
        "theta_a_audit_only": int(a),
        "theta_b_audit_only": int(b),
        "a_precedes_b_in_observed_route": int(
            route_step_lookup[int(a)] < route_step_lookup[int(b)]
        ),
    })
transformer_precedence_df = pd.DataFrame(precedence_rows)

# Auditoria mínima: uma linha de bloco para cada parâmetro do ansatz.
if len(transformer_block_tokens_df) != N_PARAMETERS:
    raise RuntimeError(
        f"Esperados {N_PARAMETERS} tokens de bloco; encontrados "
        f"{len(transformer_block_tokens_df)}."
    )

# A coluna de qubits deve estar preenchida para todos os blocos.
if transformer_block_tokens_df["n_qubits_touched"].isna().any():
    raise RuntimeError("Há tokens de bloco sem informação de qubits.")

display(transformer_asset_tokens_df)
display(transformer_block_tokens_df)
display(transformer_precedence_df)

transformer_asset_tokens_df.to_csv(
    PATTERN_TABLE_DIR / "transformer_asset_tokens_current_instance.csv", index=False
)
transformer_block_tokens_df.to_csv(
    PATTERN_TABLE_DIR / "transformer_block_tokens_current_instance.csv", index=False
)
transformer_precedence_df.to_csv(
    PATTERN_TABLE_DIR / "transformer_precedence_labels_current_instance.csv", index=False
)


## Célula 58 — arquitetura recomendada para o Transformer

Uma arquitetura mínima e coerente com os resultados seria:

$$
\text{Asset Encoder}\quad+\quad\text{Block Encoder}\quad+\quad\text{Cross-Attention}.
$$

O token global carrega

$$
[n,\;k,\;k/n,\;q,\;r_f].
$$

Os tokens dos ativos representam o Hamiltoniano; os tokens dos blocos representam a geometria do ansatz. A *cross-attention* permite ao modelo aprender relações do tipo:

> “este bloco toca justamente os ativos cuja ocupação precisa ser deslocada dadas estas relações de retorno/risco”.

### Função de perda multitarefa

Uma forma inicial é

$$
\mathcal L=
\lambda_s\mathcal L_{selection}
+\lambda_E\mathcal L_E
+\lambda_a\mathcal L_{active}
+\lambda_\theta\mathcal L_{angle}
+\lambda_r\mathcal L_{route}.
$$

Para os ângulos, em vez de MSE diretamente em $\theta$, usa-se

$$
\mathcal L_{angle}
=\left\|\hat{\mathbf u}_j-\mathbf u_j\right\|^2,
\qquad
\mathbf u_j=(\cos\phi_j,\sin\phi_j),
$$

porque $0$ e $2\pi$ representam o mesmo ponto para uma porta de período $2\pi$.

### Como generalizar entre k diferentes

O número de tokens de blocos é variável (`24`, `30`, `35`, ...), portanto usamos **máscara de atenção**, não padding tratado como dado físico. A cabeça de seleção sempre produz $n$ logits e aplica uma regra `top-k` ou outra projeção de cardinalidade.

### O que queremos que o Transformer aprenda

Não é memorizar o ótimo. É aproximar a aplicação

$$
(\mu,\Sigma,k,\text{estrutura do circuito})
\mapsto
(\text{seleção},E,\text{subespaço ativo},\text{rota},\theta^\star).
$$

O resultado `30→7→3` passa a ser **supervisão estrutural**: ele diz ao modelo que, para esta instância, a solução vive numa parte esparsa da arquitetura. Em outras instâncias, o modelo deve descobrir se o padrão é `24→r→m`, `35→r→m`, etc., sem assumir antecipadamente que $r=7$ ou $m=3$.


In [ ]:
# ============================================================
# 58. MANIFESTO FINAL PARA CONSTRUÇÃO DO DATASET DO TRANSFORMER
# ============================================================

transformer_dataset_manifest = {
    "version": NOTEBOOK_VERSION,
    "current_instance_id": str(DATA_HASH),
    "current_n_assets": int(N_ASSETS),
    "current_k": int(TARGET_K),
    "current_n_parameters": int(N_PARAMETERS),
    "current_active_dimension_qgt": int(qgt_summary_df.iloc[0]["qgt_numeric_rank"]),
    "current_active_theta_count": int(len(ACTIVE_THETA_INDICES)),
    "current_transport_block_count": int(len(TRANSPORT_THETA_INDICES)),
    "generalization_k_values": list(map(int, GENERALIZATION_K_VALUES)),
    "generalization_problem_count": int(len(generalization_manifest_df)),
    "input_tables": {
        "asset_tokens": "transformer_asset_tokens_current_instance.csv",
        "block_tokens": "transformer_block_tokens_current_instance.csv",
    },
    "target_tables": {
        "precedence": "transformer_precedence_labels_current_instance.csv",
        "history_phases": "history_amplitudes_phases_correct_vs_reverse.csv",
        "problem_manifest": "generalization_problem_manifest.csv",
    },
    "leakage_rule": "optimal bitstrings and diagnostic targets are labels only; never model inputs",
}

with open(PATTERN_TABLE_DIR / "transformer_dataset_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(transformer_dataset_manifest, handle, indent=2, ensure_ascii=False)

display(pd.DataFrame([transformer_dataset_manifest]))


# Checklist científico da Parte X

Antes de transformar o padrão em afirmação geral:

1. **Não universalizar os números 7 e 3.** Eles são resultados da instância atual.
2. Para cada novo $k$, reconstruir o ansatz correspondente; o número de $\theta$ muda.
3. Repetir pelo menos: otimização, probe estrutural, QGT, rota de bitstrings e compressão.
4. Em perturbações financeiras, mudar primeiro as quantidades que entram no Hamiltoniano ($\mu,\Sigma$).
5. Usar o ótimo exato apenas como **label** e auditoria, nunca como entrada do Transformer.
6. Comparar padrões invariantes: $k/n$, $d_H/2$, posto QGT, fração ativa, topologia da rota, multiplicidade de histórias e coerência.
7. Separar **expressividade** de **dificuldade de otimização** nas permutações usando warm starts/múltiplos reinícios quando necessário.
8. Só chamar uma relação de “padrão geral” depois de observar a mesma lei em várias cardinalidades e várias perturbações do Hamiltoniano.

## Hipótese de trabalho para a próxima campanha

A hipótese não é que sempre existam sete parâmetros. É:

> **Problemas de cardinalidade fixa podem induzir soluções variacionais esparsas cujo mecanismo é descrito por um pequeno subespaço geométrico e por um grafo de transporte de ocupação; a dimensão desse subespaço e a rota dependem de $(H,k)$ e podem ser previstas a partir de tokens estruturais por um Transformer.**

Essa hipótese é falsificável: em novas instâncias, o QGT pode não contrair, a rota pode precisar de muitos blocos ou as histórias podem não mostrar a mesma organização de fase. Esses casos também devem entrar no treinamento.


# Parte XI — a ordem está dirigindo a interferência?

Os resultados da 20.18 sugerem algo mais específico que “a ordem importa”.

Na execução observada, a ordem original `25→22→17` concentrou oito histórias no ótimo com ganho coerente máximo, enquanto a ordem reversa praticamente cancelou a amplitude do ótimo e direcionou probabilidade para outro bitstring.

Há três perguntas diferentes que precisam ser separadas:

1. **ordem com ângulos fixos:** a mesma parametrização produz outro estado apenas por trocar a precedência?
2. **reachability:** uma ordem alterada ainda consegue representar o estado ótimo se os ângulos forem escolhidos deliberadamente para isso?
3. **otimização física:** a minimização de energia consegue encontrar esse estado nessa arquitetura?

Essa separação evita confundir **causalidade estrutural**, **expressividade** e **paisagem de otimização**.

Também fazemos uma auditoria importante: como o circuito atual é construído com matrizes reais (`RY`, `CX`, `CCX`), as “fases” observadas podem se reduzir a sinais `0` e `π`. O notebook mede isso explicitamente antes de usar linguagem de interferência de fase.


In [ ]:
# ============================================================
# 59. CONFIGURAÇÃO DOS CONTROLES 20.19
# ============================================================

VALIDATION19_ROOT = OUTPUT_ROOT / "validation_v20_19"
VALIDATION19_TABLE_DIR = VALIDATION19_ROOT / "tables"
VALIDATION19_FIGURE_DIR = VALIDATION19_ROOT / "figures"
for directory in [VALIDATION19_ROOT, VALIDATION19_TABLE_DIR, VALIDATION19_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reachability é um teste diagnóstico: ele PODE usar o alvo conhecido,
# mas somente para perguntar se determinada ordem consegue representá-lo.
RUN_TARGET_REACHABILITY = True
REACHABILITY_MAXITER = 500
REACHABILITY_RESTARTS = 4

# Faixa usada apenas no scan clássico de escala global de risco.
RISK_SCAN_LEVELS_20_19 = np.linspace(0.50, 1.50, 51)

print("Saídas da Parte XI:", VALIDATION19_ROOT.resolve())
print("Reachability diagnóstico:", RUN_TARGET_REACHABILITY)


## Célula 60 — todas as seis ordens, com os mesmos θ, e histórias resolvidas

A comparação `correct` versus `reverse` era o contraste máximo. Agora fazemos as seis permutações sem reotimizar os ângulos.

Para cada ordem medimos:

\[
P(x_\star),\qquad E,\qquad x_{\rm dom},
\]

e para o ótimo original e para o dominante daquela ordem calculamos

\[
P_{\rm coh},\quad P_{\rm inc},\quad
R_\phi=\frac{|\sum_\gamma A_\gamma|}{\sum_\gamma |A_\gamma|}.
\]

Além disso, definimos

\[
N_{\rm eff}=
\frac{(\sum_\gamma |A_\gamma|)^2}
{\sum_\gamma |A_\gamma|^2},
\]

que mede quantas histórias têm peso efetivo, e

\[
B_I=\frac{P_{\rm coh}-P_{\rm inc}}
{P_{\rm coh}+P_{\rm inc}}\in[-1,1],
\]

que dá uma medida limitada de interferência construtiva/destrutiva.


In [ ]:
# ============================================================
# 60. SEIS ORDENS FIXAS + HISTÓRIAS + MÉTRICAS NORMALIZADAS
# ============================================================

all_perm_fixed_rows = []
all_perm_history_rows = []
all_perm_history_detail_frames = []
fixed_theta_probabilities = {}

for permutation in permutations(TRANSPORT_THETA_INDICES):
    permutation = tuple(map(int, permutation))
    order_label = "→".join(map(str, permutation))
    block_order = order_with_transport_permutation(permutation)

    circuit, parameters = build_circuit_from_block_order(block_order)
    measured = measure_structural_circuit(
        circuit, parameters, ACTION_THETA, keep_probabilities=True
    )
    fixed_theta_probabilities[order_label] = measured["probabilities"]

    history_state, _, fidelity = resolve_transport_histories(
        block_order, ACTION_THETA
    )
    dominant_index = int(label_to_index[str(measured["dominant_bitstring"])])

    all_perm_fixed_rows.append({
        "order": order_label,
        "is_original_order": bool(list(permutation) == transport_original_order),
        "expected_energy_fixed_theta": float(measured["expected_energy"]),
        "p_optimal_fixed_theta": float(measured["p_optimal"]),
        "dominant_bitstring": str(measured["dominant_bitstring"]),
        "dominant_probability": float(measured["dominant_probability"]),
        "history_reconstruction_fidelity": float(fidelity),
    })

    for target_label, target_index in {
        "original_optimum": int(PATH_TARGET_INDEX),
        "order_dominant": dominant_index,
    }.items():
        detail = history_contributions_to_basis(
            history_state, target_index, order_label, target_label
        )
        if not detail.empty:
            all_perm_history_detail_frames.append(detail)

        summary = summarize_history_contributions(
            detail, order_label, target_label, str(all_labels[target_index])
        )
        if not detail.empty:
            abs_amp = detail["amplitude_abs"].to_numpy(dtype=float)
            p_inc = float(summary["incoherent_probability"])
            sum_abs = float(abs_amp.sum())
            n_eff = float(sum_abs**2 / p_inc) if p_inc > 0 else np.nan
            direct_p = float(measured["probabilities"][target_index])
        else:
            sum_abs, n_eff, direct_p = 0.0, np.nan, 0.0

        denom = summary["coherent_probability"] + summary["incoherent_probability"]
        summary.update({
            "sum_abs_amplitudes": sum_abs,
            "effective_history_number": n_eff,
            "alignment_efficiency": float(summary["phase_resultant"]**2)
                if np.isfinite(summary["phase_resultant"]) else np.nan,
            "interference_balance": float(summary["net_interference"] / denom)
                if denom > 0 else np.nan,
            "direct_probability": direct_p,
        })
        all_perm_history_rows.append(summary)

all_perm_fixed_df = pd.DataFrame(all_perm_fixed_rows)
all_perm_history_summary_df = pd.DataFrame(all_perm_history_rows)
all_perm_history_details_df = (
    pd.concat(all_perm_history_detail_frames, ignore_index=True)
    if all_perm_history_detail_frames else pd.DataFrame()
)

display(all_perm_fixed_df.sort_values("p_optimal_fixed_theta", ascending=False))
display(all_perm_history_summary_df)

all_perm_fixed_df.to_csv(
    VALIDATION19_TABLE_DIR / "all_six_orders_fixed_theta.csv", index=False
)
all_perm_history_summary_df.to_csv(
    VALIDATION19_TABLE_DIR / "all_six_orders_history_summary.csv", index=False
)
if not all_perm_history_details_df.empty:
    all_perm_history_details_df.to_csv(
        VALIDATION19_TABLE_DIR / "all_six_orders_history_details.csv", index=False
    )


## Célula 61 — a ordem roteia probabilidade para quais bitstrings?

Esta figura não olha apenas para \(P(x_\star)\). Ela inclui o estado inicial, o ótimo original e todos os bitstrings que se tornaram dominantes em alguma das seis ordens.

Se diferentes ordens concentrarem probabilidade em diferentes estados com os mesmos ângulos, temos uma assinatura direta de **roteamento estrutural de amplitude**.


In [ ]:
# ============================================================
# 61. HEATMAP ORDEM × BITSTRING + COERÊNCIA NAS SEIS ORDENS
# ============================================================

candidate_bits = [str(initial_bitstring_qiskit), str(REFERENCE_OPTIMAL_BITSTRING)]
candidate_bits += all_perm_fixed_df["dominant_bitstring"].astype(str).tolist()
candidate_bits = list(dict.fromkeys(candidate_bits))

matrix = np.asarray([
    [fixed_theta_probabilities[order][label_to_index[bit]] for bit in candidate_bits]
    for order in all_perm_fixed_df["order"]
], dtype=float)

fig, ax = plt.subplots(figsize=(max(7, 1.2 * len(candidate_bits)), 5.0))
im = ax.imshow(matrix, aspect="auto", vmin=0.0, vmax=1.0)
ax.set_xticks(np.arange(len(candidate_bits)))
ax.set_xticklabels(candidate_bits, rotation=45, ha="right")
ax.set_yticks(np.arange(len(all_perm_fixed_df)))
ax.set_yticklabels(all_perm_fixed_df["order"])
ax.set_xlabel("Bitstring")
ax.set_ylabel("Ordem dos transportadores")
ax.set_title("Mesmos θ: para qual estado cada ordem roteia a probabilidade?")
fig.colorbar(im, ax=ax, label="Probabilidade")
fig.tight_layout()
fig.savefig(VALIDATION19_FIGURE_DIR / "order_to_bitstring_probability.png", dpi=180)
plt.show()

plot_df = all_perm_history_summary_df.query(
    "target_label == 'original_optimum'"
).copy()
fig, ax = plt.subplots(figsize=(8.0, 4.4))
x = np.arange(len(plot_df))
ax.bar(x, plot_df["phase_resultant"])
ax.set_xticks(x)
ax.set_xticklabels(plot_df["order"], rotation=30, ha="right")
ax.set_ylim(0.0, 1.05)
ax.set_ylabel(r"$R_\phi$")
ax.set_title("Alinhamento das histórias que chegam ao ótimo original")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(VALIDATION19_FIGURE_DIR / "order_phase_resultant_original_target.png", dpi=180)
plt.show()


## Célula 62 — auditoria: fase complexa ou apenas sinal?

Um `path-sum` pode existir mesmo quando todas as amplitudes são reais. Nesse caso a interferência ocorre principalmente por **sinais relativos** \(0\) e \(\pi\), e não por uma distribuição contínua de fases complexas.

O teste abaixo mede a parte imaginária e as classes de fase efetivamente presentes. Isso impede interpretar um gráfico de fasores no eixo real como se já tivéssemos uma dinâmica de fases complexas geral.

Também usamos `math.fsum` para a soma real e imaginária quando há cancelamento extremo. Probabilidades da ordem de \(10^{-30}\) ou menores após cancelamento devem ser tratadas como **resíduo numérico**, não como uma diferença física resolvida.


In [ ]:
# ============================================================
# 62. AUDITORIA DE FASES, SINAIS E CANCELAMENTO NUMÉRICO
# ============================================================

phase_audit_rows = []

for (order_label, target_label), group in all_perm_history_details_df.groupby(
    ["order_label", "target_label"]
):
    real = group["amplitude_real"].to_numpy(dtype=float)
    imag = group["amplitude_imag"].to_numpy(dtype=float)
    abs_amp = group["amplitude_abs"].to_numpy(dtype=float)

    # fsum reduz erro de arredondamento justamente quando amplitudes se cancelam.
    total_fsum = complex(math.fsum(real.tolist()), math.fsum(imag.tolist()))
    p_fsum = float(abs(total_fsum) ** 2)
    sum_abs = float(abs_amp.sum())
    resultant = float(abs(total_fsum) / sum_abs) if sum_abs > 0 else np.nan

    phase_pi = np.mod(np.arctan2(imag, real), 2*np.pi) / np.pi
    near_zero_or_pi = np.isclose(phase_pi, 0.0, atol=1e-10) | np.isclose(
        phase_pi, 1.0, atol=1e-10
    ) | np.isclose(phase_pi, 2.0, atol=1e-10)

    direct_p = float(
        all_perm_history_summary_df.loc[
            (all_perm_history_summary_df["order_label"] == order_label)
            & (all_perm_history_summary_df["target_label"] == target_label),
            "direct_probability",
        ].iloc[0]
    )

    phase_audit_rows.append({
        "order": order_label,
        "target_label": target_label,
        "n_histories": int(len(group)),
        "max_abs_imaginary": float(np.max(np.abs(imag))) if len(imag) else 0.0,
        "imaginary_l1_fraction": float(np.sum(np.abs(imag)) / max(sum_abs, 1e-300)),
        "fraction_phase_0_or_pi": float(np.mean(near_zero_or_pi)),
        "phase_resultant_fsum": resultant,
        "coherent_probability_fsum": p_fsum,
        "direct_probability": direct_p,
        "abs_probability_difference": float(abs(p_fsum - direct_p)),
        "cancellation_digits": float(-np.log10(max(resultant, np.finfo(float).eps))),
    })

phase_audit_df = pd.DataFrame(phase_audit_rows)
display(phase_audit_df)

phase_audit_df.to_csv(
    VALIDATION19_TABLE_DIR / "phase_sign_numerical_audit.csv", index=False
)

if (phase_audit_df["fraction_phase_0_or_pi"] > 1.0 - 1e-12).all():
    print("Diagnóstico: nesta instância, as histórias usam apenas fases 0/π (sinais reais).")
else:
    print("Diagnóstico: há fases complexas não triviais além de 0/π.")


## Célula 63 — reachability do alvo: expressividade versus otimização

Este é um **controle diagnóstico**, não o algoritmo de solução.

Para cada uma das seis ordens, minimizamos artificialmente

\[
1-P(x_\star)
\]

usando o alvo conhecido. O objetivo é apenas responder:

> “essa arquitetura reordenada consegue representar o estado ótimo em algum ponto do seu espaço de parâmetros?”

Interpretação:

- `reachability ≈ 1` e a otimização por energia falha → o estado é representável, mas a paisagem/otimizador não o recuperou;
- `reachability` persistentemente baixa em muitos reinícios → evidência de restrição estrutural da ordem, embora não seja uma prova matemática de impossibilidade.

O resultado desse teste **não pode entrar como feature do Transformer**.


In [ ]:
# ============================================================
# 63. REACHABILITY DO BITSTRING ÓTIMO PARA CADA ORDEM
# ============================================================

def target_probability(circuit, parameters, theta_values, target_index):
    bound = bind_structural_circuit(circuit, parameters, theta_values)
    probabilities = np.asarray(
        Statevector.from_instruction(bound).probabilities(), dtype=float
    )
    return float(probabilities[int(target_index)])


reachability_rows = []
if RUN_TARGET_REACHABILITY:
    periods = parameter_map_df.set_index("theta_index")["angular_period"].to_dict()

    for permutation_id, permutation in enumerate(permutations(TRANSPORT_THETA_INDICES)):
        permutation = tuple(map(int, permutation))
        order_label = "→".join(map(str, permutation))
        block_order = order_with_transport_permutation(permutation)
        circuit, parameters = build_circuit_from_block_order(block_order)

        # A ordem original já possui P(alvo)=1 no ponto conhecido.
        starts = [np.asarray(ACTION_THETA, dtype=float).copy()]
        rng_local = np.random.default_rng(RANDOM_SEED + 31000 + permutation_id)
        for _ in range(REACHABILITY_RESTARTS - 1):
            trial = np.asarray(ACTION_THETA, dtype=float).copy()
            trial += np.asarray([
                rng_local.uniform(-0.25, 0.25) * periods[int(j)]
                for j in range(N_PARAMETERS)
            ])
            starts.append(trial)

        best_p = -np.inf
        best_theta = starts[0].copy()
        total_nfev = 0

        for start in starts:
            result = minimize(
                lambda values: 1.0 - target_probability(
                    circuit, parameters, values, PATH_TARGET_INDEX
                ),
                x0=start,
                method="COBYLA",
                options={
                    "maxiter": int(REACHABILITY_MAXITER),
                    "rhobeg": 0.5,
                    "catol": 1e-10,
                },
            )
            total_nfev += int(getattr(result, "nfev", 0))
            p_candidate = target_probability(
                circuit, parameters, result.x, PATH_TARGET_INDEX
            )
            if p_candidate > best_p:
                best_p = float(p_candidate)
                best_theta = np.asarray(result.x, dtype=float).copy()

        measured_best = measure_structural_circuit(
            circuit, parameters, best_theta, keep_probabilities=False
        )
        reachability_rows.append({
            "order": order_label,
            "target_reachability": best_p,
            "energy_at_reachability_point": float(measured_best["expected_energy"]),
            "dominant_bitstring_at_reachability": str(measured_best["dominant_bitstring"]),
            "nfev_total": int(total_nfev),
        })

reachability_df = pd.DataFrame(reachability_rows)

if not reachability_df.empty:
    comparison_cols = [
        "order", "p_optimal", "expected_energy", "dominant_bitstring"
    ]
    reachability_comparison_df = reachability_df.merge(
        transport_permutations_df[comparison_cols],
        on="order",
        how="left",
        suffixes=("_reachability", "_energy_optimization"),
    )
    display(reachability_comparison_df)
    reachability_comparison_df.to_csv(
        VALIDATION19_TABLE_DIR / "reachability_vs_energy_optimization.csv", index=False
    )

    fig, ax = plt.subplots(figsize=(8.0, 4.5))
    x = np.arange(len(reachability_comparison_df))
    width = 0.36
    ax.bar(
        x - width/2,
        reachability_comparison_df["target_reachability"],
        width,
        label="reachability do alvo",
    )
    ax.bar(
        x + width/2,
        reachability_comparison_df["p_optimal"],
        width,
        label="após otimização de energia",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(reachability_comparison_df["order"], rotation=30, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Probabilidade")
    ax.set_title("Expressividade diagnóstica vs recuperação pela energia")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        VALIDATION19_FIGURE_DIR / "reachability_vs_energy_optimization.png", dpi=180
    )
    plt.show()


## Célula 64 — a rota tem um “desvio” combinatório?

A rota observada foi

\[
1111000000\rightarrow1011100000\rightarrow1001110000\rightarrow1001011000.
\]

Cada passo troca uma ocupação e preserva \(k\). Entretanto, o número de passos da rota pode ser maior que a distância mínima entre o início e o fim.

Definimos, sem chamar isso de ação física,

\[
L_{\rm route}=\sum_t \frac{d_H(x_t,x_{t+1})}{2},
\qquad
L_{\min}=\frac{d_H(x_0,x_\star)}{2},
\]

e o **alongamento combinatório**

\[
\chi_{\rm route}=\frac{L_{\rm route}}{L_{\min}}\ge1.
\]

Se \(\chi_{\rm route}>1\), existe pelo menos um **relay**: uma ocupação entra em um qubit intermediário e depois sai novamente. Esse objeto é generalizável entre valores diferentes de \(k\).


In [ ]:
# ============================================================
# 64. GRAFO DE TRANSPORTE, RELAYS E ALONGAMENTO COMBINATÓRIO
# ============================================================

transport_edge_rows = []
for row in route_pattern_df.itertuples(index=False):
    removed = tuple(map(int, row.removed_qubits))
    added = tuple(map(int, row.added_qubits))

    # Na rota atual cada passo é Hamming-2 e a correspondência é inequívoca.
    if len(removed) == 1 and len(added) == 1:
        transport_edge_rows.append({
            "route_step": int(row.route_step),
            "theta_index_audit_only": int(row.theta_index_current_instance),
            "from_qubit": int(removed[0]),
            "to_qubit": int(added[0]),
        })

transport_graph_df = pd.DataFrame(transport_edge_rows)

first_bit = str(route_pattern_df.iloc[0]["from_bitstring"])
last_bit = str(route_pattern_df.iloc[-1]["to_bitstring"])
net_signature = transition_signature(first_bit, last_bit)

route_length = int(route_pattern_df["n_replacements"].sum())
minimum_length = int(net_signature["n_replacements"])
route_stretch = float(route_length / minimum_length) if minimum_length > 0 else 1.0

removed_nodes = set(transport_graph_df["from_qubit"].astype(int))
added_nodes = set(transport_graph_df["to_qubit"].astype(int))
relay_nodes = tuple(sorted(removed_nodes & added_nodes))
source_nodes = tuple(sorted(removed_nodes - added_nodes))
sink_nodes = tuple(sorted(added_nodes - removed_nodes))

route_topology_df = pd.DataFrame([{
    "route_replacement_length": route_length,
    "minimum_replacement_length": minimum_length,
    "route_excess": int(route_length - minimum_length),
    "route_stretch": route_stretch,
    "n_sources": len(source_nodes),
    "n_sinks": len(sink_nodes),
    "n_relays": len(relay_nodes),
    "sources": source_nodes,
    "sinks": sink_nodes,
    "relays": relay_nodes,
}])

display(transport_graph_df)
display(route_topology_df)

transport_graph_df.to_csv(
    VALIDATION19_TABLE_DIR / "transport_graph_edges.csv", index=False
)
route_topology_df.to_csv(
    VALIDATION19_TABLE_DIR / "transport_graph_topology.csv", index=False
)

fig, ax = plt.subplots(figsize=(8.0, 3.0))
nodes = sorted(set(transport_graph_df["from_qubit"]) | set(transport_graph_df["to_qubit"]))
for q in nodes:
    ax.scatter(q, 0.0, s=120)
    ax.text(q, -0.08, f"q{q}", ha="center", va="top")
for edge in transport_graph_df.itertuples(index=False):
    height = 0.25 + 0.10 * int(edge.route_step)
    ax.annotate(
        "",
        xy=(edge.to_qubit, 0.02),
        xytext=(edge.from_qubit, 0.02),
        arrowprops={"arrowstyle": "->", "connectionstyle": f"arc3,rad={0.18*np.sign(edge.to_qubit-edge.from_qubit)}"},
    )
    ax.text(
        (edge.from_qubit + edge.to_qubit) / 2,
        height,
        f"passo {edge.route_step}",
        ha="center",
    )
ax.set_ylim(-0.18, 0.75)
ax.set_yticks([])
ax.set_xlabel("Qubit / ativo")
ax.set_title(f"Grafo efetivo de transporte — alongamento χ={route_stretch:.3f}")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(VALIDATION19_FIGURE_DIR / "effective_transport_graph.png", dpi=180)
plt.show()


# Parte XII — os 129 problemas clássicos já revelam fronteiras de decisão

A tabela da Célula 56 mostrou um comportamento em **platôs**: vários choques não mudam a seleção ótima e, depois de certo limiar, o bitstring muda discretamente.

Para um choque apenas no retorno do ativo \(i\),

\[
\mu_i\rightarrow\mu_i+s\,\sigma_\mu,
\]

todo portfólio que contém \(i\) recebe o mesmo deslocamento energético

\[
-(1-q)s\,\sigma_\mu.
\]

Logo, para cada par \((k,i)\), basta comparar:

- o melhor portfólio que **contém** \(i\);
- o melhor portfólio que **não contém** \(i\).

Isso produz um limiar analítico exato \(s_c\) para a troca de regime:

\[
s_c=
\frac{E_{\rm in}-E_{\rm out}}
{(1-q)\sigma_\mu}.
\]

Esse limiar é extremamente útil para escolher casos de treinamento próximos e distantes das fronteiras. Ele é um **label/auditoria obtido por enumeração**, não uma feature disponível ao Transformer.


In [ ]:
# ============================================================
# 65. LIMIAR ANALÍTICO EXATO PARA CHOQUES DE RETORNO
# ============================================================

def best_portfolios_with_asset_condition(k_value, asset_index, must_include):
    best_energy = np.inf
    best_bits = []

    for selected in combinations(range(N_ASSETS), int(k_value)):
        contains = int(asset_index) in selected
        if contains != bool(must_include):
            continue

        x = np.zeros(N_ASSETS, dtype=int)
        x[list(selected)] = 1
        energy = objective_for_instance(x, mu, sigma)
        bits = "".join(map(str, x.tolist()))

        if energy < best_energy - ENERGY_ATOL:
            best_energy, best_bits = float(energy), [bits]
        elif abs(energy - best_energy) <= ENERGY_ATOL:
            best_bits.append(bits)

    return float(best_energy), tuple(sorted(best_bits))


shock_coefficient = float((1.0 - Q_VALUE) * mu_scale)
return_threshold_rows = []

for k_value in GENERALIZATION_K_VALUES:
    for asset_index in range(N_ASSETS):
        e_in, bits_in = best_portfolios_with_asset_condition(
            k_value, asset_index, True
        )
        e_out, bits_out = best_portfolios_with_asset_condition(
            k_value, asset_index, False
        )

        critical = (
            float((e_in - e_out) / shock_coefficient)
            if abs(shock_coefficient) > 1e-15 else np.nan
        )
        return_threshold_rows.append({
            "k": int(k_value),
            "asset_index": int(asset_index),
            "critical_return_shock_sigma_units": critical,
            "absolute_margin_sigma_units": float(abs(critical)),
            "best_energy_if_included_at_zero_shock": e_in,
            "best_energy_if_excluded_at_zero_shock": e_out,
            "best_bitstrings_if_included": bits_in,
            "best_bitstrings_if_excluded": bits_out,
        })

return_threshold_df = pd.DataFrame(return_threshold_rows)

# Auditoria contra os 120 cenários de choque já enumerados na Célula 56.
audit_rows = []
for row in generalization_manifest_df.query(
    "scenario_type == 'return_shock'"
).itertuples(index=False):
    threshold = return_threshold_df.loc[
        (return_threshold_df["k"] == int(row.k))
        & (return_threshold_df["asset_index"] == int(row.asset_index_shocked))
    ].iloc[0]

    if float(row.return_shock_sigma_units) > float(
        threshold["critical_return_shock_sigma_units"]
    ):
        predicted = set(threshold["best_bitstrings_if_included"])
    else:
        predicted = set(threshold["best_bitstrings_if_excluded"])

    exact = set(row.exact_bitstrings_asset_order)
    audit_rows.append({
        "scenario_id": int(row.scenario_id),
        "k": int(row.k),
        "asset_index": int(row.asset_index_shocked),
        "shock": float(row.return_shock_sigma_units),
        "prediction_matches_exact": bool(predicted == exact),
    })

return_threshold_audit_df = pd.DataFrame(audit_rows)
display(return_threshold_df)
print(
    "Limiar analítico reproduziu todos os choques da grade:",
    bool(return_threshold_audit_df["prediction_matches_exact"].all()),
)

return_threshold_df.to_csv(
    VALIDATION19_TABLE_DIR / "exact_return_shock_thresholds.csv", index=False
)
return_threshold_audit_df.to_csv(
    VALIDATION19_TABLE_DIR / "return_threshold_manifest_audit.csv", index=False
)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
for k_value, group in return_threshold_df.groupby("k"):
    ax.plot(
        group["asset_index"],
        group["critical_return_shock_sigma_units"],
        marker="o",
        label=f"k={k_value}",
    )
ax.axhline(0.0, linewidth=0.8)
ax.set_xlabel("Índice do ativo")
ax.set_ylabel(r"choque crítico $s_c$ em unidades de $\sigma_\mu$")
ax.set_title("Fronteiras exatas de inclusão/exclusão por ativo")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(VALIDATION19_FIGURE_DIR / "exact_return_shock_thresholds.png", dpi=180)
plt.show()


## Célula 66 — mapa de robustez da decisão por retorno

O valor \(|s_c|\) funciona como uma **margem de estabilidade** da seleção em relação ao retorno daquele ativo:

- pequeno \(|s_c|\): a decisão está perto de uma fronteira;
- grande \(|s_c|\): é preciso uma perturbação maior para trocar o regime.

Para o Transformer, casos próximos da fronteira são especialmente importantes porque distinguem um modelo que apenas memoriza frequências de seleção de um modelo que aprendeu a sensibilidade ao Hamiltoniano.


In [ ]:
# ============================================================
# 66. MAPA DE MARGENS E REGIMES PARA k=3,4,5
# ============================================================

margin_matrix = return_threshold_df.pivot(
    index="k",
    columns="asset_index",
    values="absolute_margin_sigma_units",
).sort_index()

fig, ax = plt.subplots(figsize=(9.0, 3.6))
im = ax.imshow(margin_matrix.to_numpy(dtype=float), aspect="auto")
ax.set_xticks(np.arange(len(margin_matrix.columns)))
ax.set_xticklabels(margin_matrix.columns)
ax.set_yticks(np.arange(len(margin_matrix.index)))
ax.set_yticklabels([f"k={k}" for k in margin_matrix.index])
ax.set_xlabel("Ativo")
ax.set_title("Margem até a fronteira de decisão |s_c|")
fig.colorbar(im, ax=ax, label=r"$|s_c|$ em unidades de $\sigma_\mu$")
fig.tight_layout()
fig.savefig(VALIDATION19_FIGURE_DIR / "return_decision_margin_heatmap.png", dpi=180)
plt.show()

return_margin_summary_df = (
    return_threshold_df.groupby("k")["absolute_margin_sigma_units"]
    .agg(["min", "median", "mean", "max"])
    .reset_index()
)
display(return_margin_summary_df)
return_margin_summary_df.to_csv(
    VALIDATION19_TABLE_DIR / "return_margin_summary_by_k.csv", index=False
)


## Célula 67 — variar a escala global de risco

Ao multiplicar toda a matriz de covariância por um fator \(\rho\), cada portfólio passa a definir uma reta em \(\rho\). O ótimo global é a envoltória inferior dessas retas e pode atravessar vários regimes.

Aqui fazemos um scan clássico barato, apenas para localizar as regiões onde a seleção muda. Esses pontos serão usados para escolher quais instâncias merecem o custo do VQE/QGT.


In [ ]:
# ============================================================
# 67. SCAN CLÁSSICO DE ESCALA DE RISCO E MUDANÇAS DE REGIME
# ============================================================

risk_scan_rows = []
risk_transition_rows = []

for k_value in GENERALIZATION_K_VALUES:
    _, base_bits = exact_optimum_for_instance(k_value, mu, sigma)
    previous_bits = None
    previous_scale = None

    for risk_scale in RISK_SCAN_LEVELS_20_19:
        e, bits = exact_optimum_for_instance(
            k_value, mu, np.asarray(sigma, dtype=float) * float(risk_scale)
        )

        nearest_hamming = min(
            sum(a != b for a, b in zip(candidate, base))
            for candidate in bits for base in base_bits
        )
        risk_scan_rows.append({
            "k": int(k_value),
            "risk_scale": float(risk_scale),
            "exact_energy": float(e),
            "exact_bitstrings_asset_order": bits,
            "n_optima": int(len(bits)),
            "nearest_base_hamming": int(nearest_hamming),
        })

        if previous_bits is not None and set(bits) != set(previous_bits):
            risk_transition_rows.append({
                "k": int(k_value),
                "left_scale": float(previous_scale),
                "right_scale": float(risk_scale),
                "left_optima": previous_bits,
                "right_optima": bits,
            })

        previous_bits = bits
        previous_scale = float(risk_scale)

risk_scan_df = pd.DataFrame(risk_scan_rows)
risk_transition_df = pd.DataFrame(risk_transition_rows)

display(risk_transition_df if not risk_transition_df.empty else pd.DataFrame(
    {"diagnostico": ["Nenhuma mudança de regime no intervalo escaneado."]}
))

risk_scan_df.to_csv(
    VALIDATION19_TABLE_DIR / "risk_scale_scan.csv", index=False
)
risk_transition_df.to_csv(
    VALIDATION19_TABLE_DIR / "risk_scale_transition_brackets.csv", index=False
)

fig, ax = plt.subplots(figsize=(8.0, 4.5))
for k_value, group in risk_scan_df.groupby("k"):
    ax.plot(
        group["risk_scale"],
        group["nearest_base_hamming"],
        marker="o",
        markersize=3,
        label=f"k={k_value}",
    )
ax.set_xlabel("Escala global da covariância")
ax.set_ylabel("Hamming até o ótimo-base mais próximo")
ax.set_title("Mudanças discretas da seleção ao variar o peso global do risco")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(VALIDATION19_FIGURE_DIR / "risk_scale_regime_changes.png", dpi=180)
plt.show()


## Célula 68 — selecionar uma campanha quântica informativa

Não é necessário rodar VQE em todos os 129 casos clássicos.

A campanha mais informativa deve misturar:

- **base** de cada \(k\);
- casos **próximos de uma fronteira** de retorno;
- casos **distantes da fronteira**;
- pontos imediatamente antes/depois de mudanças de regime por risco.

Assim conseguimos testar se as quantidades quânticas

\[
r_{\rm QGT},\quad
N_{\rm ativo},\quad
L_{\rm route},\quad
N_{\rm eff},\quad
R_\phi
\]

mudam de maneira organizada quando o ótimo clássico atravessa uma fronteira.


In [ ]:
# ============================================================
# 68. MANIFESTO COMPACTO PARA A PRÓXIMA CAMPANHA VQE/QGT
# ============================================================

campaign_rows = []

for k_value in GENERALIZATION_K_VALUES:
    group = return_threshold_df.query("k == @k_value").sort_values(
        "absolute_margin_sigma_units"
    )
    fragile = group.head(2)
    robust = group.tail(1)

    campaign_rows.append({
        "k": int(k_value),
        "scenario_family": "base",
        "asset_index": -1,
        "return_shock_sigma_units": 0.0,
        "risk_scale": 1.0,
        "reason": "referência da cardinalidade",
    })

    for row in fragile.itertuples(index=False):
        sc = float(row.critical_return_shock_sigma_units)
        for offset in (-0.15, 0.15):
            campaign_rows.append({
                "k": int(k_value),
                "scenario_family": "near_return_boundary",
                "asset_index": int(row.asset_index),
                "return_shock_sigma_units": float(sc + offset),
                "risk_scale": 1.0,
                "reason": "imediatamente ao redor da fronteira analítica",
            })

    for row in robust.itertuples(index=False):
        direction = 1.0 if float(row.critical_return_shock_sigma_units) >= 0 else -1.0
        campaign_rows.append({
            "k": int(k_value),
            "scenario_family": "far_from_return_boundary",
            "asset_index": int(row.asset_index),
            "return_shock_sigma_units": float(-direction * min(
                1.0, 0.5 * abs(float(row.critical_return_shock_sigma_units))
            )),
            "risk_scale": 1.0,
            "reason": "controle em região de decisão estável",
        })

    if not risk_transition_df.empty:
        transitions = risk_transition_df.query("k == @k_value")
        for tr in transitions.itertuples(index=False):
            campaign_rows.extend([
                {
                    "k": int(k_value),
                    "scenario_family": "risk_boundary_left",
                    "asset_index": -1,
                    "return_shock_sigma_units": 0.0,
                    "risk_scale": float(tr.left_scale),
                    "reason": "lado esquerdo de mudança de regime por risco",
                },
                {
                    "k": int(k_value),
                    "scenario_family": "risk_boundary_right",
                    "asset_index": -1,
                    "return_shock_sigma_units": 0.0,
                    "risk_scale": float(tr.right_scale),
                    "reason": "lado direito de mudança de regime por risco",
                },
            ])

quantum_campaign_20_19_df = pd.DataFrame(campaign_rows).drop_duplicates(
    subset=["k", "scenario_family", "asset_index",
            "return_shock_sigma_units", "risk_scale"]
).reset_index(drop=True)

display(quantum_campaign_20_19_df)
print("Instâncias recomendadas para VQE/QGT:", len(quantum_campaign_20_19_df))

quantum_campaign_20_19_df.to_csv(
    VALIDATION19_TABLE_DIR / "quantum_validation_campaign.csv", index=False
)


# Parte XIII — o que entra no Transformer depois da 20.19

A 20.19 acrescenta dois objetos que são mais transferíveis que índices de θ.

### 1. Grafo de transporte

Para cada instância quântica, os targets auxiliares podem incluir:

- blocos que realizam transporte;
- precedência entre esses blocos;
- número de substituições mínimas \(L_{\min}\);
- comprimento observado da rota \(L_{\rm route}\);
- relays intermediários;
- alongamento \(\chi_{\rm route}\).

O modelo deve prever esses objetos a partir do Hamiltoniano e da estrutura do ansatz — eles **não são inputs**.

### 2. Coerência das histórias

Em vez de usar somente `n_histories`, separar:

\[
N_{\rm eff}
\]

(multiplicidade efetiva) e

\[
R_\phi
\]

(alinhamento). Isso evita confundir “muitas histórias” com “histórias que realmente se somam construtivamente”.

### 3. Margem clássica de decisão

Os limiares \(s_c\) e as mudanças por escala de risco ajudam a **estratificar o dataset**: treinar e avaliar separadamente instâncias longe e perto das fronteiras.

Eles são labels/auditorias gerados pela enumeração clássica, nunca informação que o Transformer recebe no teste.


In [ ]:
# ============================================================
# 69. TARGETS GLOBAIS DA INSTÂNCIA + ESQUEMA PARA O TRANSFORMER
# ============================================================

original_order_label = "→".join(map(str, transport_original_order))
original_history = all_perm_history_summary_df.loc[
    (all_perm_history_summary_df["order_label"] == original_order_label)
    & (all_perm_history_summary_df["target_label"] == "original_optimum")
].iloc[0]

current_margin = return_margin_summary_df.loc[
    return_margin_summary_df["k"] == int(TARGET_K)
].iloc[0]

transformer_global_targets_20_19_df = pd.DataFrame([{
    "instance_id": str(DATA_HASH),
    "n_assets": int(N_ASSETS),
    "k": int(TARGET_K),
    "k_over_n": float(TARGET_K / N_ASSETS),
    "n_parameters": int(N_PARAMETERS),
    "qgt_numeric_rank": int(qgt_summary_df.iloc[0]["qgt_numeric_rank"]),
    "active_parameter_count": int(len(ACTIVE_THETA_INDICES)),
    "transport_block_count": int(len(TRANSPORT_THETA_INDICES)),
    "minimum_replacements": int(minimum_length),
    "route_replacement_length": int(route_length),
    "route_stretch": float(route_stretch),
    "relay_count": int(len(relay_nodes)),
    "history_count": int(original_history["n_histories"]),
    "effective_history_number": float(original_history["effective_history_number"]),
    "phase_resultant": float(original_history["phase_resultant"]),
    "alignment_efficiency": float(original_history["alignment_efficiency"]),
    "interference_balance": float(original_history["interference_balance"]),
    "return_margin_min_sigma": float(current_margin["min"]),
    "return_margin_median_sigma": float(current_margin["median"]),
}])

display(transformer_global_targets_20_19_df)
transformer_global_targets_20_19_df.to_csv(
    VALIDATION19_TABLE_DIR / "transformer_global_targets_current_instance.csv",
    index=False,
)

schema_20_19_df = pd.DataFrame([
    ("mu_i, Sigma_ii, conexões de risco, k/n", "INPUT", "disponível antes da solução"),
    ("estrutura local do bloco e qubits tocados", "INPUT", "descreve o ansatz"),
    ("bitstring ótimo e energia", "TARGET principal", "saída física"),
    ("bloco ativo / bloco transportador / precedência", "TARGET auxiliar", "mecanismo variacional"),
    ("QGT rank e curvaturas", "TARGET auxiliar", "geometria útil do ansatz"),
    ("L_min, L_route, relay_count, route_stretch", "TARGET auxiliar", "topologia de transporte"),
    ("N_eff, phase_resultant, interference_balance", "TARGET auxiliar", "organização das histórias"),
    ("limiar s_c e margem até fronteira", "AUDITORIA/TARGET", "não usar como input; vem de enumeração"),
    ("theta_index nominal", "AUDITORIA", "não possui significado universal"),
], columns=["campo", "papel_no_modelo", "motivo"])

display(schema_20_19_df)
schema_20_19_df.to_csv(
    VALIDATION19_TABLE_DIR / "transformer_schema_20_19.csv", index=False
)


# Critérios de interpretação da 20.19

1. **Interferência dirigida:** só afirmar que a ordem redireciona interferência se, com os mesmos θ, as seis ordens produzirem distribuições/histórias distintas.
2. **Fases:** se `fraction_phase_0_or_pi ≈ 1`, descrever o mecanismo como interferência por sinais reais \(0/\pi\), não como uma distribuição geral de fases complexas.
3. **Cancelamento extremo:** quando `phase_resultant` chega ao nível de precisão numérica, interpretar \(P_{\rm coh}\) minúsculo como compatível com zero; usar o cancelamento, não os últimos dígitos.
4. **Expressividade:** reachability baixa após alguns reinícios é evidência, não prova, de limitação estrutural.
5. **Rota:** `route_stretch > 1` é um desvio combinatório no espaço de escolhas; não chamar isso de “ação de Feynman”.
6. **Generalização:** os 129 cenários clássicos mostram fronteiras de decisão, mas não demonstram ainda que a compressão `30→7→3` se repete em `3/10` ou `5/10`.
7. A próxima evidência decisiva é executar a campanha `quantum_validation_campaign.csv` e medir, em cada instância, o QGT, o subconjunto ativo, a rota e as histórias.
8. Para o Transformer, preservar a regra de leakage: ótimo clássico, limiares e diagnósticos quânticos são **labels**, nunca inputs.
